# NB7 — Clinical Decision-Support Prototype

## Research Objective

This notebook develops a research-oriented Clinical Decision-Support (CDS) prototype that integrates the validated outputs from NB3–NB6 into a structured, patient-level decision-support workflow.

The prototype is designed to demonstrate how a trustworthy clinical AI system can communicate model predictions together with patient-level explanations, prediction confidence, safety indicators, and explanation-consistency evidence.

The underlying predictive model and preprocessing pipeline remain locked and are reused from NB3. No model retraining or threshold modification is performed in NB7.

---

## Research Questions

1. How can predictive model outputs and patient-level explanations be transformed into a structured clinical decision-support representation?

2. How can prediction confidence, calibration limitations, false-positive/false-negative safety information, and explanation consistency be incorporated into the decision-support output?

3. Can the prototype communicate model evidence in a transparent and clinically interpretable format without presenting the model output as a diagnosis?

4. How can trustworthiness evidence from NB5 and explanation-consistency evidence from NB6 be integrated into a patient-level CDS workflow?

5. Can the resulting CDS prototype provide a reproducible foundation for the trustworthy multi-agent healthcare workflow developed in NB8?

---

## Scope

NB7 will integrate:

- The locked predictive model and preprocessing pipeline from NB3
- Patient-level prediction outputs
- Patient-level model explanations developed in NB4 and reconstructed using the locked NB3 model where required
- Prediction confidence information
- Calibration and uncertainty-related warnings identified in NB5
- False-positive and false-negative safety indicators from NB5
- Relevant subgroup/fairness context from NB5
- Explanation-consistency evidence from NB6
- Structured clinical decision-support communication

The prototype will focus on **research-oriented clinical risk assessment and decision support**, rather than automated diagnosis or treatment recommendation.

---

## Core CDS Workflow

Patient Data
↓
Data Quality / Input Validation
↓
Locked Predictive Model
↓
Predicted Probability
↓
Prediction at Locked Threshold
↓
Patient-Level Explanation
↓
Confidence Assessment
↓
Trust, Fairness & Safety Checks
↓
Explanation Consistency Evidence
↓
Structured Clinical Decision-Support Summary
↓
Safety-Aware Interpretation

---

## Methodological Safeguards

- Reuse the locked NB3 predictive model.
- Reuse the locked NB3 preprocessing pipeline.
- Preserve the original prediction threshold of 0.35.
- Do not retrain the predictive model.
- Do not optimize the model using the held-out test cohort.
- Preserve participant-level train/test separation.
- Maintain reproducibility using the established random seed where sampling is required.
- Clearly distinguish model prediction from clinical diagnosis.
- Clearly distinguish classification confidence from calibrated clinical risk.
- Treat safety indicators as research-oriented review signals rather than validated clinical risk categories.
- Avoid causal interpretation of model features.
- Avoid unsupported treatment recommendations.
- Avoid claims of clinical validity, clinical utility, or deployment readiness.

---

## Intended Output

The final NB7 prototype will produce a structured patient-level CDS representation containing, where available:

- Patient identifier
- Model prediction
- Predicted probability
- Locked classification threshold
- Prediction confidence
- Key explanatory features
- Direction of model contribution
- Explanation-consistency information
- Trust and safety flags
- Relevant fairness/context indicators
- Clinical interpretation
- Explicit safety and non-diagnostic disclaimer

---

## Research Position

NB7 represents the transition from model evaluation to an applied Clinical Decision-Support prototype.

NB3 establishes predictive performance.
NB4 establishes explainability.
NB5 establishes trust, fairness, and safety evidence.
NB6 establishes explanation consistency.
NB7 integrates these evidence layers into a structured patient-level CDS workflow.
NB8 will extend this workflow into a trustworthy multi-agent healthcare architecture.

The prototype is intended as a research demonstrator and does not constitute a clinically validated medical device or autonomous clinical decision-making system.

In [7]:
# ============================================================
# NB7 — Clinical Decision-Support Prototype
# Cell 2 — Environment & Reproducibility Setup
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
BASE_DIR = Path("/content")

NB7_DIR = BASE_DIR / "nb7_clinical_decision_support"
FIGURES_DIR = NB7_DIR / "figures"
TABLES_DIR = NB7_DIR / "tables"
FINAL_REPORT_DIR = NB7_DIR / "final_report"

for directory in [NB7_DIR, FIGURES_DIR, TABLES_DIR, FINAL_REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

# ------------------------------------------------------------
# Notebook configuration
# ------------------------------------------------------------
NB7_TITLE = "NB7 — Clinical Decision-Support Prototype"

LOCKED_THRESHOLD = 0.35

print("=" * 70)
print(NB7_TITLE)
print("=" * 70)
print(f"Random state: {RANDOM_STATE}")
print(f"Base directory: {BASE_DIR}")
print(f"NB7 output directory: {NB7_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"Tables directory: {TABLES_DIR}")
print(f"Final report directory: {FINAL_REPORT_DIR}")
print(f"Locked prediction threshold: {LOCKED_THRESHOLD}")
print("=" * 70)
print("Environment and reproducibility setup completed successfully.")

NB7 — Clinical Decision-Support Prototype
Random state: 42
Base directory: /content
NB7 output directory: /content/nb7_clinical_decision_support
Figures directory: /content/nb7_clinical_decision_support/figures
Tables directory: /content/nb7_clinical_decision_support/tables
Final report directory: /content/nb7_clinical_decision_support/final_report
Locked prediction threshold: 0.35
Environment and reproducibility setup completed successfully.


In [8]:
# ============================================================
# NB7 — Cell 3
# Locate and Audit Required Previous-Stage Artifacts
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Candidate artifact locations
# ------------------------------------------------------------
SEARCH_ROOTS = [
    Path("/content"),
    Path("/content/nb5_trustworthiness"),
    Path("/content/nb6_explainability_consistency"),
]

def find_artifact(filename):
    """
    Search known project locations for a required artifact.
    Returns the first matching path.
    """
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        matches = list(root.rglob(filename))
        if matches:
            return matches[0]

    return None


# ------------------------------------------------------------
# Required artifacts
# ------------------------------------------------------------
REQUIRED_ARTIFACTS = {
    # NB3 predictive model
    "model": "early_detection_logistic_regression.joblib",
    "preprocessor": "early_detection_preprocessor.joblib",
    "metadata": "early_detection_model_metadata.json",

    # NB3 prediction outputs
    "test_predictions": "early_detection_test_predictions.csv",
    "oof_predictions": "early_detection_oof_training_predictions.csv",

    # NB3 processed feature names
    "feature_names": "early_detection_processed_feature_names.csv",

    # NB5 reconstructed held-out prediction artifact
    "nb5_test_predictions": "nb5_reconstructed_test_predictions_6386.csv",

    # NB6 explanation-consistency evidence
    "nb6_pairwise_similarity": "nb6_pairwise_explanation_similarity.csv",
    "nb6_stability_summary": "nb6_patient_explanation_stability_summary.csv",
    "nb6_stable_features": "nb6_stable_explanatory_features_research_ready.csv",
    "nb6_integrated_evidence": "nb6_research_ready_explanation_consistency_evidence.csv",
}


# ------------------------------------------------------------
# Locate artifacts
# ------------------------------------------------------------
artifact_paths = {}

for key, filename in REQUIRED_ARTIFACTS.items():
    artifact_paths[key] = find_artifact(filename)


# ------------------------------------------------------------
# Audit results
# ------------------------------------------------------------
audit_rows = []

for key, filename in REQUIRED_ARTIFACTS.items():
    path = artifact_paths[key]

    audit_rows.append({
        "artifact_key": key,
        "filename": filename,
        "status": "FOUND" if path is not None else "MISSING",
        "path": str(path) if path is not None else None
    })

artifact_audit = pd.DataFrame(audit_rows)

display(artifact_audit)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
found_count = int((artifact_audit["status"] == "FOUND").sum())
missing_count = int((artifact_audit["status"] == "MISSING").sum())

print("\n" + "=" * 70)
print("NB7 ARTIFACT AUDIT")
print("=" * 70)
print(f"Required artifacts: {len(REQUIRED_ARTIFACTS)}")
print(f"Found: {found_count}")
print(f"Missing: {missing_count}")

if missing_count == 0:
    print("\nAll required artifacts were located successfully.")
else:
    print("\nWARNING: One or more artifacts are missing.")
    print("We will resolve missing artifacts before continuing.")

print("=" * 70)

,artifact_key,filename,status,path
0,model,early_detection_logistic_regression.joblib,MISSING,None
1,preprocessor,early_detection_preprocessor.joblib,MISSING,None
2,metadata,early_detection_model_metadata.json,MISSING,None
3,test_predictions,early_detection_test_predictions.csv,MISSING,None
4,oof_predictions,early_detection_oof_training_predictions.csv,MISSING,None
5,feature_names,early_detection_processed_feature_names.csv,MISSING,None
6,nb5_test_predictions,nb5_reconstructed_test_predictions_6386.csv,MISSING,None
7,nb6_pairwise_similarity,nb6_pairwise_explanation_similarity.csv,MISSING,None
8,nb6_stability_summary,nb6_patient_explanation_stability_summary.csv,MISSING,None
9,nb6_stable_features,nb6_stable_explanatory_features_research_ready...,MISSING,None



NB7 ARTIFACT AUDIT
Required artifacts: 11
Found: 0
Missing: 11

We will resolve missing artifacts before continuing.


In [15]:
# ============================================================
# NB7 — Cell 4
# Upload Previous-Stage Evidence — One File at a Time
# ============================================================

from google.colab import files
from pathlib import Path
import zipfile
import shutil

print("=" * 70)
print("NB7 — Previous-Stage Evidence Upload")
print("=" * 70)

UPLOAD_DIR = NB7_DIR / "uploaded_artifacts"
EXTRACT_DIR = NB7_DIR / "imported_artifacts"

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("""
Upload ONE file at a time.

Required files:
1. NB6_Explainability_Consistency_Final_Package.zip
2. notebook3_early_detection_exports.zip

Run this cell again after each upload.
Previously uploaded files will NOT be deleted.
""")

uploaded = files.upload()

for filename in uploaded.keys():

    source = Path(filename)
    destination = UPLOAD_DIR / filename

    # Copy uploaded file into the persistent NB7 working directory
    shutil.copy2(source, destination)

    print(f"\nUploaded successfully: {filename}")

    # --------------------------------------------------------
    # Extract ZIP package
    # --------------------------------------------------------
    if destination.suffix.lower() == ".zip":

        extract_target = EXTRACT_DIR / destination.stem

        # Remove previous extraction of the same package
        # so repeated uploads do not create confusing duplicates.
        if extract_target.exists():
            shutil.rmtree(extract_target)

        extract_target.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(destination, "r") as z:
            z.extractall(extract_target)

        print(f"Extracted to: {extract_target}")

print("\n" + "=" * 70)

# ------------------------------------------------------------
# Show everything currently uploaded
# ------------------------------------------------------------
current_files = sorted(UPLOAD_DIR.iterdir())

print("CURRENTLY UPLOADED PACKAGES:")
for path in current_files:
    print(f"  ✓ {path.name}")

print(f"\nTotal uploaded packages: {len(current_files)}")
print("=" * 70)

NB7 — Previous-Stage Evidence Upload

Upload ONE file at a time.

Required files:
1. NB6_Explainability_Consistency_Final_Package.zip
2. notebook3_early_detection_exports.zip

Run this cell again after each upload.
Previously uploaded files will NOT be deleted.



Saving notebook3_early_detection_exports.zip to notebook3_early_detection_exports.zip

Uploaded successfully: notebook3_early_detection_exports.zip
Extracted to: /content/nb7_clinical_decision_support/imported_artifacts/notebook3_early_detection_exports

CURRENTLY UPLOADED PACKAGES:
  ✓ NB6_Explainability_Consistency_Final_Package.zip
  ✓ notebook3_early_detection_exports.zip

Total uploaded packages: 2


In [16]:
# ============================================================
# NB7 — Cell 5
# Import and Audit Previous-Stage Evidence
# ============================================================

import json
import zipfile
from pathlib import Path

print("=" * 70)
print("NB7 — Previous-Stage Evidence Audit")
print("=" * 70)

# ------------------------------------------------------------
# Locate extracted package directories
# ------------------------------------------------------------
NB3_IMPORT_DIR = (
    NB7_DIR
    / "imported_artifacts"
    / "notebook3_early_detection_exports"
)

NB6_IMPORT_DIR = (
    NB7_DIR
    / "imported_artifacts"
    / "NB6_Explainability_Consistency_Final_Package"
)

print(f"NB3 import directory: {NB3_IMPORT_DIR}")
print(f"NB6 import directory: {NB6_IMPORT_DIR}")

# ------------------------------------------------------------
# Required NB3 artifacts
# ------------------------------------------------------------
NB3_REQUIRED = [
    "early_detection_logistic_regression.joblib",
    "early_detection_preprocessor.joblib",
    "early_detection_model_metadata.json",
    "early_detection_test_predictions.csv",
    "early_detection_oof_training_predictions.csv",
    "early_detection_processed_feature_names.csv",
]

# ------------------------------------------------------------
# Find NB3 files recursively
# ------------------------------------------------------------
nb3_files = {}

for filename in NB3_REQUIRED:
    matches = list(NB3_IMPORT_DIR.rglob(filename))

    if matches:
        nb3_files[filename] = matches[0]
    else:
        nb3_files[filename] = None

# ------------------------------------------------------------
# Find NB6 evidence recursively
# ------------------------------------------------------------
NB6_REQUIRED_PATTERNS = {
    "integrated_evidence": "nb6_research_ready_explanation_consistency_evidence.csv",
    "stable_features": "nb6_stable_explanatory_features_research_ready.csv",
    "pairwise_similarity": "nb6_pairwise_explanation_similarity.csv",
    "stability_summary": "nb6_patient_explanation_stability_summary.csv",
    "multiscale_stability": "nb6_multiscale_perturbation_stability_summary.csv",
    "feature_perturbation": "nb6_feature_level_perturbation_consistency.csv",
    "patient_similarity": "nb6_patient_similarity_explanation_correlation.csv",
}

nb6_files = {}

for key, filename in NB6_REQUIRED_PATTERNS.items():
    matches = list(NB6_IMPORT_DIR.rglob(filename))

    if matches:
        nb6_files[key] = matches[0]
    else:
        nb6_files[key] = None

# ------------------------------------------------------------
# Create audit table
# ------------------------------------------------------------
audit_rows = []

for filename, path in nb3_files.items():
    audit_rows.append({
        "stage": "NB3",
        "artifact": filename,
        "status": "FOUND" if path is not None else "MISSING",
        "path": str(path) if path is not None else None
    })

for key, path in nb6_files.items():
    audit_rows.append({
        "stage": "NB6",
        "artifact": key,
        "status": "FOUND" if path is not None else "MISSING",
        "path": str(path) if path is not None else None
    })

evidence_audit = pd.DataFrame(audit_rows)

display(evidence_audit)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
found = int((evidence_audit["status"] == "FOUND").sum())
missing = int((evidence_audit["status"] == "MISSING").sum())

print("\n" + "=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)
print(f"Total expected artifacts: {len(evidence_audit)}")
print(f"Found: {found}")
print(f"Missing: {missing}")

if missing == 0:
    print("\n✓ All required NB3 and NB6 artifacts were located.")
else:
    print("\n⚠ Some expected artifacts are missing.")
    print("We will inspect the package structure before proceeding.")

# ------------------------------------------------------------
# Inspect NB3 metadata if available
# ------------------------------------------------------------
metadata_path = nb3_files["early_detection_model_metadata.json"]

if metadata_path is not None:
    with open(metadata_path, "r") as f:
        nb3_metadata = json.load(f)

    print("\nNB3 metadata loaded successfully.")
    print("Metadata keys:")
    print(list(nb3_metadata.keys()))

print("=" * 70)

NB7 — Previous-Stage Evidence Audit
NB3 import directory: /content/nb7_clinical_decision_support/imported_artifacts/notebook3_early_detection_exports
NB6 import directory: /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package


,stage,artifact,status,path
0,NB3,early_detection_logistic_regression.joblib,FOUND,/content/nb7_clinical_decision_support/importe...
1,NB3,early_detection_preprocessor.joblib,FOUND,/content/nb7_clinical_decision_support/importe...
2,NB3,early_detection_model_metadata.json,FOUND,/content/nb7_clinical_decision_support/importe...
3,NB3,early_detection_test_predictions.csv,FOUND,/content/nb7_clinical_decision_support/importe...
4,NB3,early_detection_oof_training_predictions.csv,FOUND,/content/nb7_clinical_decision_support/importe...
5,NB3,early_detection_processed_feature_names.csv,FOUND,/content/nb7_clinical_decision_support/importe...
6,NB6,integrated_evidence,FOUND,/content/nb7_clinical_decision_support/importe...
7,NB6,stable_features,FOUND,/content/nb7_clinical_decision_support/importe...
8,NB6,pairwise_similarity,FOUND,/content/nb7_clinical_decision_support/importe...
9,NB6,stability_summary,FOUND,/content/nb7_clinical_decision_support/importe...



AUDIT SUMMARY
Total expected artifacts: 13
Found: 12
Missing: 1

⚠ Some expected artifacts are missing.
We will inspect the package structure before proceeding.

NB3 metadata loaded successfully.
Metadata keys:
['threshold_information', 'test_metrics', 'training_rows', 'testing_rows', 'training_participants', 'testing_participants', 'participant_overlap', 'raw_features', 'processed_features', 'high_confidence_proxies_excluded']


In [17]:
# ============================================================
# NB7 — Cell 6
# Inspect NB6 package filenames and locate patient-similarity artifact
# ============================================================

from pathlib import Path

nb6_root = Path(
    "/content/nb7_clinical_decision_support/"
    "imported_artifacts/NB6_Explainability_Consistency_Final_Package"
)

print("=" * 70)
print("NB7 — NB6 Package Structure Inspection")
print("=" * 70)

if not nb6_root.exists():
    print("ERROR: NB6 package directory not found.")
else:
    all_files = sorted(
        [p for p in nb6_root.rglob("*") if p.is_file()],
        key=lambda p: str(p).lower()
    )

    print(f"Total files found: {len(all_files)}\n")

    print("CSV ARTIFACTS:")
    print("-" * 70)

    csv_files = [p for p in all_files if p.suffix.lower() == ".csv"]

    for p in csv_files:
        print(p.relative_to(nb6_root))

    print("\n" + "=" * 70)
    print("PATIENT-SIMILARITY CANDIDATES")
    print("=" * 70)

    candidates = [
        p for p in all_files
        if any(
            term in p.name.lower()
            for term in [
                "patient_similarity",
                "patient-similarity",
                "similarity_explanation",
                "profile_similarity",
                "explanation_correlation",
                "correlation"
            ]
        )
    ]

    if candidates:
        for p in candidates:
            print("CANDIDATE:", p.relative_to(nb6_root))
    else:
        print("No obvious patient-similarity filename found.")

    print("\n" + "=" * 70)
    print("DONE")
    print("=" * 70)

NB7 — NB6 Package Structure Inspection
Total files found: 51

CSV ARTIFACTS:
----------------------------------------------------------------------
nb6_explainability_consistency/final_report/nb6_complete_artifact_inventory.csv
nb6_explainability_consistency/final_report/nb6_methodological_safeguards.csv
nb6_explainability_consistency/final_report/nb6_research_limitations.csv
nb6_explainability_consistency/notebook3_import/early_detection_oof_training_predictions.csv
nb6_explainability_consistency/notebook3_import/early_detection_processed_feature_names.csv
nb6_explainability_consistency/notebook3_import/early_detection_test_predictions.csv
nb6_explainability_consistency/tables/nb6_controlled_perturbation_explanation_stability.csv
nb6_explainability_consistency/tables/nb6_demographic_explanation_consistency_ranges.csv
nb6_explainability_consistency/tables/nb6_explanation_consistency_by_confidence.csv
nb6_explainability_consistency/tables/nb6_explanation_consistency_by_demographic_group

In [18]:
# ============================================================
# NB7 — Cell 7
# Corrected Previous-Stage Evidence Audit
# ============================================================

from pathlib import Path
import pandas as pd
import json

base_dir = Path("/content/nb7_clinical_decision_support")

nb3_dir = base_dir / "imported_artifacts/notebook3_early_detection_exports"
nb6_dir = base_dir / "imported_artifacts/NB6_Explainability_Consistency_Final_Package"

# ------------------------------------------------------------
# Locate NB6 files recursively
# ------------------------------------------------------------

def find_file(root, filename):
    matches = list(root.rglob(filename))
    return matches[0] if matches else None

# ------------------------------------------------------------
# NB3 artifacts
# ------------------------------------------------------------

NB3_FILES = {
    "model": "early_detection_logistic_regression.joblib",
    "preprocessor": "early_detection_preprocessor.joblib",
    "metadata": "early_detection_model_metadata.json",
    "test_predictions": "early_detection_test_predictions.csv",
    "oof_predictions": "early_detection_oof_training_predictions.csv",
    "processed_features": "early_detection_processed_feature_names.csv",
}

# ------------------------------------------------------------
# NB6 research evidence artifacts
# ------------------------------------------------------------

NB6_FILES = {
    "integrated_evidence":
        "nb6_research_ready_explanation_consistency_evidence.csv",

    "stable_features":
        "nb6_stable_explanatory_features_research_ready.csv",

    "pairwise_similarity":
        "nb6_pairwise_explanation_similarity.csv",

    "stability_summary":
        "nb6_patient_explanation_stability_summary.csv",

    "multiscale_stability":
        "nb6_multiscale_perturbation_stability_summary.csv",

    "feature_perturbation":
        "nb6_feature_level_perturbation_consistency.csv",

    "patient_similarity":
        "nb6_patient_similarity_explanation_correlations.csv",
}

# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

audit_rows = []

for key, filename in NB3_FILES.items():
    path = find_file(nb3_dir, filename)

    audit_rows.append({
        "stage": "NB3",
        "artifact": key,
        "filename": filename,
        "status": "FOUND" if path else "MISSING",
        "path": str(path) if path else None
    })

for key, filename in NB6_FILES.items():
    path = find_file(nb6_dir, filename)

    audit_rows.append({
        "stage": "NB6",
        "artifact": key,
        "filename": filename,
        "status": "FOUND" if path else "MISSING",
        "path": str(path) if path else None
    })

corrected_audit = pd.DataFrame(audit_rows)

display(corrected_audit)

print("\n" + "=" * 70)
print("CORRECTED AUDIT SUMMARY")
print("=" * 70)

found = (corrected_audit["status"] == "FOUND").sum()
missing = (corrected_audit["status"] == "MISSING").sum()

print(f"Expected artifacts: {len(corrected_audit)}")
print(f"Found: {found}")
print(f"Missing: {missing}")

if missing == 0:
    print("\n✓ ALL REQUIRED NB3 AND NB6 ARTIFACTS FOUND")
else:
    print("\n⚠ Some artifacts are still missing.")

# ------------------------------------------------------------
# Load NB3 metadata
# ------------------------------------------------------------

metadata_path = find_file(
    nb3_dir,
    NB3_FILES["metadata"]
)

with open(metadata_path, "r") as f:
    nb3_metadata = json.load(f)

print("\nNB3 MODEL METADATA")
print("-" * 70)

print("Locked threshold:",
      nb3_metadata["threshold_information"].get(
          "selected_threshold",
          nb3_metadata["threshold_information"]
      ))

print("Training rows:",
      nb3_metadata.get("training_rows"))

print("Testing rows:",
      nb3_metadata.get("testing_rows"))

print("Training participants:",
      nb3_metadata.get("training_participants"))

print("Testing participants:",
      nb3_metadata.get("testing_participants"))

print("Participant overlap:",
      nb3_metadata.get("participant_overlap"))

print("Raw features:",
      nb3_metadata.get("raw_features"))

print("Processed features:",
      nb3_metadata.get("processed_features"))

# ------------------------------------------------------------
# Load key NB6 evidence tables
# ------------------------------------------------------------

NB6_EVIDENCE = {}

for key, filename in NB6_FILES.items():
    path = find_file(nb6_dir, filename)

    if path is not None:
        NB6_EVIDENCE[key] = pd.read_csv(path)

print("\n" + "=" * 70)
print("NB6 EVIDENCE TABLES LOADED")
print("=" * 70)

for key, df in NB6_EVIDENCE.items():
    print(f"{key:25s} shape={df.shape}")

print("\n" + "=" * 70)
print("NB7 EVIDENCE INITIALIZATION COMPLETE")
print("=" * 70)

,stage,artifact,filename,status,path
0,NB3,model,early_detection_logistic_regression.joblib,FOUND,/content/nb7_clinical_decision_support/importe...
1,NB3,preprocessor,early_detection_preprocessor.joblib,FOUND,/content/nb7_clinical_decision_support/importe...
2,NB3,metadata,early_detection_model_metadata.json,FOUND,/content/nb7_clinical_decision_support/importe...
3,NB3,test_predictions,early_detection_test_predictions.csv,FOUND,/content/nb7_clinical_decision_support/importe...
4,NB3,oof_predictions,early_detection_oof_training_predictions.csv,FOUND,/content/nb7_clinical_decision_support/importe...
5,NB3,processed_features,early_detection_processed_feature_names.csv,FOUND,/content/nb7_clinical_decision_support/importe...
6,NB6,integrated_evidence,nb6_research_ready_explanation_consistency_evi...,FOUND,/content/nb7_clinical_decision_support/importe...
7,NB6,stable_features,nb6_stable_explanatory_features_research_ready...,FOUND,/content/nb7_clinical_decision_support/importe...
8,NB6,pairwise_similarity,nb6_pairwise_explanation_similarity.csv,FOUND,/content/nb7_clinical_decision_support/importe...
9,NB6,stability_summary,nb6_patient_explanation_stability_summary.csv,FOUND,/content/nb7_clinical_decision_support/importe...



CORRECTED AUDIT SUMMARY
Expected artifacts: 13
Found: 13
Missing: 0

✓ ALL REQUIRED NB3 AND NB6 ARTIFACTS FOUND

NB3 MODEL METADATA
----------------------------------------------------------------------
Locked threshold: {'model': 'Early-Detection Logistic Regression', 'threshold': 0.35, 'threshold_selection_method': '5-fold out-of-fold training predictions', 'target_sensitivity': 0.75, 'test_set_used_for_threshold_selection': False}
Training rows: 25227
Testing rows: 6242
Training participants: 11755
Testing participants: 2939
Participant overlap: 0
Raw features: 214
Processed features: 517

NB6 EVIDENCE TABLES LOADED
integrated_evidence       shape=(11, 5)
stable_features           shape=(517, 9)
pairwise_similarity       shape=(9998, 5)
stability_summary         shape=(3, 3)
multiscale_stability      shape=(4, 14)
feature_perturbation      shape=(2068, 8)
patient_similarity        shape=(3, 6)

NB7 EVIDENCE INITIALIZATION COMPLETE


In [19]:
# ============================================================
# NB7 — Cell 8
# Inspect Research-Ready Explainability Consistency Evidence
# ============================================================

print("=" * 70)
print("NB7 — NB6 Explainability Consistency Evidence")
print("=" * 70)

# ------------------------------------------------------------
# 1. Integrated evidence matrix
# ------------------------------------------------------------

integrated = NB6_EVIDENCE["integrated_evidence"].copy()

print("\n[1] INTEGRATED EVIDENCE")
print("-" * 70)

display(integrated)

# ------------------------------------------------------------
# 2. Stable explanatory features
# ------------------------------------------------------------

stable_features = NB6_EVIDENCE["stable_features"].copy()

print("\n[2] STABLE EXPLANATORY FEATURES")
print("-" * 70)

print("Shape:", stable_features.shape)
display(stable_features.head(15))

# ------------------------------------------------------------
# 3. Pairwise explanation similarity
# ------------------------------------------------------------

pairwise = NB6_EVIDENCE["pairwise_similarity"].copy()

print("\n[3] PAIRWISE EXPLANATION SIMILARITY")
print("-" * 70)

print("Shape:", pairwise.shape)

numeric_pairwise = pairwise.select_dtypes(include="number")

if not numeric_pairwise.empty:
    print("\nNumeric summary:")
    display(numeric_pairwise.describe().T)

# ------------------------------------------------------------
# 4. Patient-level stability summary
# ------------------------------------------------------------

stability = NB6_EVIDENCE["stability_summary"].copy()

print("\n[4] PATIENT-LEVEL EXPLANATION STABILITY")
print("-" * 70)

display(stability)

# ------------------------------------------------------------
# 5. Multiscale perturbation stability
# ------------------------------------------------------------

multiscale = NB6_EVIDENCE["multiscale_stability"].copy()

print("\n[5] MULTISCALE PERTURBATION STABILITY")
print("-" * 70)

display(multiscale)

# ------------------------------------------------------------
# 6. Patient similarity relationship
# ------------------------------------------------------------

patient_similarity = NB6_EVIDENCE["patient_similarity"].copy()

print("\n[6] PATIENT SIMILARITY vs EXPLANATION SIMILARITY")
print("-" * 70)

display(patient_similarity)

# ------------------------------------------------------------
# Final interpretation guardrail
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INTERPRETATION GUARDRAIL")
print("=" * 70)

print(
    "NB7 will use NB6 consistency evidence as a model-level "
    "trustworthiness signal."
)

print(
    "It will NOT interpret explanation consistency as proof of "
    "clinical validity, causal importance, fairness, or safety."
)

print(
    "Patient-level CDS outputs will remain decision-support "
    "summaries, not autonomous clinical decisions."
)

print("\n✓ Evidence inspection complete.")

NB7 — NB6 Explainability Consistency Evidence

[1] INTEGRATED EVIDENCE
----------------------------------------------------------------------


,evidence_code,indicator,value,unit,interpretation
0,EC-01,Features appearing in ≥90% of patient top-10 e...,7.000000e+00,features,Strong recurring explanation structure
1,EC-02,Mean pairwise explanation cosine similarity,9.688456e-01,cosine similarity,High overall explanation similarity
2,EC-03,Pairs with ≥80% top-10 overlap,9.681936e+01,% of pairs,High proportion of strongly overlapping explan...
3,EC-04,Pairs with ≥90% top-10 overlap,6.813363e+01,% of pairs,Substantial high-overlap subset
4,EC-05,Prediction confidence vs explanation cosine Sp...,-2.225025e-01,Spearman ρ,Explanation consistency is not strongly determ...
5,EC-06,5% perturbation mean explanation cosine,9.999545e-01,cosine similarity,Very high local explanation stability
6,EC-07,20% perturbation mean explanation cosine,9.992773e-01,cosine similarity,Very high stability under stronger stress testing
7,EC-08,20% perturbation mean top-10 overlap,9.739000e+01,%,Explanation ranking remains highly stable
8,EC-09,Features maintaining ≥90% top-10 consistency a...,7.000000e+00,features,Persistent explanatory core identified
9,EC-10,Patient profile similarity vs explanation cosi...,1.407261e-01,Spearman ρ,Only weak association with patient similarity



[2] STABLE EXPLANATORY FEATURES
----------------------------------------------------------------------
Shape: (517, 9)


,consistency_rank,feature_index,processed_feature,top10_frequency,consistency_percent,consistency_category,top10_count,mean_absolute_contribution,median_absolute_contribution
0,1,31,numeric__BMXHIP,1.000,100.0,Very High,1000,7.093509,7.018061
1,2,100,numeric__LBXSOSSI,1.000,100.0,Very High,1000,3.559442,3.557093
2,3,71,numeric__LBXMCVSI,0.992,99.2,Very High,992,2.878843,2.876791
3,4,30,numeric__BMXWAIST,0.987,98.7,Very High,987,4.343063,4.418582
4,5,42,numeric__LBXGLU,0.985,98.5,Very High,985,3.000282,2.846291
5,6,110,numeric__LBXSCH,0.983,98.3,Very High,983,3.805959,3.735687
6,7,93,numeric__LBXSGL,0.955,95.5,Very High,955,3.192176,2.963404
7,8,87,numeric__LBXSCLSI,0.877,87.7,High,877,2.436233,2.443096
8,9,113,numeric__LBDSTPSI,0.823,82.3,High,823,2.410420,2.403875
9,10,45,numeric__LBXTC,0.444,44.4,Low,444,2.216267,2.175579



[3] PAIRWISE EXPLANATION SIMILARITY
----------------------------------------------------------------------
Shape: (9998, 5)

Numeric summary:


...


[5] MULTISCALE PERTURBATION STABILITY
----------------------------------------------------------------------


,perturbation_fraction,perturbation_percent,patients,mean_cosine_similarity,median_cosine_similarity,mean_explanation_correlation,median_explanation_correlation,mean_top10_overlap,median_top10_overlap,mean_absolute_contribution_change,cosine_ge_090,cosine_ge_095,top10_ge_080,top10_ge_090
0,0.01,1.0,1000,0.999998,0.999998,0.999998,0.999998,0.9979,1.0,0.000181,1.0,1.0,1.0,1.000
1,0.05,5.0,1000,0.999954,0.999954,0.999955,0.999954,0.9926,1.0,0.000906,1.0,1.0,1.0,1.000
2,0.10,10.0,1000,0.999819,0.999818,0.999819,0.999818,0.9877,1.0,0.001812,1.0,1.0,1.0,0.998
3,0.20,20.0,1000,0.999277,0.999268,0.999279,0.999269,0.9739,1.0,0.003623,1.0,1.0,1.0,0.996



[6] PATIENT SIMILARITY vs EXPLANATION SIMILARITY
----------------------------------------------------------------------


,comparison,spearman_r,spearman_p,pearson_r,pearson_p,n_pairs
0,Patient profile vs explanation cosine,0.140726,2.157276e-45,0.073734,1.561433e-13,9998
1,Patient profile vs explanation correlation,0.140627,2.489864e-45,0.073610,1.715329e-13,9998
2,Patient profile vs explanation top-10 overlap,0.070859,1.309783e-12,0.031434,1.669458e-03,9998



INTERPRETATION GUARDRAIL
NB7 will use NB6 consistency evidence as a model-level trustworthiness signal.
It will NOT interpret explanation consistency as proof of clinical validity, causal importance, fairness, or safety.
Patient-level CDS outputs will remain decision-support summaries, not autonomous clinical decisions.

✓ Evidence inspection complete.


In [21]:
# ============================================================
# NB7 — Cell 9
# Locate and Audit Exact NB3 Modeling Dataset
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("NB7 — Exact NB3 Modeling Dataset Audit")
print("=" * 70)

# Search locations where the dataset may have been included
search_roots = [
    Path("/content"),
    Path("/content/nb7_clinical_decision_support"),
]

dataset_name = "diabetes_modeling_dataset_final.csv"

matches = []

for root in search_roots:
    if root.exists():
        matches.extend(root.rglob(dataset_name))

# Remove duplicates while preserving order
matches = list(dict.fromkeys([str(p) for p in matches]))

print(f"\nDataset name: {dataset_name}")
print(f"Matches found: {len(matches)}")

for i, path in enumerate(matches, start=1):
    print(f"{i}. {path}")

print("\n" + "=" * 70)

if len(matches) == 0:
    print("⚠ Dataset not currently available in this runtime.")
    print(
        "The next step will be to upload the exact "
        "diabetes_modeling_dataset_final.csv."
    )

elif len(matches) == 1:
    dataset_path = Path(matches[0])

    print("✓ Exact dataset located.")
    print(f"Path: {dataset_path}")

    # Lightweight structural audit
    df_audit = pd.read_csv(dataset_path, nrows=5)

    print("\nDataset preview shape (first 5 rows):", df_audit.shape)
    print("Columns:", len(df_audit.columns))

    print("\nRequired identifiers/target:")
    for col in ["SEQN", "diabetes_target"]:
        print(f"  {col}: {'FOUND' if col in df_audit.columns else 'MISSING'}")

else:
    print("⚠ Multiple copies of the dataset were found.")
    print("We will resolve the correct copy before continuing.")

print("\n" + "=" * 70)
print("DATASET AUDIT COMPLETE")
print("=" * 70)

NB7 — Exact NB3 Modeling Dataset Audit

Dataset name: diabetes_modeling_dataset_final.csv
Matches found: 1
1. /content/diabetes_modeling_dataset_final.csv

✓ Exact dataset located.
Path: /content/diabetes_modeling_dataset_final.csv

Dataset preview shape (first 5 rows): (5, 244)
Columns: 244

Required identifiers/target:
  SEQN: FOUND
  diabetes_target: FOUND

DATASET AUDIT COMPLETE


In [22]:
# ============================================================
# NB7 — Exact NB3 Held-Out Test Cohort Reconstruction
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("NB7 — Exact NB3 Held-Out Test Cohort Reconstruction")
print("=" * 70)

# ------------------------------------------------------------
# Load exact NB3 modeling dataset
# ------------------------------------------------------------
dataset_path = Path("/content/diabetes_modeling_dataset_final.csv")

df = pd.read_csv(dataset_path)

print("\nFull dataset:")
print("  Shape:", df.shape)
print("  Unique participants:", df["SEQN"].nunique())

# ------------------------------------------------------------
# Reconstruct participant-level split
# Same deterministic seed/governance used in NB3
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split

participant_ids = df["SEQN"].dropna().unique()

train_participants, test_participants = train_test_split(
    participant_ids,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

train_participants = set(train_participants)
test_participants = set(test_participants)

# ------------------------------------------------------------
# Construct held-out test cohort
# ------------------------------------------------------------
test_df = df[df["SEQN"].isin(test_participants)].copy()

train_df = df[df["SEQN"].isin(train_participants)].copy()

# ------------------------------------------------------------
# Verify participant separation
# ------------------------------------------------------------
overlap = train_participants.intersection(test_participants)

print("\nParticipant-level split:")
print("  Training participants:", len(train_participants))
print("  Testing participants:", len(test_participants))
print("  Participant overlap:", len(overlap))

print("\nRow-level split:")
print("  Training rows:", len(train_df))
print("  Testing rows:", len(test_df))

print("\nTest target distribution:")
print(test_df["diabetes_target"].value_counts().sort_index())

print("\nTest target prevalence:")
print(round(test_df["diabetes_target"].mean(), 6))

# ------------------------------------------------------------
# Save reconstructed cohort
# ------------------------------------------------------------
test_cohort_path = (
    Path("/content/nb7_clinical_decision_support")
    / "tables"
    / "nb7_exact_nb3_test_cohort.csv"
)

test_df.to_csv(test_cohort_path, index=False)

print("\nSaved:")
print(f"  {test_cohort_path}")

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------
checks = {
    "dataset_shape_correct": df.shape == (31469, 244),
    "training_participants_correct": len(train_participants) == 11755,
    "testing_participants_correct": len(test_participants) == 2939,
    "training_rows_correct": len(train_df) == 25227,
    "testing_rows_correct": len(test_df) == 6242,
    "participant_overlap_zero": len(overlap) == 0,
    "target_present": "diabetes_target" in test_df.columns,
    "seqn_present": "SEQN" in test_df.columns,
}

print("\nValidation checks:")
for name, result in checks.items():
    print(f"  {name}: {'PASS' if result else 'FAIL'}")

print("\nOverall:", "✓ ALL CHECKS PASSED" if all(checks.values())
      else "⚠ REVIEW REQUIRED")

print("\n" + "=" * 70)
print("TEST COHORT RECONSTRUCTION COMPLETE")
print("=" * 70)

NB7 — Exact NB3 Held-Out Test Cohort Reconstruction

Full dataset:
  Shape: (31469, 244)
  Unique participants: 14694

Participant-level split:
  Training participants: 11755
  Testing participants: 2939
  Participant overlap: 0

Row-level split:
  Training rows: 25083
  Testing rows: 6386

Test target distribution:
diabetes_target
0    4584
1    1802
Name: count, dtype: int64

Test target prevalence:
0.28218

Saved:
  /content/nb7_clinical_decision_support/tables/nb7_exact_nb3_test_cohort.csv

Validation checks:
  dataset_shape_correct: PASS
  training_participants_correct: PASS
  testing_participants_correct: PASS
  training_rows_correct: FAIL
  testing_rows_correct: FAIL
  participant_overlap_zero: PASS
  target_present: PASS
  seqn_present: PASS

Overall: ⚠ REVIEW REQUIRED

TEST COHORT RECONSTRUCTION COMPLETE


In [23]:
# ============================================================
# CELL 11 — NB3 Cohort Reconstruction Diagnostic
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 11 — NB3 COHORT RECONSTRUCTION DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate all imported NB3 metadata/artifacts
# ------------------------------------------------------------

base_path = Path("/content/nb7_clinical_decision_support")
imported_path = base_path / "imported_artifacts"

print("\n[1] Searching imported NB3 artifacts...")

metadata_files = list(imported_path.rglob("*.json"))
csv_files = list(imported_path.rglob("*.csv"))

print(f"JSON files found: {len(metadata_files)}")
print(f"CSV files found:  {len(csv_files)}")

for p in metadata_files:
    print("  JSON:", p)

# ------------------------------------------------------------
# 2. Inspect NB3 metadata files
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[2] Inspecting NB3 metadata")
print("-" * 70)

nb3_metadata = []

for path in metadata_files:
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        text = json.dumps(data).lower()

        if (
            "threshold" in text
            or "training" in text
            or "testing" in text
            or "participant" in text
            or "early" in text
        ):
            nb3_metadata.append((path, data))

            print(f"\nFILE: {path}")

            if isinstance(data, dict):
                for key, value in data.items():
                    print(f"  {key}: {value}")

    except Exception as e:
        print(f"Could not read {path}: {e}")

# ------------------------------------------------------------
# 3. Search CSV artifacts for SEQN / prediction information
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[3] Inspecting NB3 CSV artifacts")
print("-" * 70)

for path in csv_files:
    try:
        sample = pd.read_csv(path, nrows=5)

        cols = list(sample.columns)

        relevant = any(
            term in str(cols).lower()
            for term in [
                "seqn",
                "y_true",
                "predicted_probability",
                "participant",
                "threshold"
            ]
        )

        if relevant:
            print(f"\nFILE: {path}")
            print("  Columns:", cols)
            print("  Sample shape:", sample.shape)

    except Exception as e:
        print(f"Could not inspect {path}: {e}")

# ------------------------------------------------------------
# 4. Compare current reconstructed cohort with locked NB3
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[4] Current reconstruction summary")
print("-" * 70)

current_test = pd.read_csv(
    base_path / "tables" / "nb7_exact_nb3_test_cohort.csv"
)

print("Current reconstructed test rows:", len(current_test))
print("Current reconstructed participants:", current_test["SEQN"].nunique())

print("\nCurrent target distribution:")
print(current_test["diabetes_target"].value_counts().sort_index())

print("\nLocked NB3 target distribution:")
print("  Negative (0): 4577")
print("  Positive (1): 1665")
print("  Total:        6242")

# ------------------------------------------------------------
# 5. Check whether an exact NB3 test prediction CSV exists
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[5] Searching for exact NB3 held-out prediction rows")
print("-" * 70)

prediction_candidates = []

for path in csv_files:
    try:
        sample = pd.read_csv(path, nrows=5)
        cols_lower = [str(c).lower() for c in sample.columns]

        if (
            "y_true" in cols_lower
            and "predicted_probability" in cols_lower
        ):
            prediction_candidates.append(path)

            print(f"Candidate prediction file:")
            print(f"  {path}")

            full_sample = pd.read_csv(path)

            print("  Rows:", len(full_sample))
            print("  Columns:", list(full_sample.columns))

            if "y_true" in full_sample.columns:
                print("  y_true distribution:")
                print(
                    full_sample["y_true"]
                    .value_counts()
                    .sort_index()
                )

    except Exception as e:
        print(f"Could not inspect {path}: {e}")

# ------------------------------------------------------------
# 6. Diagnostic conclusion
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 11 DIAGNOSTIC COMPLETE")
print("=" * 70)

if len(prediction_candidates) > 0:
    print("\n✓ Exact NB3 prediction artifact(s) located.")
    print("These will be used to identify the exact 6,242-row held-out cohort.")
else:
    print("\n⚠ No exact NB3 prediction CSV was located.")

print("\nIMPORTANT:")
print("We are NOT modifying the cohort manually.")
print("We are NOT retraining the NB3 model.")
print("We are identifying the exact NB3 held-out rows first.")
print("=" * 70)

CELL 11 — NB3 COHORT RECONSTRUCTION DIAGNOSTIC

[1] Searching imported NB3 artifacts...
JSON files found: 3
CSV files found:  41
  JSON: /content/nb7_clinical_decision_support/imported_artifacts/notebook3_early_detection_exports/early_detection_model_metadata.json
  JSON: /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/final_report/nb6_final_status.json
  JSON: /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/notebook3_import/early_detection_model_metadata.json

----------------------------------------------------------------------
[2] Inspecting NB3 metadata
----------------------------------------------------------------------

FILE: /content/nb7_clinical_decision_support/imported_artifacts/notebook3_early_detection_exports/early_detection_model_metadata.json
  threshold_information: {'model': 'Early-Detection Logis

In [24]:
# ============================================================
# CELL 12 — EXACT NB3 TEST-ROW MATCHING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 12 — EXACT NB3 TEST-ROW MATCHING")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate authoritative NB3 prediction artifact
# ------------------------------------------------------------

base_path = Path("/content/nb7_clinical_decision_support")

prediction_path = (
    base_path
    / "imported_artifacts"
    / "notebook3_early_detection_exports"
    / "early_detection_test_predictions.csv"
)

dataset_path = Path("/content/diabetes_modeling_dataset_final.csv")

print("\n[1] Loading authoritative NB3 prediction artifact...")
print("Prediction file:", prediction_path)

nb3_pred = pd.read_csv(prediction_path)
df = pd.read_csv(dataset_path)

print("NB3 prediction rows:", len(nb3_pred))
print("NB3 prediction columns:", list(nb3_pred.columns))
print("Full dataset rows:", len(df))
print("Full dataset columns:", len(df.columns))

# ------------------------------------------------------------
# 2. Verify authoritative NB3 prediction distribution
# ------------------------------------------------------------

print("\n[2] NB3 prediction target distribution:")

print(
    nb3_pred["y_true"]
    .value_counts()
    .sort_index()
)

print("\nExpected:")
print("  y_true = 0: 4577")
print("  y_true = 1: 1665")
print("  Total:      6242")

# ------------------------------------------------------------
# 3. Identify the participant-level test pool
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split

participant_ids = df["SEQN"].dropna().unique()

train_participants, test_participants = train_test_split(
    participant_ids,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

test_pool = df[
    df["SEQN"].isin(set(test_participants))
].copy()

print("\n[3] Participant-level test pool:")
print("  Rows:", len(test_pool))
print("  Participants:", test_pool["SEQN"].nunique())

# ------------------------------------------------------------
# 4. Compare target sequences
# ------------------------------------------------------------

print("\n[4] Comparing target sequences...")

pool_y = test_pool["diabetes_target"].to_numpy()
nb3_y = nb3_pred["y_true"].to_numpy()

print("  NB3 prediction target length:", len(nb3_y))
print("  Test-pool target length:", len(pool_y))

print(
    "  Target sequences identical:",
    np.array_equal(pool_y, nb3_y)
)

# ------------------------------------------------------------
# 5. Locate exact ordered rows using target + prediction
# ------------------------------------------------------------
#
# The prediction CSV does not contain SEQN.
# Therefore, we first establish candidate rows from the
# participant-level test pool and use the authoritative
# prediction probabilities to identify the exact rows.
#
# We do NOT assume row order alone is sufficient.
# This diagnostic examines whether the NB3 probabilities can
# uniquely identify the corresponding rows.
# ------------------------------------------------------------

print("\n[5] Preparing row-level matching diagnostics...")

# Check whether any duplicate target rows exist in the pool.
duplicate_target_count = (
    test_pool["diabetes_target"]
    .value_counts()
    .to_dict()
)

print("  Test-pool target counts:")
print("   ", duplicate_target_count)

# ------------------------------------------------------------
# 6. Inspect exact prediction probability characteristics
# ------------------------------------------------------------

nb3_probs = nb3_pred["predicted_probability"].to_numpy()

print("\n[6] NB3 probability diagnostics:")
print("  Minimum:", float(np.min(nb3_probs)))
print("  Maximum:", float(np.max(nb3_probs)))
print("  Mean:", float(np.mean(nb3_probs)))
print("  Unique probabilities:", len(np.unique(nb3_probs)))
print(
    "  Duplicate probability values:",
    len(nb3_probs) - len(np.unique(nb3_probs))
)

# ------------------------------------------------------------
# 7. Check whether the current test pool can be aligned
#    to the authoritative prediction sequence by target.
# ------------------------------------------------------------

print("\n[7] Ordered target alignment diagnostic...")

if len(pool_y) == len(nb3_y):
    print("  ✓ Lengths match.")
else:
    print("  ⚠ Lengths differ by:", len(pool_y) - len(nb3_y))

# Compare cumulative target counts as a diagnostic.
pool_cumsum = np.cumsum(pool_y)
nb3_cumsum = np.cumsum(nb3_y)

if len(pool_cumsum) == len(nb3_cumsum):
    matching_positions = np.sum(pool_cumsum == nb3_cumsum)
    print(
        "  Cumulative target positions matching:",
        matching_positions,
        "/",
        len(nb3_y)
    )

    print(
        "  Cumulative target alignment:",
        np.array_equal(pool_cumsum, nb3_cumsum)
    )

# ------------------------------------------------------------
# 8. Save authoritative prediction artifact locally
# ------------------------------------------------------------

authoritative_copy = (
    base_path
    / "tables"
    / "nb7_authoritative_nb3_test_predictions.csv"
)

nb3_pred.to_csv(authoritative_copy, index=False)

print("\n[8] Saved authoritative NB3 predictions:")
print(authoritative_copy)

# ------------------------------------------------------------
# 9. Final diagnostic status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 12 DIAGNOSTIC COMPLETE")
print("=" * 70)

print("\nIMPORTANT:")
print("The authoritative 6,242-row NB3 prediction artifact has been")
print("identified and preserved.")
print("No rows have been manually deleted.")
print("No model has been retrained.")
print("No threshold has been changed.")

print("\nNext step:")
print("We will determine the exact SEQN mapping before constructing")
print("the NB7 clinical decision-support cohort.")

print("=" * 70)

CELL 12 — EXACT NB3 TEST-ROW MATCHING

[1] Loading authoritative NB3 prediction artifact...
Prediction file: /content/nb7_clinical_decision_support/imported_artifacts/notebook3_early_detection_exports/early_detection_test_predictions.csv
NB3 prediction rows: 6242
NB3 prediction columns: ['y_true', 'predicted_probability', 'predicted_class_threshold_0_35']
Full dataset rows: 31469
Full dataset columns: 244

[2] NB3 prediction target distribution:
y_true
0    4577
1    1665
Name: count, dtype: int64

Expected:
  y_true = 0: 4577
  y_true = 1: 1665
  Total:      6242

[3] Participant-level test pool:
  Rows: 6386
  Participants: 2939

[4] Comparing target sequences...
  NB3 prediction target length: 6242
  Test-pool target length: 6386
  Target sequences identical: False

[5] Preparing row-level matching diagnostics...
  Test-pool target counts:
    {0: 4584, 1: 1802}

[6] NB3 probability diagnostics:
  Minimum: 3.455279120217113e-15
  Maximum: 0.999999979943228
  Mean: 0.2595058692511726

In [25]:
# ============================================================
# CELL 13 — FIND EXACT NB6 SEQN MAPPING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 13 — FIND EXACT NB6 SEQN MAPPING")
print("=" * 70)

base_path = Path("/content/nb7_clinical_decision_support")
nb6_path = (
    base_path
    / "imported_artifacts"
    / "NB6_Explainability_Consistency_Final_Package"
)

print("\n[1] Searching NB6 package for SEQN-containing artifacts...")

all_csvs = list(nb6_path.rglob("*.csv"))

seqn_candidates = []

for path in all_csvs:
    try:
        sample = pd.read_csv(path, nrows=5)
        columns_lower = [str(c).lower() for c in sample.columns]

        if "seqn" in columns_lower:
            seqn_candidates.append(path)

    except Exception as e:
        pass

print(f"\nSEQN-containing CSV files found: {len(seqn_candidates)}")

for i, path in enumerate(seqn_candidates, start=1):
    print(f"{i}. {path}")

# ------------------------------------------------------------
# Inspect candidate files
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[2] Inspecting candidate SEQN artifacts")
print("-" * 70)

for path in seqn_candidates:
    try:
        data = pd.read_csv(path)

        print(f"\nFILE: {path}")
        print("  Shape:", data.shape)
        print("  Columns:", list(data.columns))
        print("  Unique SEQN:", data["SEQN"].nunique())
        print("  Duplicate SEQN rows:",
              int(data["SEQN"].duplicated().sum()))

        if "diabetes_target" in data.columns:
            print("  diabetes_target distribution:")
            print(
                data["diabetes_target"]
                .value_counts()
                .sort_index()
                .to_dict()
            )

        if "y_true" in data.columns:
            print("  y_true distribution:")
            print(
                data["y_true"]
                .value_counts()
                .sort_index()
                .to_dict()
            )

        if "predicted_probability" in data.columns:
            print(
                "  predicted_probability range:",
                float(data["predicted_probability"].min()),
                "to",
                float(data["predicted_probability"].max())
            )

    except Exception as e:
        print(f"  Could not inspect file: {e}")

# ------------------------------------------------------------
# Search all CSVs for likely patient-level explanation artifacts
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[3] Searching for patient-level explanation artifacts")
print("-" * 70)

keywords = [
    "patient",
    "explanation",
    "contribution",
    "prediction",
    "similarity",
    "stable"
]

keyword_candidates = []

for path in all_csvs:
    name = path.name.lower()

    if any(k in name for k in keywords):
        keyword_candidates.append(path)

print(f"Potential patient/explanation CSVs: {len(keyword_candidates)}")

for path in keyword_candidates:
    print(" ", path.name)

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 13 DIAGNOSTIC COMPLETE")
print("=" * 70)

if seqn_candidates:
    print("\n✓ SEQN-containing NB6 artifact(s) found.")
    print("These may provide the exact 6,242-row evaluation cohort.")
else:
    print("\n⚠ No SEQN-containing NB6 CSV was found.")

print("\nNo cohort rows have been deleted or manually altered.")
print("No model has been retrained.")
print("No threshold has been changed.")

print("=" * 70)

CELL 13 — FIND EXACT NB6 SEQN MAPPING

[1] Searching NB6 package for SEQN-containing artifacts...

SEQN-containing CSV files found: 0

----------------------------------------------------------------------
[2] Inspecting candidate SEQN artifacts
----------------------------------------------------------------------

----------------------------------------------------------------------
[3] Searching for patient-level explanation artifacts
----------------------------------------------------------------------
Potential patient/explanation CSVs: 30
  early_detection_oof_training_predictions.csv
  early_detection_test_predictions.csv
  nb6_multiscale_explanation_stability_figure_data.csv
  nb6_controlled_perturbation_explanation_stability.csv
  nb6_global_explanation_stability_baseline.csv
  nb6_explanation_consistency_by_confidence.csv
  nb6_explanation_consistency_within_prediction_group.csv
  nb6_integrated_explanation_consistency_dimension_summary.csv
  nb6_explanation_consistency_con

In [26]:
# ============================================================
# CELL 14 — INSPECT NB6 PATIENT-LEVEL IDENTIFIERS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 14 — INSPECT NB6 PATIENT-LEVEL IDENTIFIERS")
print("=" * 70)

base_path = Path("/content/nb7_clinical_decision_support")

nb6_path = (
    base_path
    / "imported_artifacts"
    / "NB6_Explainability_Consistency_Final_Package"
)

# ------------------------------------------------------------
# Relevant NB6 patient-level files
# ------------------------------------------------------------

candidate_names = [
    "nb6_patient_level_explanation_stability.csv",
    "nb6_patient_level_top10_explanation_consistency.csv",
    "nb6_patient_explanation_stability_summary.csv",
    "nb6_multiscale_perturbation_stability_patient_results.csv",
    "nb6_patient_similarity_vs_explanation_similarity.csv",
    "nb6_patient_similarity_explanation_correlations.csv",
    "nb6_pairwise_explanation_similarity.csv",
]

print("\n[1] Inspecting selected NB6 patient-level artifacts...")

found = []

for name in candidate_names:
    matches = list(nb6_path.rglob(name))

    if matches:
        for path in matches:
            found.append(path)
            print(f"\nFILE: {path}")

            try:
                data = pd.read_csv(path)

                print("  Shape:", data.shape)
                print("  Columns:")
                for col in data.columns:
                    print("   -", col)

                print("\n  First 3 rows:")
                print(data.head(3).to_string(index=False))

            except Exception as e:
                print("  ERROR:", e)

# ------------------------------------------------------------
# Search ALL NB6 CSV columns for identifier-like fields
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[2] Searching all NB6 CSVs for identifier-like columns")
print("-" * 70)

identifier_keywords = [
    "index",
    "patient",
    "row",
    "seqn",
    "id",
    "prob",
    "target",
    "true",
    "prediction"
]

identifier_hits = []

for path in nb6_path.rglob("*.csv"):

    try:
        data = pd.read_csv(path, nrows=3)

        matching_columns = [
            col for col in data.columns
            if any(
                keyword in str(col).lower()
                for keyword in identifier_keywords
            )
        ]

        if matching_columns:
            identifier_hits.append(
                (path, matching_columns, data.shape[1])
            )

    except Exception:
        pass

print(
    f"\nCSV files containing identifier/prediction-like columns: "
    f"{len(identifier_hits)}"
)

for path, columns, ncols in identifier_hits:
    print("\n", path.name)
    print("  Matching columns:", columns)
    print("  Total columns:", ncols)

# ------------------------------------------------------------
# Specifically inspect pairwise index columns
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[3] Inspecting pairwise explanation indices")
print("-" * 70)

pairwise_matches = list(
    nb6_path.rglob("nb6_pairwise_explanation_similarity.csv")
)

for path in pairwise_matches:

    pairwise = pd.read_csv(path)

    print("\nFILE:", path)
    print("Shape:", pairwise.shape)
    print("Columns:", list(pairwise.columns))

    print("\nFirst 5 rows:")
    print(pairwise.head().to_string(index=False))

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 14 DIAGNOSTIC COMPLETE")
print("=" * 70)

print(
    "\nThe purpose of this cell is to determine whether NB6 retained "
    "enough row-level indexing information to recover the exact "
    "6,242-row NB3 evaluation cohort."
)

print("\nNo model retraining.")
print("No threshold modification.")
print("No manual row deletion.")

print("=" * 70)

CELL 14 — INSPECT NB6 PATIENT-LEVEL IDENTIFIERS

[1] Inspecting selected NB6 patient-level artifacts...

FILE: /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/tables/nb6_patient_level_explanation_stability.csv
  Shape: (1000, 11)
  Columns:
   - patient_index
   - cosine_similarity
   - explanation_correlation
   - top10_overlap
   - mean_absolute_contribution_change
   - mean_relative_contribution_change
   - explanation_stability
   - cosine_stable
   - top10_stable
   - cosine_highly_stable
   - top10_highly_stable

  First 3 rows:
 patient_index  cosine_similarity  explanation_correlation  top10_overlap  mean_absolute_contribution_change  mean_relative_contribution_change explanation_stability  cosine_stable  top10_stable  cosine_highly_stable  top10_highly_stable
          4193           0.999959                 0.999959            0.9                           0.000906                       6.13

In [27]:
# ============================================================
# CELL 15 — LOCATE NB6 COHORT-RECONSTRUCTION LOGIC
# ============================================================

from pathlib import Path

print("=" * 70)
print("CELL 15 — LOCATE NB6 COHORT-RECONSTRUCTION LOGIC")
print("=" * 70)

base_path = Path("/content/nb7_clinical_decision_support")

nb6_path = (
    base_path
    / "imported_artifacts"
    / "NB6_Explainability_Consistency_Final_Package"
)

# ------------------------------------------------------------
# 1. Find notebook files inside NB6 package
# ------------------------------------------------------------

print("\n[1] Searching NB6 package for notebook files...")

notebook_files = []

for pattern in ["*.ipynb", "*.py", "*.txt"]:
    notebook_files.extend(nb6_path.rglob(pattern))

notebook_files = list(dict.fromkeys(notebook_files))

print(f"Potential source/notebook files found: {len(notebook_files)}")

for i, path in enumerate(notebook_files, start=1):
    print(f"{i}. {path}")

# ------------------------------------------------------------
# 2. Inspect filenames for cohort/split/reconstruction clues
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[2] Files with cohort/split/reconstruction keywords")
print("-" * 70)

keywords = [
    "notebook",
    "cohort",
    "split",
    "reconstruct",
    "test",
    "evaluation",
    "metadata"
]

for path in nb6_path.rglob("*"):
    if path.is_file():
        name = path.name.lower()

        if any(keyword in name for keyword in keywords):
            print(path)

# ------------------------------------------------------------
# 3. Check whether the actual NB6 notebook is present
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[3] Notebook availability")
print("-" * 70)

ipynb_files = list(nb6_path.rglob("*.ipynb"))

if ipynb_files:
    print(f"✓ NB6 notebook(s) found: {len(ipynb_files)}")

    for path in ipynb_files:
        print("\nNotebook:")
        print(path)

        try:
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"Size: {size_mb:.2f} MB")
        except Exception:
            pass

else:
    print("⚠ No .ipynb file found inside NB6 package.")

# ------------------------------------------------------------
# 4. Check for textual reconstruction documentation
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("[4] Searching text documentation for reconstruction clues")
print("-" * 70)

text_files = []

for path in nb6_path.rglob("*"):
    if path.is_file() and path.suffix.lower() in [
        ".txt", ".md", ".json"
    ]:
        text_files.append(path)

terms = [
    "6242",
    "31469",
    "2939",
    "25227",
    "participant",
    "test cohort",
    "held-out",
    "train_test_split",
    "random_state",
    "test set"
]

hits = []

for path in text_files:
    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        text_lower = text.lower()

        matched_terms = [
            term for term in terms
            if term.lower() in text_lower
        ]

        if matched_terms:
            hits.append((path, matched_terms))

    except Exception:
        pass

print(f"Documentation files containing relevant terms: {len(hits)}")

for path, matched_terms in hits:
    print("\nFILE:", path)
    print("Matched terms:", matched_terms)

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 15 DIAGNOSTIC COMPLETE")
print("=" * 70)

if ipynb_files:
    print("\n✓ Actual NB6 notebook is available.")
    print("We can inspect its exact cohort reconstruction logic.")
else:
    print("\n⚠ Actual NB6 notebook was not found in the package.")

print("\nNo data modification.")
print("No model retraining.")
print("No threshold modification.")

print("=" * 70)

CELL 15 — LOCATE NB6 COHORT-RECONSTRUCTION LOGIC

[1] Searching NB6 package for notebook files...
Potential source/notebook files found: 2
1. /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/final_report/nb6_research_ready_summary.txt
2. /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/tables/nb6_explanation_consistency_research_conclusion.txt

----------------------------------------------------------------------
[2] Files with cohort/split/reconstruction keywords
----------------------------------------------------------------------
/content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/notebook3_import/early_detection_model_metadata.json
/content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package

In [28]:
# ============================================================
# CELL 16 — INSPECT NB6 RECONSTRUCTION DOCUMENTATION
# ============================================================

from pathlib import Path

print("=" * 70)
print("CELL 16 — INSPECT NB6 RECONSTRUCTION DOCUMENTATION")
print("=" * 70)

base_path = Path("/content/nb7_clinical_decision_support")

nb6_path = (
    base_path
    / "imported_artifacts"
    / "NB6_Explainability_Consistency_Final_Package"
    / "nb6_explainability_consistency"
)

files_to_inspect = [
    nb6_path / "final_report" / "nb6_research_ready_summary.txt",
    nb6_path / "tables" / "nb6_explanation_consistency_research_conclusion.txt",
]

search_terms = [
    "6242",
    "2939",
    "25227",
    "participant",
    "held-out",
    "test cohort",
    "evaluation cohort",
    "reconstructed",
    "reconstruction",
    "SEQN",
    "row",
    "split",
    "train_test_split",
    "random_state",
]

for path in files_to_inspect:

    print("\n" + "=" * 70)
    print("FILE:", path)
    print("=" * 70)

    if not path.exists():
        print("⚠ File not found.")
        continue

    text = path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    lines = text.splitlines()

    print("Total lines:", len(lines))

    # Print lines containing relevant reconstruction terms,
    # plus nearby context.
    hit_indices = []

    for i, line in enumerate(lines):
        line_lower = line.lower()

        if any(term.lower() in line_lower for term in search_terms):
            hit_indices.append(i)

    # Remove duplicate/overlapping context ranges
    ranges = []

    for idx in hit_indices:
        start = max(0, idx - 3)
        end = min(len(lines), idx + 4)

        if not ranges or start > ranges[-1][1]:
            ranges.append([start, end])
        else:
            ranges[-1][1] = max(ranges[-1][1], end)

    if not ranges:
        print("No relevant reconstruction terms found.")
    else:
        print(f"\nRelevant sections found: {len(ranges)}")

        for rnum, (start, end) in enumerate(ranges, start=1):
            print(f"\n--- Section {rnum} ---")

            for i in range(start, end):
                print(f"{i + 1}: {lines[i]}")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 16 DIAGNOSTIC COMPLETE")
print("=" * 70)

print(
    "\nThis cell only reads documentation. "
    "No data, model, threshold, or cohort has been modified."
)

print("=" * 70)

CELL 16 — INSPECT NB6 RECONSTRUCTION DOCUMENTATION

FILE: /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/final_report/nb6_research_ready_summary.txt
Total lines: 148

Relevant sections found: 2

--- Section 1 ---
8: feature perturbations, and levels of patient-profile similarity.
9: 
10: EVALUATION SETTING
11: - Held-out evaluation cohort: 6,242 rows
12: - Unique participants: 2,939
13: - Locked predictive model: Logistic Regression
14: - Processed feature space: 517 features
15: - Prediction threshold: 0.35

--- Section 2 ---
103: OVERALL INTERPRETATION
104: ------------------------------------------------------------
105: 
106: Across the evaluated held-out cohort, the model demonstrated high
107: structural consistency of patient-level explanations across patient
108: pairs and strong stability under the specified controlled
109: perturbation protocol.

FILE: /content/nb7_clinical_decision_support

In [29]:
# ================================================================
# CELL 17 — CREATE NB7 EVIDENCE & PROVENANCE REGISTRY
# ================================================================

import os
import json
import pandas as pd
from datetime import datetime

BASE = "/content/nb7_clinical_decision_support"
TABLE_DIR = os.path.join(BASE, "tables")
REPORT_DIR = os.path.join(BASE, "final_report")

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("=" * 70)
print("CELL 17 — NB7 EVIDENCE & PROVENANCE REGISTRY")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Define authoritative artifacts
# ----------------------------------------------------------------

registry = [
    {
        "evidence_id": "NB3-MODEL",
        "source_stage": "NB3",
        "artifact": "early_detection_logistic_regression.joblib",
        "role": "Locked predictive model",
        "status": "AUTHORITATIVE",
        "patient_level_linkage": "Not available in current NB7 runtime",
        "notes": "No retraining permitted."
    },
    {
        "evidence_id": "NB3-PREPROCESSOR",
        "source_stage": "NB3",
        "artifact": "early_detection_preprocessor.joblib",
        "role": "Locked preprocessing pipeline",
        "status": "AUTHORITATIVE",
        "patient_level_linkage": "Not available in current NB7 runtime",
        "notes": "Must remain paired with locked NB3 model."
    },
    {
        "evidence_id": "NB3-METADATA",
        "source_stage": "NB3",
        "artifact": "early_detection_model_metadata.json",
        "role": "Model, cohort, threshold and evaluation metadata",
        "status": "AUTHORITATIVE",
        "patient_level_linkage": "Available",
        "notes": "Confirms threshold = 0.35 and held-out cohort = 6,242 rows / 2,939 participants."
    },
    {
        "evidence_id": "NB3-TEST-PREDICTIONS",
        "source_stage": "NB3",
        "artifact": "early_detection_test_predictions.csv",
        "role": "Authoritative held-out predictions",
        "status": "AUTHORITATIVE",
        "patient_level_linkage": "NO SEQN",
        "notes": "6,242 rows. Contains y_true, predicted_probability and locked-threshold class."
    },
    {
        "evidence_id": "NB6-PAIRWISE",
        "source_stage": "NB6",
        "artifact": "nb6_pairwise_explanation_similarity.csv",
        "role": "Patient-level explanation consistency evidence",
        "status": "AUTHORITATIVE",
        "patient_level_linkage": "Internal patient indices only",
        "notes": "Cannot safely map internal indices to SEQN without documented reconstruction logic."
    },
    {
        "evidence_id": "NB6-STABILITY",
        "source_stage": "NB6",
        "artifact": "nb6_patient_level_explanation_stability.csv",
        "role": "Patient-level perturbation stability evidence",
        "status": "AUTHORITATIVE",
        "patient_level_linkage": "Internal patient indices only",
        "notes": "1,000-patient perturbation sample."
    },
    {
        "evidence_id": "NB6-INTEGRATED",
        "source_stage": "NB6",
        "artifact": "nb6_integrated_explanation_consistency_evidence.csv",
        "role": "Integrated explanation-consistency evidence",
        "status": "AUTHORITATIVE",
        "patient_level_linkage": "Aggregate/model-level evidence",
        "notes": "Used as a model-level trustworthiness signal."
    },
    {
        "evidence_id": "NB5-TRUST",
        "source_stage": "NB5",
        "artifact": "nb5_integrated_trustworthiness_evidence.csv",
        "role": "Trust, fairness and safety evidence",
        "status": "AUTHORITATIVE",
        "patient_level_linkage": "Aggregate/group-level evidence",
        "notes": "Supports safety-aware CDS interpretation."
    },
    {
        "evidence_id": "NB7-FRESH-SPLIT",
        "source_stage": "NB7",
        "artifact": "nb7_exact_nb3_test_cohort.csv",
        "role": "Fresh participant-level split diagnostic",
        "status": "DIAGNOSTIC_ONLY",
        "patient_level_linkage": "SEQN available",
        "notes": "6,386 rows; NOT identical to NB3 held-out cohort and must not be used as an NB3 replacement."
    }
]

registry_df = pd.DataFrame(registry)

# ----------------------------------------------------------------
# 2. Add explicit cohort reconciliation record
# ----------------------------------------------------------------

reconciliation = {
    "nb3_authoritative_test_rows": 6242,
    "nb3_authoritative_test_participants": 2939,
    "nb7_fresh_participant_split_rows": 6386,
    "nb7_fresh_participant_split_participants": 2939,
    "row_difference": 144,
    "exact_cohort_reconstruction_status": "NOT_ESTABLISHED",
    "reason": (
        "NB3 test predictions contain no SEQN, while the documented "
        "NB6 package does not contain the original NB3 cohort-selection "
        "logic. Probability matching is unsafe because predicted "
        "probabilities are duplicated."
    ),
    "decision": (
        "Do not attach SEQN to authoritative NB3 predictions by inference. "
        "Use authoritative prediction/evidence artifacts for aggregate "
        "and explanation-consistency claims; any newly constructed "
        "patient-level demonstration cohort must be explicitly labelled "
        "as a prototype demonstration cohort."
    )
}

# ----------------------------------------------------------------
# 3. Save registry and reconciliation
# ----------------------------------------------------------------

registry_path = os.path.join(TABLE_DIR, "nb7_evidence_provenance_registry.csv")
reconciliation_path = os.path.join(TABLE_DIR, "nb7_cohort_reconciliation_status.json")

registry_df.to_csv(registry_path, index=False)

with open(reconciliation_path, "w") as f:
    json.dump(reconciliation, f, indent=2)

# ----------------------------------------------------------------
# 4. Research-readable provenance statement
# ----------------------------------------------------------------

provenance_text = f"""
NB7 EVIDENCE PROVENANCE AND COHORT RECONCILIATION
==================================================

Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

AUTHORITATIVE NB3 HELD-OUT EVALUATION
--------------------------------------
Rows: 6,242
Participants: 2,939
Threshold: 0.35
Prediction artifact: early_detection_test_predictions.csv

COHORT RECONSTRUCTION STATUS
-----------------------------
The original NB3 held-out cohort cannot currently be reconstructed
with exact SEQN-level linkage from the imported artifacts.

The authoritative NB3 prediction file contains 6,242 predictions but
does not contain SEQN. A fresh participant-level split of the complete
dataset produced 6,386 rows for the same 2,939 participants. Therefore,
the fresh split is not identical to the NB3 held-out cohort.

No probability-based matching was used because predicted probabilities
are not unique identifiers.

NB7 GOVERNANCE DECISION
-----------------------
1. The 6,242-row NB3 prediction artifact remains authoritative.
2. NB6 explanation-consistency evidence remains authoritative for the
   evaluation framework in which it was generated.
3. The fresh 6,386-row participant split is diagnostic only.
4. No inferred SEQN-to-prediction mapping will be created.
5. Any new patient-level CDS demonstration will be explicitly separated
   from the locked NB3 held-out evaluation cohort.
6. No model retraining or threshold modification is permitted.

This provenance decision is intended to prevent accidental leakage,
misalignment, or unsupported patient-level claims in the NB7 prototype.
"""

provenance_path = os.path.join(
    REPORT_DIR,
    "nb7_evidence_provenance_statement.txt"
)

with open(provenance_path, "w") as f:
    f.write(provenance_text.strip())

# ----------------------------------------------------------------
# 5. Print audit
# ----------------------------------------------------------------

print("\n[1] Evidence registry")
print("-" * 70)
print(registry_df[
    ["evidence_id", "source_stage", "status", "patient_level_linkage"]
].to_string(index=False))

print("\n[2] Cohort reconciliation")
print("-" * 70)
for k, v in reconciliation.items():
    print(f"{k}: {v}")

print("\n[3] Files saved")
print("-" * 70)
print(registry_path)
print(reconciliation_path)
print(provenance_path)

print("\n" + "=" * 70)
print("CELL 17 COMPLETE")
print("NB7 provenance layer created.")
print("No data modification to authoritative NB3/NB6 artifacts.")
print("No model retraining.")
print("No threshold modification.")
print("=" * 70)

CELL 17 — NB7 EVIDENCE & PROVENANCE REGISTRY

[1] Evidence registry
----------------------------------------------------------------------
         evidence_id source_stage          status                patient_level_linkage
           NB3-MODEL          NB3   AUTHORITATIVE Not available in current NB7 runtime
    NB3-PREPROCESSOR          NB3   AUTHORITATIVE Not available in current NB7 runtime
        NB3-METADATA          NB3   AUTHORITATIVE                            Available
NB3-TEST-PREDICTIONS          NB3   AUTHORITATIVE                              NO SEQN
        NB6-PAIRWISE          NB6   AUTHORITATIVE        Internal patient indices only
       NB6-STABILITY          NB6   AUTHORITATIVE        Internal patient indices only
      NB6-INTEGRATED          NB6   AUTHORITATIVE       Aggregate/model-level evidence
           NB5-TRUST          NB5   AUTHORITATIVE       Aggregate/group-level evidence
     NB7-FRESH-SPLIT          NB7 DIAGNOSTIC_ONLY                       SEQN a

In [30]:
# ================================================================
# CELL 18 — AUDIT LOCKED NB3 MODEL ARTIFACTS
# ================================================================

import os
from pathlib import Path

BASE = "/content/nb7_clinical_decision_support"
IMPORTED = os.path.join(BASE, "imported_artifacts")

print("=" * 70)
print("CELL 18 — AUDIT LOCKED NB3 MODEL ARTIFACTS")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Search for locked model/preprocessor artifacts
# ----------------------------------------------------------------

extensions = {".joblib", ".pkl", ".pickle"}

model_candidates = []
preprocessor_candidates = []
other_serialized = []

for root, dirs, files in os.walk(IMPORTED):
    for fname in files:
        path = os.path.join(root, fname)
        ext = Path(fname).suffix.lower()

        if ext not in extensions:
            continue

        lower = fname.lower()

        if "model" in lower or "logistic" in lower:
            model_candidates.append(path)
        elif "preprocess" in lower or "transform" in lower:
            preprocessor_candidates.append(path)
        else:
            other_serialized.append(path)

print("\n[1] Serialized artifacts found")
print("-" * 70)

print(f"Model candidates: {len(model_candidates)}")
for p in model_candidates:
    print("  ", p)

print(f"\nPreprocessor candidates: {len(preprocessor_candidates)}")
for p in preprocessor_candidates:
    print("  ", p)

print(f"\nOther serialized artifacts: {len(other_serialized)}")
for p in other_serialized:
    print("  ", p)

# ----------------------------------------------------------------
# 2. Explicitly check the expected NB3 package location
# ----------------------------------------------------------------

expected_dir = os.path.join(
    IMPORTED,
    "notebook3_early_detection_exports"
)

expected_model = os.path.join(
    expected_dir,
    "early_detection_logistic_regression.joblib"
)

expected_preprocessor = os.path.join(
    expected_dir,
    "early_detection_preprocessor.joblib"
)

print("\n[2] Expected NB3 artifact paths")
print("-" * 70)

print("Expected model:")
print(expected_model)
print("Exists:", os.path.exists(expected_model))

print("\nExpected preprocessor:")
print(expected_preprocessor)
print("Exists:", os.path.exists(expected_preprocessor))

# ----------------------------------------------------------------
# 3. Check file sizes
# ----------------------------------------------------------------

print("\n[3] Artifact sizes")
print("-" * 70)

for label, path in [
    ("Locked model", expected_model),
    ("Locked preprocessor", expected_preprocessor)
]:
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"{label}: {size:,} bytes")
    else:
        print(f"{label}: NOT FOUND")

# ----------------------------------------------------------------
# 4. Final status
# ----------------------------------------------------------------

model_available = os.path.exists(expected_model)
preprocessor_available = os.path.exists(expected_preprocessor)

print("\n" + "=" * 70)

if model_available and preprocessor_available:
    print("STATUS: LOCKED NB3 MODEL + PREPROCESSOR AVAILABLE")
    print("Next step can safely load and audit the exact artifacts.")
elif model_available or preprocessor_available:
    print("STATUS: PARTIAL NB3 MODEL ARTIFACTS AVAILABLE")
    print("Do NOT proceed to prediction until both artifacts are resolved.")
else:
    print("STATUS: LOCKED NB3 MODEL ARTIFACTS NOT AVAILABLE")
    print("No model loading or retraining will be attempted.")

print("=" * 70)

CELL 18 — AUDIT LOCKED NB3 MODEL ARTIFACTS

[1] Serialized artifacts found
----------------------------------------------------------------------
Model candidates: 2
   /content/nb7_clinical_decision_support/imported_artifacts/notebook3_early_detection_exports/early_detection_logistic_regression.joblib
   /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/notebook3_import/early_detection_logistic_regression.joblib

Preprocessor candidates: 2
   /content/nb7_clinical_decision_support/imported_artifacts/notebook3_early_detection_exports/early_detection_preprocessor.joblib
   /content/nb7_clinical_decision_support/imported_artifacts/NB6_Explainability_Consistency_Final_Package/nb6_explainability_consistency/notebook3_import/early_detection_preprocessor.joblib

Other serialized artifacts: 0

[2] Expected NB3 artifact paths
----------------------------------------------------------------------
Expected model:

In [31]:
# ================================================================
# CELL 19 — LOAD & VERIFY LOCKED NB3 MODEL + PREPROCESSOR
# ================================================================

import os
import joblib
import numpy as np

BASE = "/content/nb7_clinical_decision_support"
IMPORTED = os.path.join(BASE, "imported_artifacts")

PRIMARY_DIR = os.path.join(
    IMPORTED,
    "notebook3_early_detection_exports"
)

NB6_DIR = os.path.join(
    IMPORTED,
    "NB6_Explainability_Consistency_Final_Package",
    "nb6_explainability_consistency",
    "notebook3_import"
)

PRIMARY_MODEL_PATH = os.path.join(
    PRIMARY_DIR,
    "early_detection_logistic_regression.joblib"
)

PRIMARY_PREPROCESSOR_PATH = os.path.join(
    PRIMARY_DIR,
    "early_detection_preprocessor.joblib"
)

NB6_MODEL_PATH = os.path.join(
    NB6_DIR,
    "early_detection_logistic_regression.joblib"
)

NB6_PREPROCESSOR_PATH = os.path.join(
    NB6_DIR,
    "early_detection_preprocessor.joblib"
)

print("=" * 70)
print("CELL 19 — LOAD & VERIFY LOCKED NB3 MODEL + PREPROCESSOR")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load primary NB3 artifacts
# ----------------------------------------------------------------

model = joblib.load(PRIMARY_MODEL_PATH)
preprocessor = joblib.load(PRIMARY_PREPROCESSOR_PATH)

print("\n[1] Primary NB3 artifacts loaded")
print("-" * 70)

print("Model type:", type(model).__name__)
print("Preprocessor type:", type(preprocessor).__name__)

# ----------------------------------------------------------------
# 2. Inspect model structure
# ----------------------------------------------------------------

print("\n[2] Model audit")
print("-" * 70)

if hasattr(model, "coef_"):
    print("Coefficient shape:", model.coef_.shape)

if hasattr(model, "intercept_"):
    print("Intercept shape:", model.intercept_.shape)
    print("Intercept:", model.intercept_)

if hasattr(model, "classes_"):
    print("Classes:", model.classes_)

if hasattr(model, "n_features_in_"):
    print("Model n_features_in_:", model.n_features_in_)

print("Expected processed features from NB3:", 517)

# ----------------------------------------------------------------
# 3. Inspect preprocessor
# ----------------------------------------------------------------

print("\n[3] Preprocessor audit")
print("-" * 70)

if hasattr(preprocessor, "transformers_"):
    print("Transformer blocks:")
    for name, transformer, columns in preprocessor.transformers_:
        if transformer == "drop":
            print(f"  - {name}: DROP")
        elif transformer == "passthrough":
            try:
                n_cols = len(columns)
            except Exception:
                n_cols = "unknown"
            print(f"  - {name}: PASSTHROUGH | columns={n_cols}")
        else:
            try:
                n_cols = len(columns)
            except Exception:
                n_cols = "unknown"
            print(
                f"  - {name}: {type(transformer).__name__} "
                f"| input_columns={n_cols}"
            )

if hasattr(preprocessor, "n_features_in_"):
    print("Preprocessor n_features_in_:", preprocessor.n_features_in_)

# ----------------------------------------------------------------
# 4. Verify duplicate NB6 copies byte-for-byte
# ----------------------------------------------------------------

print("\n[4] Compare NB3 primary artifacts with NB6 copies")
print("-" * 70)

def compare_files(path_a, path_b):
    if not os.path.exists(path_a) or not os.path.exists(path_b):
        return False, "One or both files missing"

    with open(path_a, "rb") as fa:
        a = fa.read()

    with open(path_b, "rb") as fb:
        b = fb.read()

    return a == b, f"{len(a):,} vs {len(b):,} bytes"

model_identical, model_size_info = compare_files(
    PRIMARY_MODEL_PATH,
    NB6_MODEL_PATH
)

prep_identical, prep_size_info = compare_files(
    PRIMARY_PREPROCESSOR_PATH,
    NB6_PREPROCESSOR_PATH
)

print("Model byte-identical:", model_identical, "|", model_size_info)
print("Preprocessor byte-identical:", prep_identical, "|", prep_size_info)

# ----------------------------------------------------------------
# 5. Governance checks
# ----------------------------------------------------------------

checks = {
    "model_loaded": model is not None,
    "preprocessor_loaded": preprocessor is not None,
    "model_has_coefficients": hasattr(model, "coef_"),
    "model_has_517_processed_features": (
        hasattr(model, "n_features_in_")
        and model.n_features_in_ == 517
    ),
    "coefficient_width_517": (
        hasattr(model, "coef_")
        and model.coef_.shape[1] == 517
    ),
    "model_nb6_copy_identical": model_identical,
    "preprocessor_nb6_copy_identical": prep_identical,
}

print("\n[5] Governance checks")
print("-" * 70)

for key, value in checks.items():
    print(f"{key}: {value}")

all_pass = all(checks.values())

print("\n" + "=" * 70)

if all_pass:
    print("STATUS: LOCKED NB3 MODEL INTEGRITY VERIFIED")
    print("The primary NB3 model and preprocessor are safe to use.")
    print("No retraining or modification performed.")
else:
    print("STATUS: INTEGRITY CHECK REQUIRES REVIEW")
    print("Do NOT proceed to patient-level prediction yet.")

print("=" * 70)

CELL 19 — LOAD & VERIFY LOCKED NB3 MODEL + PREPROCESSOR

[1] Primary NB3 artifacts loaded
----------------------------------------------------------------------
Model type: LogisticRegression
Preprocessor type: ColumnTransformer

[2] Model audit
----------------------------------------------------------------------
Coefficient shape: (1, 517)
Intercept shape: (1,)
Intercept: [-0.00024179]
Classes: [0 1]
Model n_features_in_: 517
Expected processed features from NB3: 517

[3] Preprocessor audit
----------------------------------------------------------------------
Transformer blocks:
  - numeric: Pipeline | input_columns=209
  - categorical: Pipeline | input_columns=5
Preprocessor n_features_in_: 214

[4] Compare NB3 primary artifacts with NB6 copies
----------------------------------------------------------------------
Model byte-identical: True | 5,007 vs 5,007 bytes
Preprocessor byte-identical: True | 15,799 vs 15,799 bytes

[5] Governance checks
-------------------------------------

In [32]:
# ================================================================
# CELL 20 — VERIFY CDS INPUT SCHEMA AGAINST LOCKED PREPROCESSOR
# ================================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/nb7_clinical_decision_support"
DATA_PATH = "/content/diabetes_modeling_dataset_final.csv"

print("=" * 70)
print("CELL 20 — VERIFY CDS INPUT SCHEMA")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load modeling dataset
# ----------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("\n[1] Dataset audit")
print("-" * 70)
print("Dataset shape:", df.shape)
print("Unique participants:", df["SEQN"].nunique())

# ----------------------------------------------------------------
# 2. Extract exact raw columns expected by the locked preprocessor
# ----------------------------------------------------------------

numeric_columns = []
categorical_columns = []

for name, transformer, columns in preprocessor.transformers_:
    if transformer == "drop":
        continue

    if isinstance(columns, (list, tuple, np.ndarray, pd.Index)):
        cols = list(columns)
    else:
        cols = list(columns)

    if name == "numeric":
        numeric_columns.extend(cols)
    elif name == "categorical":
        categorical_columns.extend(cols)

expected_raw_features = numeric_columns + categorical_columns

print("\n[2] Locked preprocessor input schema")
print("-" * 70)
print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))
print("Total expected raw features:", len(expected_raw_features))

# ----------------------------------------------------------------
# 3. Compare against dataset
# ----------------------------------------------------------------

dataset_columns = set(df.columns)

missing_features = [
    col for col in expected_raw_features
    if col not in dataset_columns
]

available_features = [
    col for col in expected_raw_features
    if col in dataset_columns
]

print("\n[3] Dataset ↔️ preprocessor compatibility")
print("-" * 70)
print("Expected features:", len(expected_raw_features))
print("Available features:", len(available_features))
print("Missing features:", len(missing_features))

if missing_features:
    print("\nMissing feature names:")
    for col in missing_features:
        print("  -", col)

# ----------------------------------------------------------------
# 4. Check feature ordering
# ----------------------------------------------------------------

ordered_schema = list(preprocessor.feature_names_in_)

ordering_matches = (
    len(ordered_schema) == len(expected_raw_features)
    and ordered_schema == expected_raw_features
)

print("\n[4] Feature-order audit")
print("-" * 70)
print("Preprocessor feature_names_in_ length:", len(ordered_schema))
print("Expected raw schema length:", len(expected_raw_features))
print("Exact ordering match:", ordering_matches)

# ----------------------------------------------------------------
# 5. Check target and identifier are excluded from model inputs
# ----------------------------------------------------------------

forbidden_model_inputs = ["SEQN", "diabetes_target"]

forbidden_present = [
    col for col in forbidden_model_inputs
    if col in expected_raw_features
]

print("\n[5] Governance exclusion audit")
print("-" * 70)
print("Identifier/target fields found in model inputs:", forbidden_present)

# ----------------------------------------------------------------
# 6. Check data types
# ----------------------------------------------------------------

print("\n[6] Data-type audit")
print("-" * 70)

dtype_summary = []

for col in expected_raw_features:
    dtype_summary.append({
        "feature": col,
        "dtype": str(df[col].dtype),
        "missing_fraction": float(df[col].isna().mean())
    })

dtype_df = pd.DataFrame(dtype_summary)

print(dtype_df["dtype"].value_counts().to_string())

# ----------------------------------------------------------------
# 7. Check transformed dimensionality using a small sample
# ----------------------------------------------------------------

sample = df[expected_raw_features].head(10).copy()

try:
    transformed_sample = preprocessor.transform(sample)

    if hasattr(transformed_sample, "toarray"):
        transformed_sample = transformed_sample.toarray()

    transformed_shape = transformed_sample.shape

except Exception as e:
    transformed_shape = None
    print("\nTransformation error:", repr(e))

print("\n[7] Transformation audit")
print("-" * 70)
print("Sample input shape:", sample.shape)
print("Transformed sample shape:", transformed_shape)
print("Expected processed width:", model.n_features_in_)

# ----------------------------------------------------------------
# 8. Final governance checks
# ----------------------------------------------------------------

checks = {
    "dataset_loaded": len(df) > 0,
    "raw_feature_count_214": len(expected_raw_features) == 214,
    "all_raw_features_available": len(missing_features) == 0,
    "schema_order_matches": ordering_matches,
    "seqn_excluded": "SEQN" not in forbidden_present,
    "target_excluded": "diabetes_target" not in forbidden_present,
    "transformation_succeeded": transformed_shape is not None,
    "processed_width_517": (
        transformed_shape is not None
        and transformed_shape[1] == model.n_features_in_
        and model.n_features_in_ == 517
    )
}

print("\n[8] Governance checks")
print("-" * 70)

for key, value in checks.items():
    print(f"{key}: {value}")

all_pass = all(checks.values())

print("\n" + "=" * 70)

if all_pass:
    print("STATUS: CDS INPUT SCHEMA VERIFIED")
    print("The locked NB3 preprocessing pipeline accepts the dataset schema.")
else:
    print("STATUS: INPUT SCHEMA REQUIRES REVIEW")
    print("Do NOT build patient-level prediction logic yet.")

print("=" * 70)

CELL 20 — VERIFY CDS INPUT SCHEMA

[1] Dataset audit
----------------------------------------------------------------------
Dataset shape: (31469, 244)
Unique participants: 14694

[2] Locked preprocessor input schema
----------------------------------------------------------------------
Numeric features: 209
Categorical features: 5
Total expected raw features: 214

[3] Dataset ↔️ preprocessor compatibility
----------------------------------------------------------------------
Expected features: 214
Available features: 214
Missing features: 0

[4] Feature-order audit
----------------------------------------------------------------------
Preprocessor feature_names_in_ length: 214
Expected raw schema length: 214
Exact ordering match: False

[5] Governance exclusion audit
----------------------------------------------------------------------
Identifier/target fields found in model inputs: []

[6] Data-type audit
----------------------------------------------------------------------
dtype
f

In [33]:
# ================================================================
# CELL 21 — CORRECT FEATURE-SCHEMA ORDER AUDIT
# ================================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/nb7_clinical_decision_support"
TABLE_DIR = os.path.join(BASE, "tables")

print("=" * 70)
print("CELL 21 — CORRECT FEATURE-SCHEMA ORDER AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Reuse loaded dataset, model and preprocessor
# ----------------------------------------------------------------

print("\n[1] Loaded objects")
print("-" * 70)

print("Dataset:", df.shape)
print("Model:", type(model).__name__)
print("Preprocessor:", type(preprocessor).__name__)

# ----------------------------------------------------------------
# 2. Get the actual fitted input schema
# ----------------------------------------------------------------

fitted_schema = list(preprocessor.feature_names_in_)

print("\n[2] Fitted preprocessor schema")
print("-" * 70)
print("Number of fitted input features:", len(fitted_schema))
print("First 10 fitted features:")
for col in fitted_schema[:10]:
    print("  -", col)

print("\nLast 10 fitted features:")
for col in fitted_schema[-10:]:
    print("  -", col)

# ----------------------------------------------------------------
# 3. Verify exact feature-set equality
# ----------------------------------------------------------------

dataset_feature_set = set(df.columns)
fitted_feature_set = set(fitted_schema)

missing_from_dataset = [
    col for col in fitted_schema
    if col not in dataset_feature_set
]

extra_dataset_columns = [
    col for col in df.columns
    if col not in fitted_feature_set
]

print("\n[3] Feature-set equality")
print("-" * 70)

print("Fitted features:", len(fitted_schema))
print("Dataset columns:", len(df.columns))
print("Missing fitted features:", len(missing_from_dataset))
print("Dataset columns not used by preprocessor:", len(extra_dataset_columns))

if missing_from_dataset:
    print("\nMissing:")
    for col in missing_from_dataset:
        print("  -", col)

print("\nUnused dataset columns:")
for col in extra_dataset_columns:
    print("  -", col)

# ----------------------------------------------------------------
# 4. Explicitly construct the fitted-schema dataframe
# ----------------------------------------------------------------

schema_df = df.loc[:, fitted_schema].copy()

print("\n[4] Exact fitted-schema dataframe")
print("-" * 70)
print("Shape:", schema_df.shape)
print("Columns exactly equal to fitted schema:",
      list(schema_df.columns) == fitted_schema)

# ----------------------------------------------------------------
# 5. Verify column types against transformer assignments
# ----------------------------------------------------------------

numeric_set = set(numeric_columns)
categorical_set = set(categorical_columns)

numeric_not_float = [
    col for col in numeric_columns
    if not pd.api.types.is_numeric_dtype(df[col])
]

categorical_not_object = [
    col for col in categorical_columns
    if pd.api.types.is_numeric_dtype(df[col])
]

print("\n[5] Transformer/type compatibility")
print("-" * 70)
print("Numeric columns expected:", len(numeric_columns))
print("Numeric columns with non-numeric dtype:", len(numeric_not_float))
print("Categorical columns expected:", len(categorical_columns))
print("Categorical columns with numeric dtype:", len(categorical_not_object))

# ----------------------------------------------------------------
# 6. Transform exact fitted-schema dataframe
# ----------------------------------------------------------------

try:
    transformed_exact = preprocessor.transform(schema_df)

    if hasattr(transformed_exact, "toarray"):
        transformed_exact = transformed_exact.toarray()

    transformation_ok = True
    transformed_shape = transformed_exact.shape

except Exception as e:
    transformation_ok = False
    transformed_shape = None
    print("\nTransformation error:")
    print(repr(e))

print("\n[6] Exact-schema transformation")
print("-" * 70)
print("Transformation successful:", transformation_ok)
print("Transformed shape:", transformed_shape)
print("Expected processed features:", model.n_features_in_)

# ----------------------------------------------------------------
# 7. Verify prediction path on a tiny sample
# ----------------------------------------------------------------

prediction_ok = False
prediction_shape = None

if transformation_ok:
    try:
        tiny_X = schema_df.head(5)
        tiny_prob = model.predict_proba(
            preprocessor.transform(tiny_X)
        )[:, 1]

        prediction_shape = tiny_prob.shape
        prediction_ok = (
            len(tiny_prob) == 5
            and np.all(np.isfinite(tiny_prob))
            and np.all((tiny_prob >= 0) & (tiny_prob <= 1))
        )

    except Exception as e:
        print("\nPrediction-path error:")
        print(repr(e))

print("\n[7] Locked prediction-path smoke test")
print("-" * 70)
print("Prediction successful:", prediction_ok)
print("Probability output shape:", prediction_shape)

if prediction_ok:
    print("Sample probabilities:")
    for i, p in enumerate(tiny_prob, start=1):
        print(f"  Patient {i}: {p:.8f}")

# ----------------------------------------------------------------
# 8. Save corrected schema audit
# ----------------------------------------------------------------

schema_audit = pd.DataFrame({
    "feature": fitted_schema,
    "dtype": [str(df[col].dtype) for col in fitted_schema],
    "transformer_role": [
        "numeric" if col in numeric_set else "categorical"
        for col in fitted_schema
    ],
    "missing_fraction": [
        float(df[col].isna().mean())
        for col in fitted_schema
    ]
})

schema_audit_path = os.path.join(
    TABLE_DIR,
    "nb7_locked_preprocessor_input_schema.csv"
)

schema_audit.to_csv(schema_audit_path, index=False)

# ----------------------------------------------------------------
# 9. Final governance checks
# ----------------------------------------------------------------

checks = {
    "fitted_schema_214": len(fitted_schema) == 214,
    "all_fitted_features_present": len(missing_from_dataset) == 0,
    "feature_set_exact": (
        len(missing_from_dataset) == 0
        and len(extra_dataset_columns) == 30
    ),
    "numeric_types_valid": len(numeric_not_float) == 0,
    "categorical_types_valid": len(categorical_not_object) == 0,
    "transformation_517": (
        transformation_ok
        and transformed_shape[1] == 517
    ),
    "prediction_smoke_test": prediction_ok
}

print("\n[8] Corrected governance checks")
print("-" * 70)

for key, value in checks.items():
    print(f"{key}: {value}")

all_pass = all(checks.values())

print("\n" + "=" * 70)

if all_pass:
    print("STATUS: LOCKED CDS INPUT SCHEMA VERIFIED")
    print("The fitted NB3 preprocessing schema is compatible with the dataset.")
    print("The locked model successfully accepts the transformed input.")
    print("No retraining or modification performed.")
else:
    print("STATUS: SCHEMA REQUIRES REVIEW")
    print("Do NOT proceed to full CDS prediction yet.")

print("=" * 70)

CELL 21 — CORRECT FEATURE-SCHEMA ORDER AUDIT

[1] Loaded objects
----------------------------------------------------------------------
Dataset: (31469, 244)
Model: LogisticRegression
Preprocessor: ColumnTransformer

[2] Fitted preprocessor schema
----------------------------------------------------------------------
Number of fitted input features: 214
First 10 fitted features:
  - SDDSRVYR
  - RIDSTATR
  - RIAGENDR
  - RIDAGEYR
  - RIDRETH1
  - RIDRETH3
  - RIDEXMON
  - DMDBORN4
  - DMDYRUSZ
  - DMDEDUC2

Last 10 fitted features:
  - HIQ032B
  - HIQ032D
  - HIQ105
  - HIQ270
  - HIQ210
  - HUQ010
  - HUQ030
  - HUQ051
  - HUQ071
  - HUQ090

[3] Feature-set equality
----------------------------------------------------------------------
Fitted features: 214
Dataset columns: 244
Missing fitted features: 0
Dataset columns not used by preprocessor: 30

Unused dataset columns:
  - SEQN
  - WTINTPRP
  - WTMECPRP
  - SDMVPSU
  - SDMVSTRA
  - WTSAFPRP_x
  - WTSAFPRP_y
  - DID040
  - DID060
  

In [34]:
# ================================================================
# CELL 22 — CREATE LOCKED-MODEL CDS PATIENT DEMONSTRATION
# ================================================================

import os
import json
import numpy as np
import pandas as pd

BASE = "/content/nb7_clinical_decision_support"
TABLE_DIR = os.path.join(BASE, "tables")

THRESHOLD = 0.35

print("=" * 70)
print("CELL 22 — LOCKED-MODEL CDS PATIENT DEMONSTRATION")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Select a deterministic prototype participant
# ----------------------------------------------------------------
# We intentionally use a deterministic participant from the full
# modeling dataset. This is NOT claimed to be part of the
# authoritative NB3 6,242-row held-out cohort.

participant_ids = (
    df["SEQN"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

prototype_seqn = participant_ids.iloc[0]

patient_rows = df[df["SEQN"] == prototype_seqn].copy()

print("\n[1] Prototype participant")
print("-" * 70)
print("Prototype SEQN:", prototype_seqn)
print("Rows belonging to participant:", len(patient_rows))
print("Dataset rows for this participant:")
print(patient_rows[["SEQN", "diabetes_target"]].to_string(index=False))

# ----------------------------------------------------------------
# 2. Check participant consistency
# ----------------------------------------------------------------

unique_targets = patient_rows["diabetes_target"].dropna().unique()

print("\n[2] Participant target audit")
print("-" * 70)
print("Unique non-missing diabetes_target values:", unique_targets)

# ----------------------------------------------------------------
# 3. Build model input using EXACT fitted schema
# ----------------------------------------------------------------

patient_input = patient_rows.loc[:, fitted_schema].copy()

print("\n[3] Patient model input")
print("-" * 70)
print("Input shape:", patient_input.shape)
print("Expected raw features:", len(fitted_schema))
print("Missing required features:",
      int(patient_input.isna().all(axis=0).sum()))

# ----------------------------------------------------------------
# 4. Transform using locked NB3 preprocessor
# ----------------------------------------------------------------

patient_transformed = preprocessor.transform(patient_input)

if hasattr(patient_transformed, "toarray"):
    patient_transformed = patient_transformed.toarray()

print("\n[4] Locked preprocessing")
print("-" * 70)
print("Transformed shape:", patient_transformed.shape)
print("Expected processed features:", model.n_features_in_)

# ----------------------------------------------------------------
# 5. Generate locked-model predictions
# ----------------------------------------------------------------

patient_probability = model.predict_proba(patient_transformed)[:, 1]

patient_prediction = (
    patient_probability >= THRESHOLD
).astype(int)

prediction_df = pd.DataFrame({
    "SEQN": patient_rows["SEQN"].values,
    "diabetes_target": patient_rows["diabetes_target"].values,
    "predicted_probability": patient_probability,
    "predicted_class_threshold_0_35": patient_prediction
})

print("\n[5] Locked-model prediction")
print("-" * 70)
print(prediction_df.to_string(index=False))

# ----------------------------------------------------------------
# 6. Check whether repeated rows for this participant produce
#    consistent predictions
# ----------------------------------------------------------------

probability_range = (
    float(patient_probability.max() - patient_probability.min())
    if len(patient_probability) > 0
    else np.nan
)

class_unique = np.unique(patient_prediction)

print("\n[6] Within-participant prediction consistency")
print("-" * 70)
print("Probability minimum:", float(patient_probability.min()))
print("Probability maximum:", float(patient_probability.max()))
print("Probability range:", probability_range)
print("Unique predicted classes:", class_unique)

# ----------------------------------------------------------------
# 7. Create prototype CDS prediction record
# ----------------------------------------------------------------

if len(class_unique) == 1:
    locked_class = int(class_unique[0])
    consistency_status = "CONSISTENT_ACROSS_PARTICIPANT_ROWS"
else:
    locked_class = None
    consistency_status = "ROW_LEVEL_PREDICTION_VARIATION"

prototype_record = {
    "prototype_status": "NB7_PROTOTYPE_DEMONSTRATION",
    "evaluation_cohort_status": (
        "NOT_CLAIMED_AS_NB3_HELD_OUT_COHORT"
    ),
    "SEQN": str(prototype_seqn),
    "participant_rows": int(len(patient_rows)),
    "locked_model": "NB3 Early-Detection Logistic Regression",
    "threshold": THRESHOLD,
    "threshold_status": "LOCKED",
    "mean_predicted_probability": float(patient_probability.mean()),
    "min_predicted_probability": float(patient_probability.min()),
    "max_predicted_probability": float(patient_probability.max()),
    "probability_range": probability_range,
    "prediction_class": locked_class,
    "within_participant_consistency": consistency_status,
    "model_retrained": False,
    "model_modified": False,
    "threshold_modified": False
}

# ----------------------------------------------------------------
# 8. Save artifacts
# ----------------------------------------------------------------

prediction_path = os.path.join(
    TABLE_DIR,
    "nb7_prototype_patient_prediction.csv"
)

record_path = os.path.join(
    TABLE_DIR,
    "nb7_prototype_patient_prediction_metadata.json"
)

prediction_df.to_csv(prediction_path, index=False)

with open(record_path, "w") as f:
    json.dump(prototype_record, f, indent=2)

# ----------------------------------------------------------------
# 9. Final governance checks
# ----------------------------------------------------------------

checks = {
    "prototype_participant_exists": len(patient_rows) > 0,
    "raw_input_width_214": patient_input.shape[1] == 214,
    "processed_width_517": patient_transformed.shape[1] == 517,
    "probabilities_finite": bool(np.all(np.isfinite(patient_probability))),
    "probabilities_valid": bool(
        np.all((patient_probability >= 0) &
               (patient_probability <= 1))
    ),
    "threshold_locked": THRESHOLD == 0.35,
    "model_not_retrained": prototype_record["model_retrained"] is False,
    "model_not_modified": prototype_record["model_modified"] is False,
    "threshold_not_modified": (
        prototype_record["threshold_modified"] is False
    )
}

print("\n[7] Governance checks")
print("-" * 70)

for key, value in checks.items():
    print(f"{key}: {value}")

all_pass = all(checks.values())

print("\n[8] Saved artifacts")
print("-" * 70)
print(prediction_path)
print(record_path)

print("\n" + "=" * 70)

if all_pass:
    print("STATUS: CDS PROTOTYPE PREDICTION CREATED")
    print("Locked NB3 model used without retraining.")
    print("Threshold 0.35 preserved.")
    print("Prototype is explicitly separated from NB3 evaluation cohort.")
else:
    print("STATUS: PROTOTYPE PREDICTION REQUIRES REVIEW")

print("=" * 70)

CELL 22 — LOCKED-MODEL CDS PATIENT DEMONSTRATION

[1] Prototype participant
----------------------------------------------------------------------
Prototype SEQN: 109263.0
Rows belonging to participant: 1
Dataset rows for this participant:
    SEQN  diabetes_target
109263.0                0

[2] Participant target audit
----------------------------------------------------------------------
Unique non-missing diabetes_target values: [0]

[3] Patient model input
----------------------------------------------------------------------
Input shape: (1, 214)
Expected raw features: 214
Missing required features: 185

[4] Locked preprocessing
----------------------------------------------------------------------
Transformed shape: (1, 517)
Expected processed features: 517

[5] Locked-model prediction
----------------------------------------------------------------------
    SEQN  diabetes_target  predicted_probability  predicted_class_threshold_0_35
109263.0                0               0.017

In [35]:
# ================================================================
# CELL 23 — PATIENT INPUT QUALITY & CLINICAL COMPLETENESS AUDIT
# ================================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 23 — PATIENT INPUT QUALITY & CLINICAL COMPLETENESS AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Recover prototype participant
# ----------------------------------------------------------------
prototype_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_patient_prediction.csv"
)

prototype_df = pd.read_csv(prototype_path)

prototype_seqn = prototype_df["SEQN"].iloc[0]

print("\n[1] Prototype participant")
print("-" * 70)
print("SEQN:", prototype_seqn)

# ----------------------------------------------------------------
# 2. Load exact dataset and fitted preprocessing schema
# ----------------------------------------------------------------
data_path = "/content/diabetes_modeling_dataset_final.csv"

df = pd.read_csv(data_path, low_memory=False)

# Recover the exact fitted raw-feature schema from Cell 21
schema_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_locked_preprocessor_input_schema.csv"
)

schema_df = pd.read_csv(schema_path)

required_features = schema_df["feature"].tolist()

print("\n[2] Dataset / fitted schema")
print("-" * 70)
print("Dataset shape:", df.shape)
print("Required raw features:", len(required_features))

# ----------------------------------------------------------------
# 3. Extract prototype row
# ----------------------------------------------------------------
participant_rows = df[df["SEQN"] == prototype_seqn].copy()

if participant_rows.empty:
    raise ValueError("Prototype participant not found in dataset.")

# This prototype currently has one row, but preserve the logic
# for multiple-row participants.
prototype_row = participant_rows.iloc[0]

# ----------------------------------------------------------------
# 4. Calculate raw input completeness
# ----------------------------------------------------------------
available_features = []
missing_features = []

for feature in required_features:
    value = prototype_row[feature]

    if pd.isna(value):
        missing_features.append(feature)
    else:
        available_features.append(feature)

n_available = len(available_features)
n_missing = len(missing_features)
n_required = len(required_features)

completeness_pct = 100 * n_available / n_required

print("\n[3] Raw feature completeness")
print("-" * 70)
print("Required features:", n_required)
print("Available features:", n_available)
print("Missing features:", n_missing)
print(f"Completeness: {completeness_pct:.2f}%")

# ----------------------------------------------------------------
# 5. Missingness by feature type
# ----------------------------------------------------------------
numeric_features = schema_df.loc[
    schema_df["dtype"].astype(str).str.contains(
        "float|int|number", case=False, regex=True, na=False
    ),
    "feature"
].tolist()

categorical_features = [
    f for f in required_features if f not in numeric_features
]

numeric_missing = [
    f for f in numeric_features
    if pd.isna(prototype_row[f])
]

categorical_missing = [
    f for f in categorical_features
    if pd.isna(prototype_row[f])
]

print("\n[4] Missingness by fitted feature type")
print("-" * 70)
print("Numeric features:", len(numeric_features))
print("Numeric missing:", len(numeric_missing))
print("Categorical features:", len(categorical_features))
print("Categorical missing:", len(categorical_missing))

# ----------------------------------------------------------------
# 6. Clinical measurement audit
# ----------------------------------------------------------------
clinical_measurements = [
    "LBXGLU",
    "LBDGLUSI",
    "LBXGH",
    "URXUMA",
    "URXUMS",
    "URXUCR",
    "URXCRS"
]

clinical_audit = []

for feature in clinical_measurements:
    if feature in df.columns:
        value = prototype_row[feature]
        clinical_audit.append({
            "feature": feature,
            "available": not pd.isna(value),
            "value": value if not pd.isna(value) else np.nan
        })

clinical_audit_df = pd.DataFrame(clinical_audit)

print("\n[5] Prediction-time clinical measurement audit")
print("-" * 70)

if not clinical_audit_df.empty:
    print(clinical_audit_df.to_string(index=False))
else:
    print("No specified clinical measurement fields found.")

# ----------------------------------------------------------------
# 7. Key demographic / anthropometric input audit
# ----------------------------------------------------------------
priority_features = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "BMXBMI",
    "BMXWAIST",
    "BMXHIP",
    "LBXGLU",
    "LBXGH",
    "LBXSCH",
    "LBXSGL",
    "LBXSOSSI",
    "LBXMCVSI"
]

priority_audit = []

for feature in priority_features:
    if feature in df.columns and feature in required_features:
        value = prototype_row[feature]
        priority_audit.append({
            "feature": feature,
            "available": not pd.isna(value),
            "value": value if not pd.isna(value) else np.nan
        })

priority_audit_df = pd.DataFrame(priority_audit)

print("\n[6] Priority clinical / demographic input audit")
print("-" * 70)

if not priority_audit_df.empty:
    print(priority_audit_df.to_string(index=False))
else:
    print("No priority features found.")

# ----------------------------------------------------------------
# 8. Missing-feature list
# ----------------------------------------------------------------
print("\n[7] Missing raw features")
print("-" * 70)

if missing_features:
    print("Total missing:", len(missing_features))
    print(missing_features)
else:
    print("No missing raw features.")

# ----------------------------------------------------------------
# 9. Prototype input-quality classification
# ----------------------------------------------------------------
if completeness_pct >= 90:
    quality_class = "HIGH"
elif completeness_pct >= 75:
    quality_class = "MODERATE"
elif completeness_pct >= 50:
    quality_class = "LOW"
else:
    quality_class = "VERY_LOW"

print("\n[8] Input-quality classification")
print("-" * 70)
print("Prototype completeness:", f"{completeness_pct:.2f}%")
print("Research input-quality class:", quality_class)

# ----------------------------------------------------------------
# 10. Governance interpretation
# ----------------------------------------------------------------
governance = {
    "prototype_only": True,
    "locked_model_used": True,
    "threshold_locked": True,
    "prediction_valid_under_pipeline": True,
    "clinical_completeness_high": completeness_pct >= 90,
    "clinical_completeness_requires_review": completeness_pct < 90,
    "should_not_be_interpreted_as_clinically_complete": completeness_pct < 90
}

print("\n[9] Governance interpretation")
print("-" * 70)

for k, v in governance.items():
    print(f"{k}: {v}")

# ----------------------------------------------------------------
# 11. Save audit artifacts
# ----------------------------------------------------------------
output_dir = "/content/nb7_clinical_decision_support/tables"

audit_path = os.path.join(
    output_dir,
    "nb7_prototype_input_quality_audit.csv"
)

clinical_path = os.path.join(
    output_dir,
    "nb7_prototype_clinical_measurement_audit.csv"
)

metadata_path = os.path.join(
    output_dir,
    "nb7_prototype_input_quality_metadata.json"
)

audit_summary = pd.DataFrame([{
    "SEQN": prototype_seqn,
    "required_features": n_required,
    "available_features": n_available,
    "missing_features": n_missing,
    "completeness_percent": completeness_pct,
    "numeric_features": len(numeric_features),
    "numeric_missing": len(numeric_missing),
    "categorical_features": len(categorical_features),
    "categorical_missing": len(categorical_missing),
    "input_quality_class": quality_class,
    "prediction_valid_under_locked_pipeline": True,
    "clinical_completeness_requires_review": completeness_pct < 90
}])

audit_summary.to_csv(audit_path, index=False)
clinical_audit_df.to_csv(clinical_path, index=False)

with open(metadata_path, "w") as f:
    json.dump(
        {
            "prototype_type": "NB7_PROTOTYPE_DEMONSTRATION",
            "SEQN": float(prototype_seqn),
            "required_raw_features": n_required,
            "available_raw_features": n_available,
            "missing_raw_features": n_missing,
            "completeness_percent": completeness_pct,
            "input_quality_class": quality_class,
            "governance": governance
        },
        f,
        indent=2,
        default=str
    )

print("\n[10] Saved artifacts")
print("-" * 70)
print(audit_path)
print(clinical_path)
print(metadata_path)

print("\n" + "=" * 70)
print("STATUS: PATIENT INPUT QUALITY AUDIT COMPLETED")
print("=" * 70)

CELL 23 — PATIENT INPUT QUALITY & CLINICAL COMPLETENESS AUDIT

[1] Prototype participant
----------------------------------------------------------------------
SEQN: 109263.0

[2] Dataset / fitted schema
----------------------------------------------------------------------
Dataset shape: (31469, 244)
Required raw features: 214

[3] Raw feature completeness
----------------------------------------------------------------------
Required features: 214
Available features: 29
Missing features: 185
Completeness: 13.55%

[4] Missingness by fitted feature type
----------------------------------------------------------------------
Numeric features: 209
Numeric missing: 180
Categorical features: 5
Categorical missing: 5

[5] Prediction-time clinical measurement audit
----------------------------------------------------------------------
 feature  available  value
  LBXGLU      False    NaN
LBDGLUSI      False    NaN
   LBXGH      False    NaN
  URXUMA      False    NaN
  URXUMS      False    Na

In [36]:
# ================================================================
# CELL 24 — OBSERVED VS MISSING/IMPUTED EXPLANATION CONTRIBUTIONS
# ================================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 24 — OBSERVED VS MISSING/IMPUTED EXPLANATION CONTRIBUTIONS")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load locked model and preprocessor
# ----------------------------------------------------------------
model_path = (
    "/content/nb7_clinical_decision_support/"
    "imported_artifacts/notebook3_early_detection_exports/"
    "early_detection_logistic_regression.joblib"
)

preprocessor_path = (
    "/content/nb7_clinical_decision_support/"
    "imported_artifacts/notebook3_early_detection_exports/"
    "early_detection_preprocessor.joblib"
)

# Locate files robustly if the extracted directory differs
import glob

model_matches = glob.glob(
    "/content/nb7_clinical_decision_support/"
    "**/early_detection_logistic_regression.joblib",
    recursive=True
)

preprocessor_matches = glob.glob(
    "/content/nb7_clinical_decision_support/"
    "**/early_detection_preprocessor.joblib",
    recursive=True
)

if not model_matches or not preprocessor_matches:
    raise FileNotFoundError(
        "Locked NB3 model/preprocessor artifacts could not be located."
    )

model_path = model_matches[0]
preprocessor_path = preprocessor_matches[0]

import joblib

model = joblib.load(model_path)
preprocessor = joblib.load(preprocessor_path)

print("\n[1] Locked model artifacts")
print("-" * 70)
print("Model:", type(model).__name__)
print("Preprocessor:", type(preprocessor).__name__)
print("Coefficient count:", model.coef_.shape[1])

# ----------------------------------------------------------------
# 2. Load prototype and dataset
# ----------------------------------------------------------------
prediction_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_patient_prediction.csv"
)

prototype_prediction = pd.read_csv(prediction_path)
prototype_seqn = prototype_prediction["SEQN"].iloc[0]

df = pd.read_csv(
    "/content/diabetes_modeling_dataset_final.csv",
    low_memory=False
)

schema_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_locked_preprocessor_input_schema.csv"
)

schema_df = pd.read_csv(schema_path)
required_features = schema_df["feature"].tolist()

patient_df = df[df["SEQN"] == prototype_seqn].copy()

if patient_df.empty:
    raise ValueError("Prototype participant not found.")

patient_row = patient_df.iloc[[0]][required_features].copy()

# ----------------------------------------------------------------
# 3. Determine observed vs missing raw features
# ----------------------------------------------------------------
observed_raw = []
missing_raw = []

for feature in required_features:
    if pd.isna(patient_row.iloc[0][feature]):
        missing_raw.append(feature)
    else:
        observed_raw.append(feature)

print("\n[2] Raw patient-input status")
print("-" * 70)
print("Observed raw features:", len(observed_raw))
print("Missing raw features:", len(missing_raw))
print("Total raw features:", len(required_features))

# ----------------------------------------------------------------
# 4. Transform using locked preprocessor
# ----------------------------------------------------------------
X_transformed = preprocessor.transform(patient_row)

if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

X_transformed = np.asarray(X_transformed)

feature_names = preprocessor.get_feature_names_out()

if X_transformed.shape[1] != len(feature_names):
    raise ValueError(
        f"Feature-name mismatch: transformed={X_transformed.shape[1]}, "
        f"names={len(feature_names)}"
    )

# ----------------------------------------------------------------
# 5. Calculate exact logistic contributions
# ----------------------------------------------------------------
coefficients = model.coef_[0]
intercept = float(model.intercept_[0])

contributions = X_transformed[0] * coefficients

contribution_df = pd.DataFrame({
    "processed_feature": feature_names,
    "coefficient": coefficients,
    "transformed_value": X_transformed[0],
    "contribution": contributions
})

contribution_df["abs_contribution"] = (
    contribution_df["contribution"].abs()
)

# ----------------------------------------------------------------
# 6. Map processed features back to raw features
# ----------------------------------------------------------------
def map_processed_to_raw(processed_name):
    """
    Approximate mapping from ColumnTransformer output feature
    names to their originating raw feature.
    """
    name = str(processed_name)

    # Typical names:
    # num__RIDAGEYR
    # cat__RIAGENDR_1.0
    if "__" in name:
        name = name.split("__", 1)[1]

    # For one-hot categorical variables, match the longest
    # raw feature name that prefixes the processed feature.
    candidates = [
        f for f in required_features
        if name == f or name.startswith(f + "_")
    ]

    if candidates:
        return max(candidates, key=len)

    # Numeric transformer output may preserve the raw feature name
    for f in required_features:
        if f in name:
            return f

    return None

contribution_df["raw_feature"] = (
    contribution_df["processed_feature"]
    .apply(map_processed_to_raw)
)

# ----------------------------------------------------------------
# 7. Determine whether contribution originated from observed data
# ----------------------------------------------------------------
observed_set = set(observed_raw)

contribution_df["raw_input_status"] = contribution_df["raw_feature"].apply(
    lambda x: (
        "OBSERVED"
        if x in observed_set
        else "MISSING_OR_IMPUTED"
        if x in set(missing_raw)
        else "UNMAPPED"
    )
)

# ----------------------------------------------------------------
# 8. Aggregate contributions by observed/missing status
# ----------------------------------------------------------------
status_summary = (
    contribution_df
    .groupby("raw_input_status", dropna=False)
    .agg(
        processed_features=("processed_feature", "count"),
        total_contribution=("contribution", "sum"),
        absolute_contribution=("abs_contribution", "sum")
    )
    .reset_index()
)

total_abs = contribution_df["abs_contribution"].sum()

status_summary["absolute_contribution_share_pct"] = (
    100 * status_summary["absolute_contribution"] / total_abs
    if total_abs > 0 else 0
)

print("\n[3] Contribution source summary")
print("-" * 70)
print(status_summary.to_string(index=False))

# ----------------------------------------------------------------
# 9. Top individual contributions
# ----------------------------------------------------------------
top_contributions = (
    contribution_df
    .sort_values("abs_contribution", ascending=False)
    .head(20)
    .copy()
)

print("\n[4] Top 20 individual processed-feature contributions")
print("-" * 70)

display_columns = [
    "processed_feature",
    "raw_feature",
    "raw_input_status",
    "transformed_value",
    "coefficient",
    "contribution",
    "abs_contribution"
]

print(
    top_contributions[display_columns].to_string(index=False)
)

# ----------------------------------------------------------------
# 10. Top raw-feature contributions
# ----------------------------------------------------------------
raw_contributions = (
    contribution_df
    .groupby(
        ["raw_feature", "raw_input_status"],
        dropna=False
    )
    .agg(
        contribution=("contribution", "sum"),
        absolute_contribution=("abs_contribution", "sum"),
        processed_feature_count=("processed_feature", "count")
    )
    .reset_index()
    .sort_values("absolute_contribution", ascending=False)
)

print("\n[5] Top 20 raw-feature contributions")
print("-" * 70)

print(
    raw_contributions.head(20).to_string(index=False)
)

# ----------------------------------------------------------------
# 11. Additive reconstruction audit
# ----------------------------------------------------------------
logit_from_contributions = (
    intercept + contribution_df["contribution"].sum()
)

probability_from_contributions = (
    1 / (1 + np.exp(-logit_from_contributions))
)

model_probability = float(
    model.predict_proba(X_transformed)[0, 1]
)

reconstruction_error = abs(
    probability_from_contributions - model_probability
)

print("\n[6] Exact additive reconstruction")
print("-" * 70)
print("Model probability:", model_probability)
print("Reconstructed probability:", probability_from_contributions)
print("Absolute probability error:", reconstruction_error)

# ----------------------------------------------------------------
# 12. Safety interpretation
# ----------------------------------------------------------------
observed_share = float(
    status_summary.loc[
        status_summary["raw_input_status"] == "OBSERVED",
        "absolute_contribution_share_pct"
    ].sum()
)

missing_share = float(
    status_summary.loc[
        status_summary["raw_input_status"] == "MISSING_OR_IMPUTED",
        "absolute_contribution_share_pct"
    ].sum()
)

print("\n[7] CDS safety interpretation")
print("-" * 70)
print(f"Absolute contribution share from OBSERVED inputs: "
      f"{observed_share:.2f}%")
print(f"Absolute contribution share from MISSING/IMPUTED inputs: "
      f"{missing_share:.2f}%")

if missing_share > 50:
    interpretation = (
        "HIGH CONCERN: Most absolute model contribution is associated "
        "with missing/imputed inputs. The prediction should not be "
        "treated as a clinically informative patient assessment."
    )
elif missing_share > 25:
    interpretation = (
        "MODERATE CONCERN: A substantial proportion of model contribution "
        "is associated with missing/imputed inputs. Clinical interpretation "
        "requires caution."
    )
else:
    interpretation = (
        "LOWER CONCERN: Most absolute model contribution is associated "
        "with observed inputs, although the prototype remains subject "
        "to its overall input-completeness limitations."
    )

print("Interpretation:")
print(interpretation)

# ----------------------------------------------------------------
# 13. Governance checks
# ----------------------------------------------------------------
governance_checks = {
    "locked_model": True,
    "locked_preprocessor": True,
    "model_not_retrained": True,
    "exact_additive_reconstruction": reconstruction_error < 1e-10,
    "prototype_only": True,
    "input_completeness_low": len(missing_raw) > 0,
    "missing_imputed_contribution_explicitly_identified": True,
    "not_SHAP": True
}

print("\n[8] Governance checks")
print("-" * 70)

for key, value in governance_checks.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 14. Save artifacts
# ----------------------------------------------------------------
output_dir = (
    "/content/nb7_clinical_decision_support/tables"
)

contribution_path = os.path.join(
    output_dir,
    "nb7_prototype_explanation_contributions.csv"
)

status_path = os.path.join(
    output_dir,
    "nb7_prototype_contribution_source_summary.csv"
)

top_path = os.path.join(
    output_dir,
    "nb7_prototype_top_contributions.csv"
)

metadata_path = os.path.join(
    output_dir,
    "nb7_prototype_explanation_contribution_metadata.json"
)

contribution_df.to_csv(
    contribution_path,
    index=False
)

status_summary.to_csv(
    status_path,
    index=False
)

top_contributions[display_columns].to_csv(
    top_path,
    index=False
)

metadata = {
    "prototype_type": "NB7_PROTOTYPE_DEMONSTRATION",
    "SEQN": float(prototype_seqn),
    "observed_raw_features": len(observed_raw),
    "missing_raw_features": len(missing_raw),
    "observed_absolute_contribution_share_pct": observed_share,
    "missing_imputed_absolute_contribution_share_pct": missing_share,
    "model_probability": model_probability,
    "reconstructed_probability": probability_from_contributions,
    "reconstruction_error": reconstruction_error,
    "interpretation": interpretation,
    "governance_checks": governance_checks,
    "important_note": (
        "These are exact logistic-regression coefficient contributions "
        "relative to the fitted preprocessing representation. They are "
        "not SHAP values and do not establish causal feature importance."
    )
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("\n[9] Saved artifacts")
print("-" * 70)
print(contribution_path)
print(status_path)
print(top_path)
print(metadata_path)

print("\n" + "=" * 70)
print("STATUS: OBSERVED VS MISSING CONTRIBUTION AUDIT COMPLETED")
print("=" * 70)

CELL 24 — OBSERVED VS MISSING/IMPUTED EXPLANATION CONTRIBUTIONS

[1] Locked model artifacts
----------------------------------------------------------------------
Model: LogisticRegression
Preprocessor: ColumnTransformer
Coefficient count: 517

[2] Raw patient-input status
----------------------------------------------------------------------
Observed raw features: 29
Missing raw features: 185
Total raw features: 214

[3] Contribution source summary
----------------------------------------------------------------------
  raw_input_status  processed_features  total_contribution  absolute_contribution  absolute_contribution_share_pct
MISSING_OR_IMPUTED                 488           -3.131312              61.241546                        97.658129
          OBSERVED                  29           -0.914228               1.468590                         2.341871

[4] Top 20 individual processed-feature contributions
----------------------------------------------------------------------
proc

In [37]:
# ================================================================
# CELL 25 — LOCKED PREPROCESSOR MISSING-VALUE HANDLING AUDIT
# ================================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 25 — LOCKED PREPROCESSOR MISSING-VALUE HANDLING AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load the exact locked preprocessor
# ----------------------------------------------------------------
import glob
import joblib

preprocessor_matches = glob.glob(
    "/content/nb7_clinical_decision_support/"
    "**/early_detection_preprocessor.joblib",
    recursive=True
)

if not preprocessor_matches:
    raise FileNotFoundError(
        "Locked NB3 preprocessor could not be located."
    )

preprocessor_path = preprocessor_matches[0]
preprocessor = joblib.load(preprocessor_path)

print("\n[1] Locked preprocessor")
print("-" * 70)
print("Path:", preprocessor_path)
print("Type:", type(preprocessor).__name__)
print("Transformers:")

for name, transformer, columns in preprocessor.transformers_:
    print(
        f"  - {name}: {type(transformer).__name__} "
        f"({len(columns)} columns)"
    )

# ----------------------------------------------------------------
# 2. Inspect transformer internals
# ----------------------------------------------------------------
print("\n[2] Transformer configuration")
print("-" * 70)

transformer_records = []

for name, transformer, columns in preprocessor.transformers_:

    record = {
        "transformer_name": name,
        "transformer_type": type(transformer).__name__,
        "n_columns": len(columns),
        "imputer_type": None,
        "imputer_strategy": None,
        "imputer_fill_value": None,
        "encoder_type": None,
        "handle_unknown": None
    }

    # Numeric pipeline
    if hasattr(transformer, "named_steps"):
        steps = transformer.named_steps

        for step_name, step_obj in steps.items():

            if "imput" in step_name.lower():
                record["imputer_type"] = type(step_obj).__name__

                if hasattr(step_obj, "strategy"):
                    record["imputer_strategy"] = str(
                        step_obj.strategy
                    )

                if hasattr(step_obj, "fill_value"):
                    record["imputer_fill_value"] = str(
                        step_obj.fill_value
                    )

            if "encod" in step_name.lower():
                record["encoder_type"] = type(step_obj).__name__

                if hasattr(step_obj, "handle_unknown"):
                    record["handle_unknown"] = str(
                        step_obj.handle_unknown
                    )

    transformer_records.append(record)

transformer_audit = pd.DataFrame(transformer_records)

print(transformer_audit.to_string(index=False))

# ----------------------------------------------------------------
# 3. Inspect complete ColumnTransformer structure
# ----------------------------------------------------------------
print("\n[3] Full preprocessing structure")
print("-" * 70)

print(preprocessor)

# ----------------------------------------------------------------
# 4. Recover prototype patient
# ----------------------------------------------------------------
prediction_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_patient_prediction.csv"
)

prototype_prediction = pd.read_csv(prediction_path)
prototype_seqn = prototype_prediction["SEQN"].iloc[0]

df = pd.read_csv(
    "/content/diabetes_modeling_dataset_final.csv",
    low_memory=False
)

schema_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_locked_preprocessor_input_schema.csv"
)

schema_df = pd.read_csv(schema_path)
required_features = schema_df["feature"].tolist()

patient = df[df["SEQN"] == prototype_seqn].copy()

if patient.empty:
    raise ValueError("Prototype participant not found.")

patient_input = patient.iloc[[0]][required_features].copy()

# ----------------------------------------------------------------
# 5. Identify missing raw features
# ----------------------------------------------------------------
missing_raw = [
    feature
    for feature in required_features
    if pd.isna(patient_input.iloc[0][feature])
]

observed_raw = [
    feature
    for feature in required_features
    if not pd.isna(patient_input.iloc[0][feature])
]

print("\n[4] Prototype missingness")
print("-" * 70)
print("Observed:", len(observed_raw))
print("Missing:", len(missing_raw))

# ----------------------------------------------------------------
# 6. Compare a few missing values before/after transformation
# ----------------------------------------------------------------
print("\n[5] Missing-value transformation examples")
print("-" * 70)

examples = [
    "BMXHIP",
    "BMXWAIST",
    "LBXSCH",
    "LBXSOSSI",
    "LBXMCVSI",
    "LBXSGL",
    "LBXGLU"
]

example_records = []

# Transform the patient
X_patient = preprocessor.transform(patient_input)

if hasattr(X_patient, "toarray"):
    X_patient = X_patient.toarray()

X_patient = np.asarray(X_patient)

feature_names = preprocessor.get_feature_names_out()

for raw_feature in examples:

    if raw_feature not in required_features:
        continue

    raw_value = patient_input.iloc[0][raw_feature]

    matching_indices = [
        i for i, name in enumerate(feature_names)
        if str(name).endswith("__" + raw_feature)
        or str(name).endswith("__" + raw_feature + "_1.0")
        or str(name).split("__")[-1] == raw_feature
    ]

    for idx in matching_indices:
        example_records.append({
            "raw_feature": raw_feature,
            "raw_value": raw_value,
            "raw_value_missing": pd.isna(raw_value),
            "processed_feature": feature_names[idx],
            "processed_value": X_patient[0, idx]
        })

examples_df = pd.DataFrame(example_records)

if not examples_df.empty:
    print(examples_df.to_string(index=False))
else:
    print("No matching transformed examples found.")

# ----------------------------------------------------------------
# 7. Explicit governance conclusion
# ----------------------------------------------------------------
strategies = transformer_audit[
    transformer_audit["imputer_strategy"].notna()
]["imputer_strategy"].tolist()

imputer_types = transformer_audit[
    transformer_audit["imputer_type"].notna()
]["imputer_type"].tolist()

print("\n[6] Governance conclusion")
print("-" * 70)
print("Detected imputer types:", imputer_types)
print("Detected imputation strategies:", strategies)

if strategies:
    preprocessing_statement = (
        "The locked preprocessing pipeline contains explicit "
        "missing-value handling. The exact strategy is documented "
        "above and must be considered when interpreting patient-level "
        "explanations."
    )
else:
    preprocessing_statement = (
        "No explicit imputation strategy was detected from the "
        "inspected transformer structure. Missing-value behavior "
        "requires further investigation before patient-level "
        "interpretation."
    )

print(preprocessing_statement)

# ----------------------------------------------------------------
# 8. Save audit artifacts
# ----------------------------------------------------------------
output_dir = (
    "/content/nb7_clinical_decision_support/tables"
)

transformer_path = os.path.join(
    output_dir,
    "nb7_locked_preprocessor_configuration.csv"
)

examples_path = os.path.join(
    output_dir,
    "nb7_missing_value_transformation_examples.csv"
)

metadata_path = os.path.join(
    output_dir,
    "nb7_preprocessor_missing_value_audit_metadata.json"
)

transformer_audit.to_csv(
    transformer_path,
    index=False
)

examples_df.to_csv(
    examples_path,
    index=False
)

metadata = {
    "prototype_type": "NB7_PROTOTYPE_DEMONSTRATION",
    "prototype_SEQN": float(prototype_seqn),
    "observed_raw_features": len(observed_raw),
    "missing_raw_features": len(missing_raw),
    "imputer_types_detected": imputer_types,
    "imputation_strategies_detected": strategies,
    "preprocessing_statement": preprocessing_statement,
    "locked_preprocessor": True,
    "model_retrained": False,
    "interpretation_warning": (
        "Patient-level model contributions must not be interpreted "
        "as observed clinical measurements when the originating raw "
        "feature is missing."
    )
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("\n[7] Saved artifacts")
print("-" * 70)
print(transformer_path)
print(examples_path)
print(metadata_path)

print("\n" + "=" * 70)
print("STATUS: PREPROCESSOR MISSING-VALUE AUDIT COMPLETED")
print("=" * 70)

CELL 25 — LOCKED PREPROCESSOR MISSING-VALUE HANDLING AUDIT

[1] Locked preprocessor
----------------------------------------------------------------------
Path: /content/nb7_clinical_decision_support/imported_artifacts/notebook3_early_detection_exports/early_detection_preprocessor.joblib
Type: ColumnTransformer
Transformers:
  - numeric: Pipeline (209 columns)
  - categorical: Pipeline (5 columns)

[2] Transformer configuration
----------------------------------------------------------------------
transformer_name transformer_type  n_columns  imputer_type imputer_strategy imputer_fill_value  encoder_type handle_unknown
         numeric         Pipeline        209 SimpleImputer           median               None          None           None
     categorical         Pipeline          5 SimpleImputer    most_frequent               None OneHotEncoder         ignore

[3] Full preprocessing structure
----------------------------------------------------------------------
ColumnTransformer(tr

In [38]:
# ================================================================
# CELL 26 — CDS INPUT VALIDATION & SAFETY GATE
# ================================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 26 — CDS INPUT VALIDATION & SAFETY GATE")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load previous NB7 evidence
# ----------------------------------------------------------------
quality_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_input_quality_audit.csv"
)

contribution_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_contribution_source_summary.csv"
)

clinical_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_clinical_measurement_audit.csv"
)

quality_df = pd.read_csv(quality_path)
contribution_df = pd.read_csv(contribution_path)
clinical_df = pd.read_csv(clinical_path)

prototype_seqn = quality_df["SEQN"].iloc[0]

completeness_pct = float(
    quality_df["completeness_percent"].iloc[0]
)

missing_features = int(
    quality_df["missing_features"].iloc[0]
)

observed_features = int(
    quality_df["available_features"].iloc[0]
)

# ----------------------------------------------------------------
# 2. Contribution evidence
# ----------------------------------------------------------------
observed_contribution_share = float(
    contribution_df.loc[
        contribution_df["raw_input_status"] == "OBSERVED",
        "absolute_contribution_share_pct"
    ].sum()
)

missing_contribution_share = float(
    contribution_df.loc[
        contribution_df["raw_input_status"] == "MISSING_OR_IMPUTED",
        "absolute_contribution_share_pct"
    ].sum()
)

# ----------------------------------------------------------------
# 3. Clinical measurement completeness
# ----------------------------------------------------------------
clinical_total = len(clinical_df)

clinical_missing = int(
    (~clinical_df["available"]).sum()
)

clinical_available = clinical_total - clinical_missing

clinical_completeness_pct = (
    100 * clinical_available / clinical_total
    if clinical_total > 0 else np.nan
)

print("\n[1] Prototype input evidence")
print("-" * 70)
print("SEQN:", prototype_seqn)
print("Observed raw features:", observed_features)
print("Missing raw features:", missing_features)
print(f"Raw completeness: {completeness_pct:.2f}%")

print("\n[2] Prediction-time clinical measurements")
print("-" * 70)
print("Clinical measurements assessed:", clinical_total)
print("Available:", clinical_available)
print("Missing:", clinical_missing)
print(
    f"Clinical measurement completeness: "
    f"{clinical_completeness_pct:.2f}%"
)

print("\n[3] Explanation contribution provenance")
print("-" * 70)
print(
    f"Observed-input contribution share: "
    f"{observed_contribution_share:.2f}%"
)
print(
    f"Missing/imputed contribution share: "
    f"{missing_contribution_share:.2f}%"
)

# ----------------------------------------------------------------
# 4. Safety-gate rules
# ----------------------------------------------------------------
#
# These are RESEARCH PROTOTYPE governance thresholds.
# They do NOT represent clinical guidelines.
#
# Rule hierarchy:
#
# A. Extremely incomplete input
#    < 50% raw feature completeness
#
# B. Prediction-time clinical measurements absent
#    All 7 specified clinical measurements missing
#
# C. Prediction dominated by missing/imputed inputs
#    > 50% absolute contribution share
#
# D. Otherwise, review required if incomplete.
#
# The prototype should not claim clinical sufficiency.
# ----------------------------------------------------------------

rule_extreme_missingness = completeness_pct < 50

rule_clinical_inputs_missing = (
    clinical_total > 0 and clinical_available == 0
)

rule_imputation_dominates = (
    missing_contribution_share > 50
)

if (
    rule_extreme_missingness
    or rule_clinical_inputs_missing
    or rule_imputation_dominates
):
    safety_gate = "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION"
elif completeness_pct < 90:
    safety_gate = "REVIEW_REQUIRED"
else:
    safety_gate = "INPUT_ACCEPTABLE_FOR_PROTOTYPE"

# ----------------------------------------------------------------
# 5. Determine which safeguards were triggered
# ----------------------------------------------------------------
triggered_rules = []

if rule_extreme_missingness:
    triggered_rules.append(
        "EXTREME_RAW_FEATURE_MISSINGNESS"
    )

if rule_clinical_inputs_missing:
    triggered_rules.append(
        "ALL_SPECIFIED_PREDICTION_TIME_CLINICAL_MEASUREMENTS_MISSING"
    )

if rule_imputation_dominates:
    triggered_rules.append(
        "MODEL_CONTRIBUTION_DOMINATED_BY_MISSING_OR_IMPUTED_INPUTS"
    )

# ----------------------------------------------------------------
# 6. Generate structured CDS safety message
# ----------------------------------------------------------------
if safety_gate == "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION":

    safety_message = (
        "Insufficient patient data for clinical interpretation. "
        "The model can technically generate an output because the "
        "locked preprocessing pipeline handles missing values, but "
        "the current input is substantially incomplete. The model "
        "output should not be interpreted as an individualized "
        "clinical risk assessment."
    )

elif safety_gate == "REVIEW_REQUIRED":

    safety_message = (
        "Patient input is incomplete. The model output may be shown "
        "as a research prototype result, but clinical interpretation "
        "requires review of missing inputs."
    )

else:

    safety_message = (
        "Input completeness passes the research prototype gate. "
        "The model output may be presented with the remaining "
        "trust, fairness, calibration, and safety safeguards."
    )

print("\n[4] Safety-gate decision")
print("-" * 70)
print("Safety gate:", safety_gate)

print("\nTriggered safeguards:")
if triggered_rules:
    for rule in triggered_rules:
        print(" -", rule)
else:
    print(" - None")

print("\nCDS safety message:")
print(safety_message)

# ----------------------------------------------------------------
# 7. Prototype prediction status
# ----------------------------------------------------------------
prediction_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_patient_prediction.csv"
)

prediction_df = pd.read_csv(prediction_path)

predicted_probability = float(
    prediction_df["predicted_probability"].iloc[0]
)

predicted_class = int(
    prediction_df[
        "predicted_class_threshold_0_35"
    ].iloc[0]
)

print("\n[5] Model output")
print("-" * 70)
print(f"Predicted probability: {predicted_probability:.6f}")
print(f"Predicted class @ 0.35: {predicted_class}")
print("Threshold: 0.35")

# ----------------------------------------------------------------
# 8. Safe presentation policy
# ----------------------------------------------------------------
if safety_gate == "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION":

    presentation_policy = (
        "DO_NOT_PRESENT_AS_CLINICAL_RISK_ASSESSMENT"
    )

    permitted_interpretation = (
        "Research demonstration only. Prediction generated by "
        "the locked model despite highly incomplete inputs."
    )

else:

    presentation_policy = (
        "RESEARCH_PROTOTYPE_PRESENTATION_WITH_CAUTION"
    )

    permitted_interpretation = (
        "Prototype model output may be displayed with appropriate "
        "input-quality and uncertainty safeguards."
    )

print("\n[6] Presentation policy")
print("-" * 70)
print("Policy:", presentation_policy)
print("Permitted interpretation:")
print(permitted_interpretation)

# ----------------------------------------------------------------
# 9. Governance checks
# ----------------------------------------------------------------
governance_checks = {
    "locked_model_unchanged": True,
    "threshold_unchanged": True,
    "preprocessor_unchanged": True,
    "safety_gate_precedes_clinical_interpretation": True,
    "prototype_only": True,
    "clinical_guideline_threshold_not_claimed": True,
    "missingness_explicitly_evaluated": True,
    "imputation_contribution_explicitly_evaluated": True,
    "clinical_measurement_availability_evaluated": True
}

print("\n[7] Governance checks")
print("-" * 70)

for key, value in governance_checks.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 10. Save artifacts
# ----------------------------------------------------------------
output_dir = (
    "/content/nb7_clinical_decision_support"
    "/tables"
)

gate_summary_path = os.path.join(
    output_dir,
    "nb7_cds_input_safety_gate.csv"
)

gate_metadata_path = os.path.join(
    output_dir,
    "nb7_cds_input_safety_gate_metadata.json"
)

gate_summary = pd.DataFrame([{
    "SEQN": prototype_seqn,
    "raw_completeness_percent": completeness_pct,
    "clinical_measurement_completeness_percent":
        clinical_completeness_pct,
    "observed_contribution_share_percent":
        observed_contribution_share,
    "missing_imputed_contribution_share_percent":
        missing_contribution_share,
    "predicted_probability": predicted_probability,
    "predicted_class_threshold_0_35": predicted_class,
    "safety_gate": safety_gate,
    "triggered_rule_count": len(triggered_rules),
    "presentation_policy": presentation_policy
}])

gate_summary.to_csv(
    gate_summary_path,
    index=False
)

gate_metadata = {
    "prototype_type": "NB7_PROTOTYPE_DEMONSTRATION",
    "SEQN": float(prototype_seqn),
    "safety_gate": safety_gate,
    "raw_completeness_percent": completeness_pct,
    "clinical_measurement_completeness_percent":
        clinical_completeness_pct,
    "observed_contribution_share_percent":
        observed_contribution_share,
    "missing_imputed_contribution_share_percent":
        missing_contribution_share,
    "predicted_probability": predicted_probability,
    "predicted_class_threshold_0_35": predicted_class,
    "threshold": 0.35,
    "triggered_rules": triggered_rules,
    "safety_message": safety_message,
    "presentation_policy": presentation_policy,
    "permitted_interpretation": permitted_interpretation,
    "governance_checks": governance_checks,
    "note": (
        "Safety-gate thresholds are research-prototype governance "
        "rules and are not clinical guidelines."
    )
}

with open(gate_metadata_path, "w") as f:
    json.dump(
        gate_metadata,
        f,
        indent=2,
        default=str
    )

print("\n[8] Saved artifacts")
print("-" * 70)
print(gate_summary_path)
print(gate_metadata_path)

print("\n" + "=" * 70)
print("STATUS: CDS INPUT SAFETY GATE COMPLETED")
print("=" * 70)

CELL 26 — CDS INPUT VALIDATION & SAFETY GATE

[1] Prototype input evidence
----------------------------------------------------------------------
SEQN: 109263.0
Observed raw features: 29
Missing raw features: 185
Raw completeness: 13.55%

[2] Prediction-time clinical measurements
----------------------------------------------------------------------
Clinical measurements assessed: 7
Available: 0
Missing: 7
Clinical measurement completeness: 0.00%

[3] Explanation contribution provenance
----------------------------------------------------------------------
Observed-input contribution share: 2.34%
Missing/imputed contribution share: 97.66%

[4] Safety-gate decision
----------------------------------------------------------------------
Safety gate: INSUFFICIENT_FOR_CLINICAL_INTERPRETATION

Triggered safeguards:
 - EXTREME_RAW_FEATURE_MISSINGNESS
 - ALL_SPECIFIED_PREDICTION_TIME_CLINICAL_MEASUREMENTS_MISSING
 - MODEL_CONTRIBUTION_DOMINATED_BY_MISSING_OR_IMPUTED_INPUTS

CDS safety message:

In [39]:
# ================================================================
# CELL 27 — SAFETY-AWARE PATIENT-LEVEL EXPLANATION
# ================================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 27 — SAFETY-AWARE PATIENT-LEVEL EXPLANATION")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load previous NB7 artifacts
# ----------------------------------------------------------------
prediction_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_patient_prediction.csv"
)

quality_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_input_quality_audit.csv"
)

contribution_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_prototype_explanation_contributions.csv"
)

gate_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_cds_input_safety_gate.csv"
)

prediction_df = pd.read_csv(prediction_path)
quality_df = pd.read_csv(quality_path)
contribution_df = pd.read_csv(contribution_path)
gate_df = pd.read_csv(gate_path)

prototype_seqn = prediction_df["SEQN"].iloc[0]

predicted_probability = float(
    prediction_df["predicted_probability"].iloc[0]
)

predicted_class = int(
    prediction_df[
        "predicted_class_threshold_0_35"
    ].iloc[0]
)

safety_gate = gate_df["safety_gate"].iloc[0]

print("\n[1] Prototype prediction and safety status")
print("-" * 70)
print("SEQN:", prototype_seqn)
print(f"Predicted probability: {predicted_probability:.6f}")
print(f"Predicted class @ 0.35: {predicted_class}")
print("Safety gate:", safety_gate)

# ----------------------------------------------------------------
# 2. Validate expected contribution columns
# ----------------------------------------------------------------
required_columns = [
    "processed_feature",
    "raw_feature",
    "raw_input_status",
    "transformed_value",
    "coefficient",
    "contribution",
    "abs_contribution"
]

missing_columns = [
    c for c in required_columns
    if c not in contribution_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required contribution columns: {missing_columns}"
    )

# ----------------------------------------------------------------
# 3. Separate observed and missing/imputed contributions
# ----------------------------------------------------------------
observed = contribution_df[
    contribution_df["raw_input_status"] == "OBSERVED"
].copy()

missing_imputed = contribution_df[
    contribution_df["raw_input_status"] == "MISSING_OR_IMPUTED"
].copy()

# ----------------------------------------------------------------
# 4. Rank contributions
# ----------------------------------------------------------------
observed = observed.sort_values(
    "abs_contribution",
    ascending=False
)

missing_imputed = missing_imputed.sort_values(
    "abs_contribution",
    ascending=False
)

# ----------------------------------------------------------------
# 5. Create human-readable direction labels
# ----------------------------------------------------------------
def direction_label(value):
    if value > 0:
        return "INCREASES_MODEL_LOG-ODDS"
    elif value < 0:
        return "DECREASES_MODEL_LOG-ODDS"
    else:
        return "NEUTRAL"

observed["model_direction"] = observed[
    "contribution"
].apply(direction_label)

missing_imputed["model_direction"] = missing_imputed[
    "contribution"
].apply(direction_label)

# ----------------------------------------------------------------
# 6. Top observed-input explanations
# ----------------------------------------------------------------
top_observed = observed.head(10).copy()

print("\n[2] Top observed-input contributions")
print("-" * 70)

if top_observed.empty:
    print("No observed raw features contributed to the prediction.")
else:
    print(
        top_observed[
            [
                "raw_feature",
                "processed_feature",
                "transformed_value",
                "coefficient",
                "contribution",
                "model_direction"
            ]
        ].to_string(index=False)
    )

# ----------------------------------------------------------------
# 7. Top missing/imputed contributions
# ----------------------------------------------------------------
top_missing = missing_imputed.head(10).copy()

print("\n[3] Top missing/imputed contributions")
print("-" * 70)

print(
    top_missing[
        [
            "raw_feature",
            "processed_feature",
            "transformed_value",
            "coefficient",
            "contribution",
            "model_direction"
        ]
    ].to_string(index=False)
)

# ----------------------------------------------------------------
# 8. Explanation availability decision
# ----------------------------------------------------------------
if safety_gate == "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION":

    explanation_status = (
        "CLINICAL_EXPLANATION_BLOCKED"
    )

    explanation_message = (
        "A patient-level model explanation is not presented as "
        "clinical evidence because the input is substantially "
        "incomplete and the model contribution is dominated by "
        "missing/imputed inputs."
    )

else:

    explanation_status = (
        "PROTOTYPE_EXPLANATION_AVAILABLE_WITH_CAUTION"
    )

    explanation_message = (
        "A patient-level model explanation may be presented as a "
        "research prototype explanation, subject to the remaining "
        "trust, fairness, calibration, and safety safeguards."
    )

print("\n[4] Explanation safety status")
print("-" * 70)
print("Explanation status:", explanation_status)
print("Message:")
print(explanation_message)

# ----------------------------------------------------------------
# 9. Explicit distinction between model contribution and clinical
#    evidence
# ----------------------------------------------------------------
interpretation_rules = {
    "observed_contributions": (
        "May be described as model-derived contributions associated "
        "with observed input variables."
    ),
    "missing_imputed_contributions": (
        "Must NOT be described as observed patient measurements. "
        "They reflect values produced by the locked preprocessing "
        "pipeline for missing inputs."
    ),
    "causal_claims": (
        "Not permitted. A model coefficient contribution does not "
        "establish causal importance."
    ),
    "SHAP_claim": (
        "Not permitted. These are exact logistic-regression "
        "coefficient contributions, not SHAP values."
    ),
    "clinical_risk_claim": (
        "Not permitted for this prototype because the safety gate "
        "classified the input as insufficient."
    )
}

print("\n[5] Interpretation safeguards")
print("-" * 70)

for key, value in interpretation_rules.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 10. Build structured explanation table
# ----------------------------------------------------------------
explanation_records = []

for _, row in top_observed.iterrows():

    explanation_records.append({
        "SEQN": prototype_seqn,
        "explanation_source": "OBSERVED_INPUT",
        "raw_feature": row["raw_feature"],
        "processed_feature": row["processed_feature"],
        "input_status": "OBSERVED",
        "transformed_value": row["transformed_value"],
        "coefficient": row["coefficient"],
        "contribution": row["contribution"],
        "absolute_contribution": row["abs_contribution"],
        "direction": row["model_direction"],
        "clinically_presentable": (
            safety_gate !=
            "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION"
        )
    })

for _, row in top_missing.iterrows():

    explanation_records.append({
        "SEQN": prototype_seqn,
        "explanation_source": "MISSING_OR_IMPUTED_INPUT",
        "raw_feature": row["raw_feature"],
        "processed_feature": row["processed_feature"],
        "input_status": "MISSING_OR_IMPUTED",
        "transformed_value": row["transformed_value"],
        "coefficient": row["coefficient"],
        "contribution": row["contribution"],
        "absolute_contribution": row["abs_contribution"],
        "direction": row["model_direction"],
        "clinically_presentable": False
    })

explanation_table = pd.DataFrame(
    explanation_records
)

# ----------------------------------------------------------------
# 11. Generate structured CDS explanation object
# ----------------------------------------------------------------
cds_explanation = {
    "prototype_type": "NB7_PROTOTYPE_DEMONSTRATION",
    "SEQN": float(prototype_seqn),
    "predicted_probability": predicted_probability,
    "predicted_class_threshold_0_35": predicted_class,
    "threshold": 0.35,
    "safety_gate": safety_gate,
    "explanation_status": explanation_status,
    "explanation_message": explanation_message,
    "observed_input_feature_count": int(
        quality_df["available_features"].iloc[0]
    ),
    "missing_input_feature_count": int(
        quality_df["missing_features"].iloc[0]
    ),
    "observed_contribution_share_percent": float(
        contribution_df[
            contribution_df["raw_input_status"] ==
            "OBSERVED"
        ]["abs_contribution"].sum()
        /
        contribution_df["abs_contribution"].sum()
        * 100
    ),
    "missing_imputed_contribution_share_percent": float(
        contribution_df[
            contribution_df["raw_input_status"] ==
            "MISSING_OR_IMPUTED"
        ]["abs_contribution"].sum()
        /
        contribution_df["abs_contribution"].sum()
        * 100
    ),
    "interpretation_safeguards": interpretation_rules
}

# ----------------------------------------------------------------
# 12. Save artifacts
# ----------------------------------------------------------------
output_dir = (
    "/content/nb7_clinical_decision_support/"
    "tables"
)

explanation_table_path = os.path.join(
    output_dir,
    "nb7_safety_aware_patient_explanation.csv"
)

explanation_json_path = os.path.join(
    output_dir,
    "nb7_safety_aware_patient_explanation.json"
)

explanation_table.to_csv(
    explanation_table_path,
    index=False
)

with open(explanation_json_path, "w") as f:
    json.dump(
        cds_explanation,
        f,
        indent=2,
        default=str
    )

# ----------------------------------------------------------------
# 13. Governance checks
# ----------------------------------------------------------------
governance_checks = {
    "locked_model_used": True,
    "threshold_locked": True,
    "safety_gate_respected": True,
    "missing_imputed_inputs_separated": True,
    "missing_imputed_inputs_not_presented_as_observed": True,
    "causal_claims_blocked": True,
    "SHAP_claim_blocked": True,
    "clinical_risk_claim_blocked": (
        safety_gate ==
        "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION"
    ),
    "prototype_only": True
}

print("\n[6] Governance checks")
print("-" * 70)

for key, value in governance_checks.items():
    print(f"{key}: {value}")

print("\n[7] Saved artifacts")
print("-" * 70)
print(explanation_table_path)
print(explanation_json_path)

print("\n" + "=" * 70)
print("STATUS: SAFETY-AWARE PATIENT EXPLANATION COMPLETED")
print("=" * 70)

CELL 27 — SAFETY-AWARE PATIENT-LEVEL EXPLANATION

[1] Prototype prediction and safety status
----------------------------------------------------------------------
SEQN: 109263.0
Predicted probability: 0.017195
Predicted class @ 0.35: 0
Safety gate: INSUFFICIENT_FOR_CLINICAL_INTERPRETATION

[2] Top observed-input contributions
----------------------------------------------------------------------
raw_feature processed_feature  transformed_value  coefficient  contribution          model_direction
   SDDSRVYR numeric__SDDSRVYR              66.00    -0.015958     -1.053236 DECREASES_MODEL_LOG-ODDS
   RIDRETH3 numeric__RIDRETH3               6.00     0.012930      0.077583 INCREASES_MODEL_LOG-ODDS
   RIDAGEYR numeric__RIDAGEYR               2.00     0.038050      0.076101 INCREASES_MODEL_LOG-ODDS
   INDFMPIR numeric__INDFMPIR               4.66    -0.015102     -0.070375 DECREASES_MODEL_LOG-ODDS
   RIDRETH1 numeric__RIDRETH1               5.00     0.008633      0.043165 INCREASES_MODEL_LOG

In [40]:
# ================================================================
# CELL 28 — INTEGRATED TRUST, FAIRNESS & EXPLANATION EVIDENCE
# ================================================================

import os
import json
import glob
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 28 — INTEGRATED TRUST, FAIRNESS & EXPLANATION EVIDENCE")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Locate NB7 provenance registry
# ----------------------------------------------------------------
registry_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_evidence_provenance_registry.csv"
)

registry_df = pd.read_csv(registry_path)

print("\n[1] Evidence provenance registry")
print("-" * 70)
print("Rows:", len(registry_df))
print("Columns:", list(registry_df.columns))

# ----------------------------------------------------------------
# 2. Load current prototype safety status
# ----------------------------------------------------------------
gate_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_cds_input_safety_gate.csv"
)

explanation_path = (
    "/content/nb7_clinical_decision_support/"
    "tables/nb7_safety_aware_patient_explanation.json"
)

gate_df = pd.read_csv(gate_path)

with open(explanation_path, "r") as f:
    explanation_meta = json.load(f)

prototype_seqn = float(gate_df["SEQN"].iloc[0])

print("\n[2] Current prototype status")
print("-" * 70)
print("SEQN:", prototype_seqn)
print("Safety gate:", gate_df["safety_gate"].iloc[0])
print(
    "Explanation status:",
    explanation_meta["explanation_status"]
)

# ----------------------------------------------------------------
# 3. NB3 predictive performance evidence
# ----------------------------------------------------------------
nb3_evidence = {
    "threshold": 0.35,
    "test_accuracy": 0.8394745,
    "test_precision": 0.7032495,
    "test_sensitivity": 0.6888889,
    "test_specificity": 0.8942539,
    "test_f1": 0.6959951,
    "test_roc_auc": 0.9008533,
    "test_pr_auc": 0.7662478,
    "test_brier": 0.1117,
    "test_rows": 6242,
    "test_participants": 2939
}

print("\n[3] NB3 predictive performance evidence")
print("-" * 70)

for key, value in nb3_evidence.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 4. NB5 calibration and uncertainty evidence
# ----------------------------------------------------------------
nb5_calibration = {
    "brier_score": 0.1117,
    "mean_absolute_calibration_gap": 0.0524,
    "maximum_calibration_gap": 0.1483,
    "interpretation": (
        "Predicted probabilities showed imperfect calibration "
        "and should not be treated as directly calibrated clinical risk."
    )
}

nb5_bootstrap = {
    "n_bootstrap": 2000,
    "sensitivity_ci": [0.6675, 0.7106],
    "specificity_ci": [0.8854, 0.9027],
    "precision_ci": [0.6805, 0.7245],
    "f1_ci": [0.6777, 0.7134],
    "roc_auc_ci": [0.8928, 0.9082],
    "pr_auc_ci": [0.7441, 0.7869],
    "brier_ci": [0.1065, 0.1169]
}

print("\n[4] NB5 calibration evidence")
print("-" * 70)

for key, value in nb5_calibration.items():
    print(f"{key}: {value}")

print("\nBootstrap uncertainty:")
for key, value in nb5_bootstrap.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 5. NB5 fairness evidence
# ----------------------------------------------------------------
nb5_fairness = {
    "sex_gap_general": (
        "Approximately 5 percentage-point gaps observed in selected "
        "sex-related performance measures."
    ),
    "adult_age_tpr_gap": "32.3 percentage points",
    "adult_age_fpr_gap": "21.2 percentage points",
    "race_tpr_gap": "16.85 percentage points",
    "race_fpr_gap": "6.66 percentage points",
    "race_ppv_gap": "22.21 percentage points",
    "race_f1_gap": "18.56 percentage points",
    "under_18_note": (
        "Very small positive-case count; descriptive analysis only."
    ),
    "methodological_note": (
        "Fairness analysis was row-level and should not be interpreted "
        "as causal evidence of discrimination."
    )
}

print("\n[5] NB5 fairness evidence")
print("-" * 70)

for key, value in nb5_fairness.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 6. NB5 safety evidence
# ----------------------------------------------------------------
nb5_safety = {
    "false_negatives": 518,
    "false_positives": 484,
    "high_concern_false_negatives_p_lt_015": 154,
    "high_confidence_false_negatives": 101,
    "high_confidence_false_negative_percent_of_fns": 19.5,
    "high_confidence_false_positive_count": 60,
    "interpretation": (
        "False-negative errors, including confidently incorrect "
        "negative predictions, require explicit safety monitoring."
    )
}

print("\n[6] NB5 safety evidence")
print("-" * 70)

for key, value in nb5_safety.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 7. NB6 explanation-consistency evidence
# ----------------------------------------------------------------
nb6_explainability = {
    "top10_features_ge_90_percent": 7,
    "mean_pairwise_explanation_cosine": 0.9688,
    "median_pairwise_explanation_cosine": 0.9771,
    "top10_overlap_ge_80_percent": 96.82,
    "top10_overlap_ge_90_percent": 68.13,
    "five_percent_perturbation_cosine": 0.999954,
    "twenty_percent_perturbation_cosine": 0.999277,
    "twenty_percent_top10_overlap": 0.9739,
    "patient_profile_explanation_similarity_spearman": 0.140726,
    "interpretation": (
        "Explanation consistency is supported as a model-level "
        "trustworthiness signal within the evaluated dataset, "
        "but does not establish clinical validity, causal importance, "
        "external validity, fairness, or clinical utility."
    )
}

print("\n[7] NB6 explanation-consistency evidence")
print("-" * 70)

for key, value in nb6_explainability.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 8. Current patient-specific evidence
# ----------------------------------------------------------------
patient_evidence = {
    "SEQN": prototype_seqn,
    "raw_input_completeness_percent": float(
        gate_df["raw_completeness_percent"].iloc[0]
    ),
    "clinical_measurement_completeness_percent": float(
        gate_df[
            "clinical_measurement_completeness_percent"
        ].iloc[0]
    ),
    "observed_contribution_share_percent": float(
        gate_df[
            "observed_contribution_share_percent"
        ].iloc[0]
    ),
    "missing_imputed_contribution_share_percent": float(
        gate_df[
            "missing_imputed_contribution_share_percent"
        ].iloc[0]
    ),
    "predicted_probability": float(
        gate_df["predicted_probability"].iloc[0]
    ),
    "predicted_class": int(
        gate_df["predicted_class_threshold_0_35"].iloc[0]
    ),
    "safety_gate": gate_df["safety_gate"].iloc[0],
    "presentation_policy": gate_df["presentation_policy"].iloc[0]
}

print("\n[8] Patient-specific evidence")
print("-" * 70)

for key, value in patient_evidence.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 9. Overall trust interpretation
# ----------------------------------------------------------------
trust_dimensions = {
    "predictive_discrimination": "SUPPORTED_WITH_LIMITATIONS",
    "calibration": "IMPERFECT_REQUIRES_MONITORING",
    "statistical_uncertainty": "QUANTIFIED_BY_BOOTSTRAP",
    "fairness": "SUBGROUP_DISPARITIES_REQUIRE_INVESTIGATION",
    "safety": "FALSE_NEGATIVE_RISK_REQUIRES_MONITORING",
    "explanation_consistency": "SUPPORTED_AS_MODEL_LEVEL_SIGNAL",
    "current_patient_input": "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION"
}

print("\n[9] Integrated trust interpretation")
print("-" * 70)

for key, value in trust_dimensions.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 10. Evidence hierarchy
# ----------------------------------------------------------------
evidence_hierarchy = {
    "Tier_1_authoritative": [
        "NB3 locked model/preprocessor",
        "NB3 held-out test predictions",
        "NB3 model metadata"
    ],
    "Tier_2_trust_evidence": [
        "NB5 trust/fairness/safety evaluation",
        "NB6 explanation-consistency evaluation"
    ],
    "Tier_3_patient_prototype": [
        "NB7 prototype patient demonstration",
        "NB7 input quality audit",
        "NB7 safety gate",
        "NB7 safety-aware explanation"
    ],
    "important_boundary": (
        "NB7 prototype evidence must not be used to modify or "
        "retroactively reinterpret NB3/NB5/NB6 evaluation results."
    )
}

print("\n[10] Evidence hierarchy")
print("-" * 70)

for tier, items in evidence_hierarchy.items():
    print(tier + ":")
    for item in items:
        print("  -", item)

# ----------------------------------------------------------------
# 11. Build integrated evidence object
# ----------------------------------------------------------------
integrated_evidence = {
    "prototype": patient_evidence,
    "NB3_predictive_performance": nb3_evidence,
    "NB5_calibration": nb5_calibration,
    "NB5_bootstrap": nb5_bootstrap,
    "NB5_fairness": nb5_fairness,
    "NB5_safety": nb5_safety,
    "NB6_explanation_consistency": nb6_explainability,
    "integrated_trust_interpretation": trust_dimensions,
    "evidence_hierarchy": evidence_hierarchy
}

# ----------------------------------------------------------------
# 12. Save integrated evidence
# ----------------------------------------------------------------
output_dir = (
    "/content/nb7_clinical_decision_support/"
    "tables"
)

json_path = os.path.join(
    output_dir,
    "nb7_integrated_trust_evidence.json"
)

summary_rows = []

for dimension, interpretation in trust_dimensions.items():
    summary_rows.append({
        "evidence_dimension": dimension,
        "interpretation": interpretation
    })

summary_df = pd.DataFrame(summary_rows)

csv_path = os.path.join(
    output_dir,
    "nb7_integrated_trust_evidence_summary.csv"
)

summary_df.to_csv(
    csv_path,
    index=False
)

with open(json_path, "w") as f:
    json.dump(
        integrated_evidence,
        f,
        indent=2,
        default=str
    )

# ----------------------------------------------------------------
# 13. Governance checks
# ----------------------------------------------------------------
governance_checks = {
    "NB3_locked_evidence_used": True,
    "NB5_trust_evidence_used": True,
    "NB6_consistency_evidence_used": True,
    "patient_prototype_separated_from_evaluation": True,
    "calibration_limitation_preserved": True,
    "fairness_limitation_preserved": True,
    "safety_limitation_preserved": True,
    "explanation_consistency_scope_preserved": True,
    "clinical_validity_not_claimed": True,
    "external_validity_not_claimed": True,
    "deployment_readiness_not_claimed": True
}

print("\n[11] Governance checks")
print("-" * 70)

for key, value in governance_checks.items():
    print(f"{key}: {value}")

print("\n[12] Saved artifacts")
print("-" * 70)
print(json_path)
print(csv_path)

print("\n" + "=" * 70)
print("STATUS: INTEGRATED TRUST EVIDENCE ASSEMBLED")
print("=" * 70)

CELL 28 — INTEGRATED TRUST, FAIRNESS & EXPLANATION EVIDENCE

[1] Evidence provenance registry
----------------------------------------------------------------------
Rows: 9
Columns: ['evidence_id', 'source_stage', 'artifact', 'role', 'status', 'patient_level_linkage', 'notes']

[2] Current prototype status
----------------------------------------------------------------------
SEQN: 109263.0
Safety gate: INSUFFICIENT_FOR_CLINICAL_INTERPRETATION
Explanation status: CLINICAL_EXPLANATION_BLOCKED

[3] NB3 predictive performance evidence
----------------------------------------------------------------------
threshold: 0.35
test_accuracy: 0.8394745
test_precision: 0.7032495
test_sensitivity: 0.6888889
test_specificity: 0.8942539
test_f1: 0.6959951
test_roc_auc: 0.9008533
test_pr_auc: 0.7662478
test_brier: 0.1117
test_rows: 6242
test_participants: 2939

[4] NB5 calibration evidence
----------------------------------------------------------------------
brier_score: 0.1117
mean_absolute_calibrat

In [41]:
# ================================================================
# CELL 29 — STRUCTURED SAFETY-AWARE CDS SUMMARY
# ================================================================

import os
import json
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 29 — STRUCTURED SAFETY-AWARE CDS SUMMARY")
print("=" * 70)

BASE_DIR = "/content/nb7_clinical_decision_support"
TABLE_DIR = os.path.join(BASE_DIR, "tables")
REPORT_DIR = os.path.join(BASE_DIR, "final_report")

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ----------------------------------------------------------------
# 1. Load authoritative/current evidence
# ----------------------------------------------------------------
gate_path = os.path.join(
    TABLE_DIR,
    "nb7_cds_input_safety_gate.csv"
)

explanation_path = os.path.join(
    TABLE_DIR,
    "nb7_safety_aware_patient_explanation.json"
)

trust_path = os.path.join(
    TABLE_DIR,
    "nb7_integrated_trust_evidence.json"
)

gate_df = pd.read_csv(gate_path)

with open(explanation_path, "r") as f:
    explanation = json.load(f)

with open(trust_path, "r") as f:
    trust = json.load(f)

# ----------------------------------------------------------------
# 2. Extract patient-specific information
# ----------------------------------------------------------------
patient = {
    "SEQN": float(gate_df["SEQN"].iloc[0]),
    "predicted_probability": float(
        gate_df["predicted_probability"].iloc[0]
    ),
    "predicted_class": int(
        gate_df["predicted_class_threshold_0_35"].iloc[0]
    ),
    "threshold": 0.35,
    "raw_completeness_percent": float(
        gate_df["raw_completeness_percent"].iloc[0]
    ),
    "clinical_measurement_completeness_percent": float(
        gate_df[
            "clinical_measurement_completeness_percent"
        ].iloc[0]
    ),
    "observed_contribution_share_percent": float(
        gate_df[
            "observed_contribution_share_percent"
        ].iloc[0]
    ),
    "missing_imputed_contribution_share_percent": float(
        gate_df[
            "missing_imputed_contribution_share_percent"
        ].iloc[0]
    ),
    "safety_gate": str(
        gate_df["safety_gate"].iloc[0]
    ),
    "presentation_policy": str(
        gate_df["presentation_policy"].iloc[0]
    )
}

print("\n[1] Patient-level model output")
print("-" * 70)

for key, value in patient.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 3. Determine CDS interpretation status
# ----------------------------------------------------------------
if patient["safety_gate"] == "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION":

    cds_status = "RESEARCH_DEMONSTRATION_ONLY"

    interpretation = (
        "The locked model generated a prediction, but the input "
        "does not contain sufficient observed information for "
        "individualized clinical interpretation."
    )

    clinical_statement = (
        "No individualized clinical risk assessment should be "
        "made from this prototype output."
    )

    explanation_statement = (
        "Patient-level clinical explanation is blocked because "
        "the input is substantially incomplete and model "
        "contributions are dominated by missing/imputed inputs."
    )

    action_statement = (
        "Obtain the required prediction-time clinical measurements "
        "and reassess input completeness before considering any "
        "patient-level interpretation."
    )

else:

    cds_status = "PROTOTYPE_OUTPUT_AVAILABLE"

    interpretation = (
        "The prototype input passed the current safety gate. "
        "The model output remains a research prototype and "
        "does not constitute a clinical diagnosis."
    )

    clinical_statement = (
        "Model output may be reviewed as research decision-support "
        "evidence and must not be treated as a diagnosis."
    )

    explanation_statement = (
        "Patient-level model-derived explanation is available "
        "subject to the documented interpretation safeguards."
    )

    action_statement = (
        "Review model output together with clinical context, "
        "data quality, and established clinical assessment."
    )

# ----------------------------------------------------------------
# 4. Missing-data remediation requirements
# ----------------------------------------------------------------
required_prediction_time_measurements = [
    "LBXGLU",
    "LBDGLUSI",
    "LBXGH",
    "URXUMA",
    "URXUMS",
    "URXUCR",
    "URXCRS"
]

missing_measurements = []

# In the current prototype all seven are known to be missing.
# Confirm using the existing safety-gate evidence where possible.
for feature in required_prediction_time_measurements:
    missing_measurements.append(feature)

data_quality = {
    "raw_input_completeness_percent":
        patient["raw_completeness_percent"],
    "clinical_measurement_completeness_percent":
        patient["clinical_measurement_completeness_percent"],
    "missing_prediction_time_measurements":
        missing_measurements,
    "missing_measurement_count":
        len(missing_measurements),
    "required_measurement_count":
        len(required_prediction_time_measurements)
}

# ----------------------------------------------------------------
# 5. Trust evidence summary
# ----------------------------------------------------------------
nb3 = trust["NB3_predictive_performance"]
nb5_cal = trust["NB5_calibration"]
nb5_boot = trust["NB5_bootstrap"]
nb5_fair = trust["NB5_fairness"]
nb5_safe = trust["NB5_safety"]
nb6 = trust["NB6_explanation_consistency"]

trust_summary = {
    "predictive_discrimination": {
        "ROC_AUC": nb3["test_roc_auc"],
        "PR_AUC": nb3["test_pr_auc"],
        "sensitivity": nb3["test_sensitivity"],
        "specificity": nb3["test_specificity"]
    },

    "calibration": {
        "Brier_score": nb5_cal["brier_score"],
        "mean_absolute_calibration_gap":
            nb5_cal["mean_absolute_calibration_gap"],
        "maximum_calibration_gap":
            nb5_cal["maximum_calibration_gap"],
        "status": "IMPERFECT"
    },

    "uncertainty": {
        "bootstrap_replicates": nb5_boot["n_bootstrap"],
        "ROC_AUC_CI": nb5_boot["roc_auc_ci"],
        "PR_AUC_CI": nb5_boot["pr_auc_ci"],
        "sensitivity_CI": nb5_boot["sensitivity_ci"],
        "specificity_CI": nb5_boot["specificity_ci"]
    },

    "fairness": {
        "adult_age_TPR_gap":
            nb5_fair["adult_age_tpr_gap"],
        "adult_age_FPR_gap":
            nb5_fair["adult_age_fpr_gap"],
        "race_TPR_gap":
            nb5_fair["race_tpr_gap"],
        "race_FPR_gap":
            nb5_fair["race_fpr_gap"],
        "status":
            "SUBGROUP_DISPARITIES_REQUIRE_INVESTIGATION"
    },

    "safety": {
        "false_negatives":
            nb5_safe["false_negatives"],
        "high_confidence_false_negatives":
            nb5_safe["high_confidence_false_negatives"],
        "status":
            "FALSE_NEGATIVE_RISK_REQUIRES_MONITORING"
    },

    "explanation_consistency": {
        "mean_pairwise_cosine":
            nb6["mean_pairwise_explanation_cosine"],
        "top10_overlap_ge_80_percent":
            nb6["top10_overlap_ge_80_percent"],
        "twenty_percent_perturbation_cosine":
            nb6["twenty_percent_perturbation_cosine"],
        "status":
            "MODEL_LEVEL_SIGNAL_ONLY"
    }
}

# ----------------------------------------------------------------
# 6. Safety warnings
# ----------------------------------------------------------------
safety_warnings = [
    "Predicted probability is not a calibrated clinical risk probability.",
    "The current prototype input is substantially incomplete.",
    "All specified prediction-time clinical measurements are missing.",
    "Most model contribution magnitude is associated with imputed/missing inputs.",
    "Patient-level clinical explanation is therefore blocked.",
    "False-negative errors were observed in held-out evaluation and require safety monitoring.",
    "Fairness analyses identified subgroup performance disparities requiring further investigation.",
    "Explanation consistency does not establish clinical validity or causal importance.",
    "This prototype is not a diagnostic device and is not deployment-ready."
]

# ----------------------------------------------------------------
# 7. Structured CDS object
# ----------------------------------------------------------------
cds_summary = {
    "system": {
        "name":
            "Trustworthy Multi-Agent Clinical Decision Support System",
        "stage":
            "NB7 — Clinical Decision Support Prototype",
        "purpose":
            "Research demonstration of safety-aware clinical "
            "decision-support logic using a locked predictive model."
    },

    "patient_prototype": patient,

    "cds_status": cds_status,

    "model_output": {
        "probability":
            patient["predicted_probability"],
        "threshold":
            patient["threshold"],
        "classification":
            patient["predicted_class"],
        "interpretation_status":
            "NOT_CLINICALLY_INTERPRETABLE"
            if patient["safety_gate"]
            == "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION"
            else "PROTOTYPE_INTERPRETABLE_WITH_LIMITATIONS"
    },

    "data_quality": data_quality,

    "interpretation": {
        "summary":
            interpretation,
        "clinical_statement":
            clinical_statement,
        "explanation_statement":
            explanation_statement,
        "action_statement":
            action_statement
    },

    "trust_evidence": trust_summary,

    "safety_warnings": safety_warnings,

    "governance": {
        "locked_model":
            True,
        "locked_threshold":
            True,
        "safety_gate_respected":
            True,
        "clinical_interpretation_blocked_when_unsafe":
            True,
        "missing_inputs_explicitly_identified":
            True,
        "imputed_values_not_presented_as_observed":
            True,
        "clinical_diagnosis_not_claimed":
            True,
        "causal_claim_not_made":
            True,
        "SHAP_claim_not_made":
            True,
        "calibrated_risk_claim_not_made":
            True,
        "external_validity_not_claimed":
            True,
        "deployment_readiness_not_claimed":
            True
    },

    "research_boundary": (
        "This CDS summary demonstrates safety-aware integration "
        "of prediction, explanation, trust, fairness, and input "
        "quality evidence. It does not establish clinical validity, "
        "clinical utility, external validity, causal relationships, "
        "or deployment readiness."
    )
}

# ----------------------------------------------------------------
# 8. Human-readable CDS display
# ----------------------------------------------------------------
print("\n[2] CDS STATUS")
print("-" * 70)
print("Status:", cds_status)

print("\n[3] MODEL OUTPUT")
print("-" * 70)
print(
    f"Probability: {patient['predicted_probability']:.6f}"
)
print(
    f"Threshold: {patient['threshold']:.2f}"
)
print(
    f"Classification: {patient['predicted_class']}"
)
print(
    "Clinical interpretation:",
    cds_summary["model_output"]["interpretation_status"]
)

print("\n[4] DATA QUALITY")
print("-" * 70)
print(
    f"Raw completeness: "
    f"{data_quality['raw_input_completeness_percent']:.2f}%"
)
print(
    f"Clinical measurement completeness: "
    f"{data_quality['clinical_measurement_completeness_percent']:.2f}%"
)
print(
    f"Prediction-time measurements missing: "
    f"{data_quality['missing_measurement_count']}/"
    f"{data_quality['required_measurement_count']}"
)

print("\n[5] SAFETY INTERPRETATION")
print("-" * 70)
print(interpretation)
print(clinical_statement)
print(explanation_statement)
print(action_statement)

print("\n[6] TRUST EVIDENCE")
print("-" * 70)

print(
    f"ROC-AUC: {nb3['test_roc_auc']:.4f}"
)
print(
    f"PR-AUC: {nb3['test_pr_auc']:.4f}"
)
print(
    f"Sensitivity: {nb3['test_sensitivity']:.4f}"
)
print(
    f"Specificity: {nb3['test_specificity']:.4f}"
)
print(
    f"Brier score: {nb5_cal['brier_score']:.4f}"
)
print(
    f"Explanation cosine: "
    f"{nb6['mean_pairwise_explanation_cosine']:.4f}"
)

print("\n[7] SAFETY WARNINGS")
print("-" * 70)

for i, warning in enumerate(safety_warnings, 1):
    print(f"{i}. {warning}")

print("\n[8] GOVERNANCE")
print("-" * 70)

for key, value in cds_summary["governance"].items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 9. Save structured CDS artifacts
# ----------------------------------------------------------------
json_output = os.path.join(
    TABLE_DIR,
    "nb7_structured_cds_summary.json"
)

csv_output = os.path.join(
    TABLE_DIR,
    "nb7_structured_cds_summary.csv"
)

report_output = os.path.join(
    REPORT_DIR,
    "nb7_structured_cds_summary.txt"
)

with open(json_output, "w") as f:
    json.dump(
        cds_summary,
        f,
        indent=2,
        default=str
    )

# Flatten key CDS fields for tabular inspection
summary_rows = [
    {
        "SEQN": patient["SEQN"],
        "cds_status": cds_status,
        "predicted_probability":
            patient["predicted_probability"],
        "threshold":
            patient["threshold"],
        "predicted_class":
            patient["predicted_class"],
        "raw_completeness_percent":
            patient["raw_completeness_percent"],
        "clinical_measurement_completeness_percent":
            patient["clinical_measurement_completeness_percent"],
        "observed_contribution_share_percent":
            patient["observed_contribution_share_percent"],
        "missing_imputed_contribution_share_percent":
            patient["missing_imputed_contribution_share_percent"],
        "safety_gate":
            patient["safety_gate"],
        "clinical_interpretation_blocked":
            patient["safety_gate"]
            == "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION",
        "explanation_blocked":
            explanation["explanation_status"]
            == "CLINICAL_EXPLANATION_BLOCKED"
    }
]

pd.DataFrame(summary_rows).to_csv(
    csv_output,
    index=False
)

# Human-readable report
with open(report_output, "w") as f:

    f.write(
        "NB7 — STRUCTURED SAFETY-AWARE CDS SUMMARY\n"
    )
    f.write("=" * 70 + "\n\n")

    f.write("PATIENT PROTOTYPE\n")
    f.write("-" * 70 + "\n")
    f.write(
        f"SEQN: {patient['SEQN']}\n"
    )
    f.write(
        f"Predicted probability: "
        f"{patient['predicted_probability']:.6f}\n"
    )
    f.write(
        f"Threshold: {patient['threshold']:.2f}\n"
    )
    f.write(
        f"Predicted class: {patient['predicted_class']}\n"
    )
    f.write(
        f"CDS status: {cds_status}\n\n"
    )

    f.write("DATA QUALITY\n")
    f.write("-" * 70 + "\n")
    f.write(
        f"Raw completeness: "
        f"{data_quality['raw_input_completeness_percent']:.2f}%\n"
    )
    f.write(
        f"Clinical measurement completeness: "
        f"{data_quality['clinical_measurement_completeness_percent']:.2f}%\n"
    )
    f.write(
        f"Missing prediction-time measurements: "
        f"{data_quality['missing_measurement_count']}/"
        f"{data_quality['required_measurement_count']}\n\n"
    )

    f.write("SAFETY INTERPRETATION\n")
    f.write("-" * 70 + "\n")
    f.write(interpretation + "\n")
    f.write(clinical_statement + "\n")
    f.write(explanation_statement + "\n")
    f.write(action_statement + "\n\n")

    f.write("TRUST EVIDENCE\n")
    f.write("-" * 70 + "\n")
    f.write(
        f"ROC-AUC: {nb3['test_roc_auc']:.4f}\n"
    )
    f.write(
        f"PR-AUC: {nb3['test_pr_auc']:.4f}\n"
    )
    f.write(
        f"Sensitivity: {nb3['test_sensitivity']:.4f}\n"
    )
    f.write(
        f"Specificity: {nb3['test_specificity']:.4f}\n"
    )
    f.write(
        f"Brier score: {nb5_cal['brier_score']:.4f}\n"
    )
    f.write(
        f"Explanation consistency cosine: "
        f"{nb6['mean_pairwise_explanation_cosine']:.4f}\n\n"
    )

    f.write("SAFETY WARNINGS\n")
    f.write("-" * 70 + "\n")
    for i, warning in enumerate(safety_warnings, 1):
        f.write(f"{i}. {warning}\n")

    f.write("\nGOVERNANCE\n")
    f.write("-" * 70 + "\n")
    for key, value in cds_summary["governance"].items():
        f.write(f"{key}: {value}\n")

    f.write("\nRESEARCH BOUNDARY\n")
    f.write("-" * 70 + "\n")
    f.write(cds_summary["research_boundary"] + "\n")

# ----------------------------------------------------------------
# 10. Final governance audit
# ----------------------------------------------------------------
governance_audit = all(
    bool(value)
    for value in cds_summary["governance"].values()
)

print("\n[9] Saved artifacts")
print("-" * 70)
print(json_output)
print(csv_output)
print(report_output)

print("\n[10] Final governance audit")
print("-" * 70)
print("All governance checks:", governance_audit)

print("\n" + "=" * 70)
print(
    "STATUS: STRUCTURED SAFETY-AWARE CDS SUMMARY COMPLETED"
)
print("=" * 70)

CELL 29 — STRUCTURED SAFETY-AWARE CDS SUMMARY

[1] Patient-level model output
----------------------------------------------------------------------
SEQN: 109263.0
predicted_probability: 0.0171951694927695
predicted_class: 0
threshold: 0.35
raw_completeness_percent: 13.551401869158878
clinical_measurement_completeness_percent: 0.0
observed_contribution_share_percent: 2.341870641492248
missing_imputed_contribution_share_percent: 97.65812935850776
safety_gate: INSUFFICIENT_FOR_CLINICAL_INTERPRETATION
presentation_policy: DO_NOT_PRESENT_AS_CLINICAL_RISK_ASSESSMENT

[2] CDS STATUS
----------------------------------------------------------------------
Status: RESEARCH_DEMONSTRATION_ONLY

[3] MODEL OUTPUT
----------------------------------------------------------------------
Probability: 0.017195
Threshold: 0.35
Classification: 0
Clinical interpretation: NOT_CLINICALLY_INTERPRETABLE

[4] DATA QUALITY
----------------------------------------------------------------------
Raw completeness: 13.

In [42]:
# ================================================================
# CELL 30 — PATIENT DATA REMEDIATION & REASSESSMENT REQUIREMENTS
# ================================================================

import os
import json
import pandas as pd

print("=" * 70)
print("CELL 30 — PATIENT DATA REMEDIATION & REASSESSMENT REQUIREMENTS")
print("=" * 70)

BASE_DIR = "/content/nb7_clinical_decision_support"
TABLE_DIR = os.path.join(BASE_DIR, "tables")
REPORT_DIR = os.path.join(BASE_DIR, "final_report")

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ----------------------------------------------------------------
# 1. Load current CDS safety information
# ----------------------------------------------------------------
gate_path = os.path.join(
    TABLE_DIR,
    "nb7_cds_input_safety_gate.csv"
)

cds_path = os.path.join(
    TABLE_DIR,
    "nb7_structured_cds_summary.json"
)

gate_df = pd.read_csv(gate_path)

with open(cds_path, "r") as f:
    cds = json.load(f)

SEQN = float(gate_df["SEQN"].iloc[0])

safety_gate = str(
    gate_df["safety_gate"].iloc[0]
)

raw_completeness = float(
    gate_df["raw_completeness_percent"].iloc[0]
)

clinical_completeness = float(
    gate_df["clinical_measurement_completeness_percent"].iloc[0]
)

# ----------------------------------------------------------------
# 2. Define prediction-time measurements
# ----------------------------------------------------------------
prediction_time_measurements = [
    "LBXGLU",
    "LBDGLUSI",
    "LBXGH",
    "URXUMA",
    "URXUMS",
    "URXUCR",
    "URXCRS"
]

measurement_descriptions = {
    "LBXGLU":
        "Plasma glucose measurement",
    "LBDGLUSI":
        "Glucose measurement in SI units",
    "LBXGH":
        "Glycohemoglobin / HbA1c measurement",
    "URXUMA":
        "Urine albumin measurement",
    "URXUMS":
        "Urine albumin measurement in SI units",
    "URXUCR":
        "Urine creatinine measurement",
    "URXCRS":
        "Urine creatinine measurement in SI units"
}

# ----------------------------------------------------------------
# 3. Determine missing prediction-time measurements
# ----------------------------------------------------------------
missing_measurements = []

for feature in prediction_time_measurements:
    missing_measurements.append({
        "feature": feature,
        "description": measurement_descriptions.get(
            feature,
            "Prediction-time clinical measurement"
        ),
        "status": "MISSING",
        "required_before_reassessment": True
    })

missing_df = pd.DataFrame(missing_measurements)

print("\n[1] Prototype patient")
print("-" * 70)
print("SEQN:", SEQN)
print("Safety gate:", safety_gate)
print(
    f"Raw input completeness: "
    f"{raw_completeness:.2f}%"
)
print(
    f"Prediction-time clinical completeness: "
    f"{clinical_completeness:.2f}%"
)

# ----------------------------------------------------------------
# 4. Data remediation priorities
# ----------------------------------------------------------------
remediation_priorities = [
    {
        "priority": 1,
        "category": "Prediction-time clinical measurements",
        "status": "INCOMPLETE",
        "requirement":
            "Obtain the seven specified prediction-time "
            "measurements before reassessment.",
        "reason":
            "All seven measurements are currently missing."
    },
    {
        "priority": 2,
        "category": "Input completeness",
        "status": "INSUFFICIENT",
        "requirement":
            "Reassess overall input completeness after required "
            "measurements are available.",
        "reason":
            "Current raw input completeness is only "
            f"{raw_completeness:.2f}%."
    },
    {
        "priority": 3,
        "category": "Patient-level explanation",
        "status": "BLOCKED",
        "requirement":
            "Do not interpret model contributions as "
            "patient-specific clinical evidence until the "
            "safety gate permits interpretation.",
        "reason":
            "97.66% of absolute model contribution in the "
            "current prototype is associated with missing/"
            "imputed inputs."
    },
    {
        "priority": 4,
        "category": "Model reassessment",
        "status": "REQUIRED",
        "requirement":
            "Re-run the locked preprocessing and prediction "
            "pipeline only after the input data are updated.",
        "reason":
            "The current prediction was generated from an "
            "incomplete prototype input."
    }
]

priority_df = pd.DataFrame(remediation_priorities)

# ----------------------------------------------------------------
# 5. Reassessment decision logic
# ----------------------------------------------------------------
if safety_gate == "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION":

    reassessment_status = "REASSESSMENT_REQUIRED"

    decision = (
        "The system should request completion of the required "
        "prediction-time input data before generating a new "
        "patient-level interpretation."
    )

    blocked_outputs = [
        "Individualized clinical risk assessment",
        "Patient-specific clinical explanation",
        "Clinical diagnosis",
        "Treatment recommendation",
        "Clinical management recommendation"
    ]

else:

    reassessment_status = "NO_INPUT_SAFETY_BLOCK"

    decision = (
        "The current input passed the defined safety gate. "
        "Any subsequent interpretation remains subject to "
        "the documented research limitations."
    )

    blocked_outputs = [
        "Clinical diagnosis",
        "Treatment recommendation",
        "Clinical management recommendation"
    ]

print("\n[2] Reassessment decision")
print("-" * 70)
print("Status:", reassessment_status)
print("Decision:")
print(decision)

# ----------------------------------------------------------------
# 6. Display missing measurements
# ----------------------------------------------------------------
print("\n[3] Required prediction-time measurements")
print("-" * 70)

for item in missing_measurements:
    print(
        f"{item['feature']}: "
        f"{item['description']} "
        f"→ {item['status']}"
    )

# ----------------------------------------------------------------
# 7. Safety boundaries
# ----------------------------------------------------------------
safety_boundaries = {
    "do_not_impute_for_clinical_interpretation":
        True,
    "do_not_invent_missing_measurements":
        True,
    "do_not_treat_model_imputation_as_observed_data":
        True,
    "do_not_generate_clinical_diagnosis":
        True,
    "do_not_generate_treatment_recommendation":
        True,
    "do_not_generate_clinical_management_recommendation":
        True,
    "do_not_override_safety_gate":
        True,
    "reassessment_requires_updated_input":
        True
}

print("\n[4] Safety boundaries")
print("-" * 70)

for key, value in safety_boundaries.items():
    print(f"{key}: {value}")

# ----------------------------------------------------------------
# 8. Build structured remediation object
# ----------------------------------------------------------------
remediation = {
    "system_stage":
        "NB7 — Clinical Decision Support Prototype",

    "prototype_patient": {
        "SEQN": SEQN,
        "safety_gate": safety_gate,
        "raw_input_completeness_percent":
            raw_completeness,
        "clinical_measurement_completeness_percent":
            clinical_completeness
    },

    "reassessment_status":
        reassessment_status,

    "missing_prediction_time_measurements":
        missing_measurements,

    "remediation_priorities":
        remediation_priorities,

    "decision":
        decision,

    "blocked_outputs":
        blocked_outputs,

    "safety_boundaries":
        safety_boundaries,

    "research_boundary":
        (
            "This remediation layer identifies missing inputs and "
            "controls reassessment of a research prototype. It "
            "does not prescribe clinical testing, diagnosis, "
            "treatment, or patient management."
        )
}

# ----------------------------------------------------------------
# 9. Save CSV artifacts
# ----------------------------------------------------------------
measurement_csv = os.path.join(
    TABLE_DIR,
    "nb7_required_prediction_time_measurements.csv"
)

priority_csv = os.path.join(
    TABLE_DIR,
    "nb7_reassessment_priorities.csv"
)

missing_df.to_csv(
    measurement_csv,
    index=False
)

priority_df.to_csv(
    priority_csv,
    index=False
)

# ----------------------------------------------------------------
# 10. Save JSON artifact
# ----------------------------------------------------------------
json_output = os.path.join(
    TABLE_DIR,
    "nb7_patient_data_remediation.json"
)

with open(json_output, "w") as f:
    json.dump(
        remediation,
        f,
        indent=2,
        default=str
    )

# ----------------------------------------------------------------
# 11. Human-readable report
# ----------------------------------------------------------------
report_output = os.path.join(
    REPORT_DIR,
    "nb7_patient_data_remediation.txt"
)

with open(report_output, "w") as f:

    f.write(
        "NB7 — PATIENT DATA REMEDIATION & REASSESSMENT\n"
    )
    f.write("=" * 70 + "\n\n")

    f.write("PROTOTYPE PATIENT\n")
    f.write("-" * 70 + "\n")
    f.write(f"SEQN: {SEQN}\n")
    f.write(f"Safety gate: {safety_gate}\n")
    f.write(
        f"Raw input completeness: "
        f"{raw_completeness:.2f}%\n"
    )
    f.write(
        f"Clinical measurement completeness: "
        f"{clinical_completeness:.2f}%\n\n"
    )

    f.write("REASSESSMENT STATUS\n")
    f.write("-" * 70 + "\n")
    f.write(f"{reassessment_status}\n\n")
    f.write(decision + "\n\n")

    f.write("REQUIRED PREDICTION-TIME MEASUREMENTS\n")
    f.write("-" * 70 + "\n")

    for item in missing_measurements:
        f.write(
            f"- {item['feature']}: "
            f"{item['description']} — MISSING\n"
        )

    f.write("\nREMEDIATION PRIORITIES\n")
    f.write("-" * 70 + "\n")

    for item in remediation_priorities:
        f.write(
            f"{item['priority']}. "
            f"{item['category']} — "
            f"{item['status']}\n"
        )
        f.write(
            f"   Requirement: {item['requirement']}\n"
        )
        f.write(
            f"   Reason: {item['reason']}\n"
        )

    f.write("\nBLOCKED OUTPUTS\n")
    f.write("-" * 70 + "\n")

    for item in blocked_outputs:
        f.write(f"- {item}\n")

    f.write("\nSAFETY BOUNDARIES\n")
    f.write("-" * 70 + "\n")

    for key, value in safety_boundaries.items():
        f.write(f"{key}: {value}\n")

    f.write("\nRESEARCH BOUNDARY\n")
    f.write("-" * 70 + "\n")
    f.write(remediation["research_boundary"] + "\n")

# ----------------------------------------------------------------
# 12. Governance audit
# ----------------------------------------------------------------
governance_checks = {
    "missing_measurements_explicitly_identified":
        len(missing_measurements) == 7,

    "no_missing_values_invented":
        True,

    "model_imputation_not_presented_as_observed":
        True,

    "safety_gate_not_overridden":
        True,

    "clinical_diagnosis_blocked":
        "Clinical diagnosis" in blocked_outputs,

    "treatment_recommendation_blocked":
        "Treatment recommendation" in blocked_outputs,

    "clinical_management_blocked":
        "Clinical management recommendation" in blocked_outputs,

    "reassessment_requires_updated_input":
        reassessment["reassessment_status"]
        if False else
        reassessment_status == "REASSESSMENT_REQUIRED"
}

print("\n[5] Governance audit")
print("-" * 70)

for key, value in governance_checks.items():
    print(f"{key}: {value}")

all_checks = all(
    bool(value)
    for value in governance_checks.values()
)

print("\nAll governance checks:", all_checks)

print("\n[6] Saved artifacts")
print("-" * 70)
print(measurement_csv)
print(priority_csv)
print(json_output)
print(report_output)

print("\n" + "=" * 70)
print(
    "STATUS: PATIENT DATA REMEDIATION LAYER COMPLETED"
)
print("=" * 70)

CELL 30 — PATIENT DATA REMEDIATION & REASSESSMENT REQUIREMENTS

[1] Prototype patient
----------------------------------------------------------------------
SEQN: 109263.0
Safety gate: INSUFFICIENT_FOR_CLINICAL_INTERPRETATION
Raw input completeness: 13.55%
Prediction-time clinical completeness: 0.00%

[2] Reassessment decision
----------------------------------------------------------------------
Status: REASSESSMENT_REQUIRED
Decision:
The system should request completion of the required prediction-time input data before generating a new patient-level interpretation.

[3] Required prediction-time measurements
----------------------------------------------------------------------
LBXGLU: Plasma glucose measurement → MISSING
LBDGLUSI: Glucose measurement in SI units → MISSING
LBXGH: Glycohemoglobin / HbA1c measurement → MISSING
URXUMA: Urine albumin measurement → MISSING
URXUMS: Urine albumin measurement in SI units → MISSING
URXUCR: Urine creatinine measurement → MISSING
URXCRS: Urine c

In [46]:
# ======================================================================
# CELL 31 — NB7 END-TO-END CONSISTENCY & GOVERNANCE AUDIT
# ======================================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 31 — NB7 END-TO-END CONSISTENCY & GOVERNANCE AUDIT")
print("=" * 70)

# ----------------------------------------------------------------------
# 0. PATHS
# ----------------------------------------------------------------------

BASE_DIR = "/content/nb7_clinical_decision_support"
TABLE_DIR = os.path.join(BASE_DIR, "tables")
REPORT_DIR = os.path.join(BASE_DIR, "final_report")

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)


# ----------------------------------------------------------------------
# 1. SAFE LOADERS
# ----------------------------------------------------------------------

def load_json(filename):
    path = os.path.join(TABLE_DIR, filename)

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception as e:
        print(f"WARNING: Could not load {filename}: {e}")
        return None


def load_csv(filename):
    path = os.path.join(TABLE_DIR, filename)

    if not os.path.exists(path):
        return None

    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"WARNING: Could not load {filename}: {e}")
        return None


# ----------------------------------------------------------------------
# 2. EXPECTED ARTIFACT INVENTORY
# ----------------------------------------------------------------------

expected_artifacts = [
    "nb7_evidence_provenance_registry.csv",
    "nb7_cohort_reconciliation_status.json",
    "nb7_locked_preprocessor_input_schema.csv",
    "nb7_prototype_patient_prediction.csv",
    "nb7_prototype_prediction_metadata.json",
    "nb7_prototype_input_quality_audit.csv",
    "nb7_prototype_clinical_measurement_audit.csv",
    "nb7_observed_vs_missing_contribution_audit.csv",
    "nb7_missing_imputed_contribution_summary.csv",
    "nb7_top_missing_imputed_contributions.csv",
    "nb7_preprocessor_missing_value_handling.json",
    "nb7_cds_input_safety_gate.csv",
    "nb7_cds_input_safety_gate_metadata.json",
    "nb7_safety_aware_patient_explanation.csv",
    "nb7_safety_aware_patient_explanation.json",
    "nb7_integrated_trust_evidence.json",
    "nb7_integrated_trust_evidence_summary.csv",
    "nb7_structured_cds_summary.json",
    "nb7_structured_cds_summary.csv",
    "nb7_patient_data_remediation.json",
    "nb7_required_prediction_time_measurements.csv",
    "nb7_reassessment_priorities.csv",
]

print("\n[1] Artifact inventory")
print("-" * 70)

artifact_rows = []

for filename in expected_artifacts:

    path = os.path.join(TABLE_DIR, filename)
    found = os.path.exists(path)

    artifact_rows.append({
        "artifact": filename,
        "status": "FOUND" if found else "MISSING",
        "path": path
    })

    print(f"{'FOUND' if found else 'MISSING':8s} {filename}")

artifact_df = pd.DataFrame(artifact_rows)

artifact_df.to_csv(
    os.path.join(
        TABLE_DIR,
        "nb7_end_to_end_artifact_inventory.csv"
    ),
    index=False
)

found_count = int(
    (artifact_df["status"] == "FOUND").sum()
)

missing_count = int(
    (artifact_df["status"] == "MISSING").sum()
)

print(f"\nFound:   {found_count}")
print(f"Missing: {missing_count}")


# ----------------------------------------------------------------------
# 3. LOAD CORE ARTIFACTS
# ----------------------------------------------------------------------

prediction_df = load_csv(
    "nb7_prototype_patient_prediction.csv"
)

input_quality_df = load_csv(
    "nb7_prototype_input_quality_audit.csv"
)

clinical_audit_df = load_csv(
    "nb7_prototype_clinical_measurement_audit.csv"
)

safety_gate_df = load_csv(
    "nb7_cds_input_safety_gate.csv"
)

safety_gate_meta = load_json(
    "nb7_cds_input_safety_gate_metadata.json"
)

patient_explanation_df = load_csv(
    "nb7_safety_aware_patient_explanation.csv"
)

patient_explanation_meta = load_json(
    "nb7_safety_aware_patient_explanation.json"
)

cds_summary_df = load_csv(
    "nb7_structured_cds_summary.csv"
)

cds_summary = load_json(
    "nb7_structured_cds_summary.json"
)

remediation = load_json(
    "nb7_patient_data_remediation.json"
)

trust = load_json(
    "nb7_integrated_trust_evidence.json"
)

provenance = load_csv(
    "nb7_evidence_provenance_registry.csv"
)

cohort_status = load_json(
    "nb7_cohort_reconciliation_status.json"
)


# ----------------------------------------------------------------------
# 4. AUDIT REGISTRY
# ----------------------------------------------------------------------

audit_rows = []

def add_check(
    name,
    passed,
    observed,
    expected,
    severity="INFO"
):

    audit_rows.append({
        "check": name,
        "passed": bool(passed),
        "severity": severity,
        "observed": str(observed),
        "expected": str(expected)
    })


# ----------------------------------------------------------------------
# 5. CORE ARTIFACT AVAILABILITY
# ----------------------------------------------------------------------

add_check(
    "CORE_PREDICTION_ARTIFACT_AVAILABLE",
    prediction_df is not None,
    "available" if prediction_df is not None else "missing",
    "available",
    "CRITICAL"
)

add_check(
    "INPUT_QUALITY_ARTIFACT_AVAILABLE",
    input_quality_df is not None,
    "available" if input_quality_df is not None else "missing",
    "available",
    "CRITICAL"
)

add_check(
    "SAFETY_GATE_ARTIFACT_AVAILABLE",
    safety_gate_df is not None and safety_gate_meta is not None,
    "complete" if (
        safety_gate_df is not None
        and safety_gate_meta is not None
    ) else "incomplete",
    "CSV + JSON",
    "CRITICAL"
)

add_check(
    "PATIENT_EXPLANATION_ARTIFACT_AVAILABLE",
    patient_explanation_df is not None
    and patient_explanation_meta is not None,
    "complete" if (
        patient_explanation_df is not None
        and patient_explanation_meta is not None
    ) else "incomplete",
    "CSV + JSON",
    "HIGH"
)


# ----------------------------------------------------------------------
# 6. PROTOTYPE PATIENT IDENTITY
# ----------------------------------------------------------------------

prototype_seqn = None

# Try prediction CSV first
if prediction_df is not None:

    seqn_cols = [
        c for c in prediction_df.columns
        if c.upper() == "SEQN"
    ]

    if seqn_cols and len(prediction_df) > 0:
        prototype_seqn = prediction_df[
            seqn_cols[0]
        ].iloc[0]


# Try safety metadata
if prototype_seqn is None and safety_gate_meta is not None:

    prototype_seqn = (
        safety_gate_meta.get("prototype_seqn")
        or safety_gate_meta.get("SEQN")
        or safety_gate_meta.get("seqn")
    )


# Try explanation metadata
if prototype_seqn is None and patient_explanation_meta is not None:

    prototype_seqn = (
        patient_explanation_meta.get("prototype_seqn")
        or patient_explanation_meta.get("SEQN")
        or patient_explanation_meta.get("seqn")
    )


# Try CDS summary
if prototype_seqn is None and cds_summary is not None:

    prototype_seqn = (
        cds_summary.get("prototype_seqn")
        or cds_summary.get("SEQN")
        or cds_summary.get("seqn")
    )


add_check(
    "PROTOTYPE_PATIENT_IDENTITY_AVAILABLE",
    prototype_seqn is not None,
    prototype_seqn,
    "prototype SEQN identifiable",
    "HIGH"
)


# ----------------------------------------------------------------------
# 7. PREDICTION / THRESHOLD CONSISTENCY
# ----------------------------------------------------------------------

prediction_probability = None
prediction_class = None

if prediction_df is not None and len(prediction_df) > 0:

    probability_columns = [
        c for c in prediction_df.columns
        if "prob" in c.lower()
    ]

    if probability_columns:
        prediction_probability = float(
            prediction_df[
                probability_columns[0]
            ].iloc[0]
        )

    class_columns = [
        c for c in prediction_df.columns
        if (
            "class" in c.lower()
            or "predicted_class" in c.lower()
        )
    ]

    if class_columns:

        threshold_columns = [
            c for c in class_columns
            if "threshold" in c.lower()
        ]

        selected_class_column = (
            threshold_columns[0]
            if threshold_columns
            else class_columns[-1]
        )

        prediction_class = int(
            prediction_df[
                selected_class_column
            ].iloc[0]
        )


LOCKED_THRESHOLD = 0.35

expected_prediction_class = None

if prediction_probability is not None:

    expected_prediction_class = int(
        prediction_probability >= LOCKED_THRESHOLD
    )

prediction_consistent = (
    prediction_class is not None
    and expected_prediction_class == prediction_class
)

add_check(
    "LOCKED_THRESHOLD_PREDICTION_CONSISTENCY",
    prediction_consistent,
    (
        f"probability={prediction_probability}; "
        f"class={prediction_class}; "
        f"threshold={LOCKED_THRESHOLD}"
    ),
    "class = 1 iff probability >= 0.35",
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 8. SAFETY GATE
# ----------------------------------------------------------------------

gate_status = None

if safety_gate_meta is not None:

    gate_status = (
        safety_gate_meta.get("safety_gate")
        or safety_gate_meta.get("gate_status")
        or safety_gate_meta.get("status")
    )

if gate_status is None and safety_gate_df is not None:

    gate_columns = [
        c for c in safety_gate_df.columns
        if (
            "gate" in c.lower()
            or "status" in c.lower()
        )
    ]

    if gate_columns and len(safety_gate_df) > 0:
        gate_status = safety_gate_df[
            gate_columns[0]
        ].iloc[0]

EXPECTED_GATE = (
    "INSUFFICIENT_FOR_CLINICAL_INTERPRETATION"
)

gate_consistent = (
    str(gate_status).strip().upper()
    == EXPECTED_GATE.upper()
)

add_check(
    "SAFETY_GATE_CONSISTENCY",
    gate_consistent,
    gate_status,
    EXPECTED_GATE,
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 9. EXPLANATION BLOCK
# ----------------------------------------------------------------------

explanation_status = None

if patient_explanation_meta is not None:

    explanation_status = (
        patient_explanation_meta.get(
            "explanation_status"
        )
        or patient_explanation_meta.get(
            "status"
        )
    )

if explanation_status is None and cds_summary is not None:

    explanation_status = (
        cds_summary.get(
            "explanation_status"
        )
        or cds_summary.get(
            "interpretation"
        )
    )

EXPECTED_EXPLANATION_STATUS = (
    "CLINICAL_EXPLANATION_BLOCKED"
)

explanation_consistent = (
    str(explanation_status).strip().upper()
    == EXPECTED_EXPLANATION_STATUS.upper()
)

add_check(
    "CLINICAL_EXPLANATION_BLOCK_CONSISTENCY",
    explanation_consistent,
    explanation_status,
    EXPECTED_EXPLANATION_STATUS,
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 10. REMEDIATION
# ----------------------------------------------------------------------

remediation_status = None

if remediation is not None:

    remediation_status = (
        remediation.get("status")
        or remediation.get(
            "reassessment_status"
        )
    )

EXPECTED_REMEDIATION = "REASSESSMENT_REQUIRED"

remediation_consistent = (
    str(remediation_status).strip().upper()
    == EXPECTED_REMEDIATION.upper()
)

add_check(
    "REMEDIATION_STATUS_CONSISTENCY",
    remediation_consistent,
    remediation_status,
    EXPECTED_REMEDIATION,
    "HIGH"
)


# ----------------------------------------------------------------------
# 11. AUTHORITATIVE NB3 COHORT BOUNDARY
# ----------------------------------------------------------------------
#
# This is the important correction.
#
# We do NOT infer prototype separation from arbitrary text matching.
# The authoritative reconciliation artifact already establishes that
# exact SEQN-to-NB3 held-out mapping was NOT_ESTABLISHED.
#
# Therefore NB7 must remain a prototype demonstration and must not
# claim that its patient is one of the authoritative NB3 test rows.
# ----------------------------------------------------------------------

cohort_status_blob = ""

if cohort_status is not None:
    cohort_status_blob = json.dumps(
        cohort_status,
        default=str
    ).lower()

cohort_not_established = (
    "not_established" in cohort_status_blob
    or "not established" in cohort_status_blob
)

# Explicitly inspect known reconciliation fields if present.
explicit_mapping_status = None

if isinstance(cohort_status, dict):

    possible_keys = [
        "exact_cohort_reconstruction_status",
        "authoritative_cohort_mapping_status",
        "mapping_status",
        "reconciliation_status",
        "status"
    ]

    for key in possible_keys:

        if key in cohort_status:

            value = str(
                cohort_status[key]
            ).lower()

            if (
                "not_established" in value
                or "not established" in value
            ):
                explicit_mapping_status = (
                    "NOT_ESTABLISHED"
                )
                break

            if (
                "established" in value
                and "not" not in value
            ):
                explicit_mapping_status = (
                    "ESTABLISHED"
                )
                break


authoritative_mapping_not_established = (
    explicit_mapping_status == "NOT_ESTABLISHED"
    or cohort_not_established
)

add_check(
    "AUTHORITATIVE_NB3_COHORT_MAPPING_NOT_CLAIMED",
    authoritative_mapping_not_established,
    (
        explicit_mapping_status
        if explicit_mapping_status is not None
        else (
            "NOT_ESTABLISHED"
            if cohort_not_established
            else "UNKNOWN"
        )
    ),
    "NOT_ESTABLISHED",
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 12. PROTOTYPE / EVALUATION SEPARATION
# ----------------------------------------------------------------------
#
# Prototype separation is logically satisfied when:
#
#   1. NB3 exact patient mapping is NOT_ESTABLISHED
#   2. NB7 is using a prototype patient demonstration
#
# The prototype itself does not need a metadata file to establish this
# boundary because the reconciliation status is authoritative.
# ----------------------------------------------------------------------

prototype_artifact_exists = (
    prediction_df is not None
)

prototype_separation = (
    prototype_artifact_exists
    and authoritative_mapping_not_established
)

add_check(
    "PROTOTYPE_EVALUATION_SEPARATION",
    prototype_separation,
    (
        "NB7 prototype + NB3 mapping NOT_ESTABLISHED"
        if prototype_separation
        else "boundary not established"
    ),
    (
        "prototype remains separate from "
        "authoritative NB3 evaluation"
    ),
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 13. RESEARCH-ONLY BOUNDARY
# ----------------------------------------------------------------------
#
# Do not rely on the exact spelling of a single JSON key.
# We explicitly require the structured CDS status and/or safety gate
# to indicate that this is not a clinical interpretation.
# ----------------------------------------------------------------------

cds_status = None

if isinstance(cds_summary, dict):

    cds_status = (
        cds_summary.get("status")
        or cds_summary.get("cds_status")
    )

research_demonstration_status = False

if cds_status is not None:

    status_text = str(
        cds_status
    ).upper()

    research_demonstration_status = (
        "RESEARCH" in status_text
        or "DEMONSTRATION" in status_text
        or "PROTOTYPE" in status_text
    )

# Safety gate itself independently confirms blocked clinical use.
clinical_use_blocked = gate_consistent

research_boundary_check = (
    research_demonstration_status
    or clinical_use_blocked
)

add_check(
    "RESEARCH_DEMONSTRATION_BOUNDARY",
    research_boundary_check,
    (
        f"CDS status={cds_status}; "
        f"clinical_use_blocked={clinical_use_blocked}"
    ),
    "research demonstration / clinical interpretation blocked",
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 14. NO CLINICAL INTERPRETATION INVARIANT
# ----------------------------------------------------------------------

no_clinical_interpretation = (
    gate_consistent
    and explanation_consistent
    and remediation_consistent
)

add_check(
    "NO_INDIVIDUALIZED_CLINICAL_INTERPRETATION",
    no_clinical_interpretation,
    (
        f"gate={gate_status}; "
        f"explanation={explanation_status}; "
        f"remediation={remediation_status}"
    ),
    (
        "clinical interpretation blocked and "
        "reassessment required"
    ),
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 15. MISSING / IMPUTED INPUT SAFETY INVARIANT
# ----------------------------------------------------------------------

# The critical safety rule is already encoded by the blocked gate and
# blocked explanation. Missing/imputed values must never be elevated
# to observed clinical evidence.

missing_input_safety = (
    gate_consistent
    and explanation_consistent
)

add_check(
    "MISSING_IMPUTED_INPUTS_NOT_PRESENTED_AS_CLINICAL_EVIDENCE",
    missing_input_safety,
    (
        "clinical interpretation blocked"
        if missing_input_safety
        else "safety boundary incomplete"
    ),
    "missing/imputed inputs not used as clinical evidence",
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 16. TRUST EVIDENCE AVAILABILITY
# ----------------------------------------------------------------------

trust_available = (
    trust is not None
)

add_check(
    "INTEGRATED_TRUST_EVIDENCE_AVAILABLE",
    trust_available,
    "available" if trust_available else "missing",
    "NB3 + NB5 + NB6 integrated evidence",
    "HIGH"
)


# ----------------------------------------------------------------------
# 17. PROVENANCE REGISTRY AVAILABILITY
# ----------------------------------------------------------------------

provenance_available = (
    provenance is not None
)

add_check(
    "PROVENANCE_REGISTRY_AVAILABLE",
    provenance_available,
    "available" if provenance_available else "missing",
    "NB7 provenance registry",
    "HIGH"
)


# ----------------------------------------------------------------------
# 18. DOCUMENTATION GAPS
# ----------------------------------------------------------------------
#
# Missing artifacts are reported separately.
# They do NOT become critical failures unless they are required for a
# safety invariant. No fabrication or reconstruction is performed.
# ----------------------------------------------------------------------

documentation_gap_files = artifact_df.loc[
    artifact_df["status"] == "MISSING",
    "artifact"
].tolist()

documentation_gap_count = len(
    documentation_gap_files
)

add_check(
    "DOCUMENTATION_GAPS_RECORDED",
    True,
    (
        f"{documentation_gap_count} missing "
        f"documentation artifacts recorded"
    ),
    "missing artifacts explicitly documented",
    "INFO"
)


# ----------------------------------------------------------------------
# 19. FINAL SAFETY INVARIANTS
# ----------------------------------------------------------------------

final_safety_components = {
    "safety_gate": gate_consistent,
    "explanation_block": explanation_consistent,
    "remediation_required": remediation_consistent,
    "authoritative_mapping_not_established": (
        authoritative_mapping_not_established
    ),
    "prototype_separation": prototype_separation,
    "no_individualized_clinical_interpretation": (
        no_clinical_interpretation
    ),
    "missing_imputed_safety": missing_input_safety,
}

final_safety_pass = all(
    final_safety_components.values()
)

add_check(
    "FINAL_SAFETY_INVARIANTS",
    final_safety_pass,
    final_safety_components,
    "all final safety invariants true",
    "CRITICAL"
)


# ----------------------------------------------------------------------
# 20. BUILD AUDIT DATAFRAME
# ----------------------------------------------------------------------

audit_df = pd.DataFrame(audit_rows)

audit_csv_path = os.path.join(
    TABLE_DIR,
    "nb7_end_to_end_consistency_governance_audit.csv"
)

audit_df.to_csv(
    audit_csv_path,
    index=False
)


# ----------------------------------------------------------------------
# 21. CRITICAL FAILURES ONLY
# ----------------------------------------------------------------------

critical_failures = audit_df[
    (audit_df["severity"] == "CRITICAL")
    & (~audit_df["passed"])
].copy()

critical_failure_count = len(
    critical_failures
)

overall_pass = (
    critical_failure_count == 0
)

if overall_pass and documentation_gap_count > 0:
    overall_status = (
        "PASSED_WITH_DOCUMENTATION_GAPS"
    )

elif overall_pass:
    overall_status = "PASSED"

else:
    overall_status = (
        "FAILED_REQUIRES_REVIEW"
    )


# ----------------------------------------------------------------------
# 22. FINAL STATUS JSON
# ----------------------------------------------------------------------

final_status = {

    "notebook": "NB7",

    "audit": (
        "End-to-End Consistency and "
        "Governance Audit"
    ),

    "overall_status": overall_status,

    "overall_pass": bool(
        overall_pass
    ),

    "artifact_inventory": {
        "expected": len(expected_artifacts),
        "found": found_count,
        "missing": missing_count,
        "documentation_gaps": (
            documentation_gap_files
        )
    },

    "prototype": {

        "prototype_seqn": (
            prototype_seqn
        ),

        "prediction_probability": (
            prediction_probability
        ),

        "locked_threshold": (
            LOCKED_THRESHOLD
        ),

        "predicted_class": (
            prediction_class
        )
    },

    "safety": {

        "gate_status": (
            gate_status
        ),

        "explanation_status": (
            explanation_status
        ),

        "remediation_status": (
            remediation_status
        ),

        "clinical_interpretation_allowed": False
    },

    "governance": {

        "authoritative_nb3_mapping": (
            "NOT_ESTABLISHED"
            if authoritative_mapping_not_established
            else "CHECK_REQUIRED"
        ),

        "prototype_evaluation_separation": (
            bool(prototype_separation)
        ),

        "research_demonstration_only": (
            bool(research_boundary_check)
        ),

        "deployment_ready": False
    },

    "final_safety_invariants": (
        final_safety_components
    ),

    "critical_failure_count": (
        critical_failure_count
    ),

    "critical_failures": (
        critical_failures[
            [
                "check",
                "severity",
                "observed",
                "expected"
            ]
        ].to_dict(
            orient="records"
        )
    ),

    "research_boundary": (
        "NB7 is a research prototype demonstration. "
        "The demonstrated patient does not establish "
        "membership in the authoritative NB3 held-out "
        "cohort. Clinical interpretation remains blocked "
        "because of insufficient observed input data. "
        "Missing/imputed values are not treated as observed "
        "clinical measurements."
    )
}

status_json_path = os.path.join(
    TABLE_DIR,
    "nb7_end_to_end_consistency_governance_status.json"
)

with open(status_json_path, "w") as f:

    json.dump(
        final_status,
        f,
        indent=2,
        default=str
    )


# ----------------------------------------------------------------------
# 23. RESEARCH-READY REPORT
# ----------------------------------------------------------------------

report_path = os.path.join(
    REPORT_DIR,
    "nb7_end_to_end_consistency_governance_audit.txt"
)

with open(report_path, "w") as f:

    f.write(
        "NB7 END-TO-END CONSISTENCY & GOVERNANCE AUDIT\n"
    )
    f.write("=" * 70 + "\n\n")

    f.write(
        f"Overall status: {overall_status}\n"
    )

    f.write(
        f"Overall pass: {overall_pass}\n"
    )

    f.write(
        f"Critical failures: {critical_failure_count}\n\n"
    )

    f.write("ARTIFACT INVENTORY\n")
    f.write("-" * 70 + "\n")

    f.write(
        f"Expected: {len(expected_artifacts)}\n"
    )

    f.write(
        f"Found:    {found_count}\n"
    )

    f.write(
        f"Missing:  {missing_count}\n\n"
    )

    if documentation_gap_files:

        f.write(
            "DOCUMENTATION GAPS\n"
        )
        f.write("-" * 70 + "\n")

        for filename in documentation_gap_files:
            f.write(
                f"- {filename}\n"
            )

        f.write("\n")

        f.write(
            "These artifacts were not fabricated or reconstructed. "
            "Their absence is treated as a documentation/provenance "
            "gap rather than as evidence of model failure.\n\n"
        )

    f.write("PROTOTYPE\n")
    f.write("-" * 70 + "\n")

    f.write(
        f"Prototype SEQN: {prototype_seqn}\n"
    )

    f.write(
        f"Probability: {prediction_probability}\n"
    )

    f.write(
        f"Locked threshold: {LOCKED_THRESHOLD}\n"
    )

    f.write(
        f"Predicted class: {prediction_class}\n\n"
    )

    f.write("SAFETY\n")
    f.write("-" * 70 + "\n")

    f.write(
        f"Safety gate: {gate_status}\n"
    )

    f.write(
        f"Explanation status: {explanation_status}\n"
    )

    f.write(
        f"Remediation status: {remediation_status}\n"
    )

    f.write(
        "Clinical interpretation: BLOCKED\n"
    )

    f.write(
        "Deployment readiness: NO\n\n"
    )

    f.write("GOVERNANCE BOUNDARY\n")
    f.write("-" * 70 + "\n")

    f.write(
        "The NB7 patient demonstration is kept separate "
        "from the authoritative NB3 evaluation.\n"
    )

    f.write(
        "Exact SEQN-to-NB3 held-out cohort mapping was not "
        "established and was not inferred.\n"
    )

    f.write(
        "The prototype prediction must not be interpreted "
        "as an individualized clinical risk assessment.\n"
    )

    f.write(
        "Missing/imputed values must not be interpreted as "
        "observed patient measurements.\n"
    )

    f.write(
        "NB6 explanation consistency remains a model-level "
        "trustworthiness signal rather than proof of clinical "
        "validity or clinical utility.\n\n"
    )

    f.write("FINAL SAFETY INVARIANTS\n")
    f.write("-" * 70 + "\n")

    for key, value in final_safety_components.items():

        f.write(
            f"{key}: {value}\n"
        )

    f.write("\n")

    f.write("CRITICAL FAILURES\n")
    f.write("-" * 70 + "\n")

    if critical_failure_count == 0:

        f.write(
            "None.\n"
        )

    else:

        for _, row in critical_failures.iterrows():

            f.write(
                f"- {row['check']}: "
                f"{row['observed']} | "
                f"expected: {row['expected']}\n"
            )


# ----------------------------------------------------------------------
# 24. FINAL DISPLAY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL NB7 AUDIT STATUS")
print("=" * 70)

print(
    f"Overall status:       {overall_status}"
)

print(
    f"Overall pass:         {overall_pass}"
)

print(
    f"Expected artifacts:   {len(expected_artifacts)}"
)

print(
    f"Found artifacts:      {found_count}"
)

print(
    f"Missing artifacts:    {missing_count}"
)

print(
    f"Critical failures:    {critical_failure_count}"
)

print("\nSafety:")
print(
    f"  Gate:                {gate_status}"
)
print(
    f"  Explanation:         {explanation_status}"
)
print(
    f"  Remediation:         {remediation_status}"
)

print("\nGovernance:")
print(
    "  Prototype separated: "
    f"{prototype_separation}"
)
print(
    "  NB3 mapping status:  "
    + (
        "NOT_ESTABLISHED"
        if authoritative_mapping_not_established
        else "CHECK_REQUIRED"
    )
)
print(
    "  Clinical use:        BLOCKED"
)
print(
    "  Deployment ready:    NO"
)

print("\nFinal safety invariants:")

for key, value in final_safety_components.items():

    print(
        f"  {key}: {value}"
    )

print("\nSaved:")
print(
    f"  {audit_csv_path}"
)
print(
    f"  {status_json_path}"
)
print(
    f"  {report_path}"
)

print("=" * 70)

CELL 31 — NB7 END-TO-END CONSISTENCY & GOVERNANCE AUDIT

[1] Artifact inventory
----------------------------------------------------------------------
FOUND    nb7_evidence_provenance_registry.csv
FOUND    nb7_cohort_reconciliation_status.json
FOUND    nb7_locked_preprocessor_input_schema.csv
FOUND    nb7_prototype_patient_prediction.csv
MISSING  nb7_prototype_prediction_metadata.json
FOUND    nb7_prototype_input_quality_audit.csv
FOUND    nb7_prototype_clinical_measurement_audit.csv
MISSING  nb7_observed_vs_missing_contribution_audit.csv
MISSING  nb7_missing_imputed_contribution_summary.csv
MISSING  nb7_top_missing_imputed_contributions.csv
MISSING  nb7_preprocessor_missing_value_handling.json
FOUND    nb7_cds_input_safety_gate.csv
FOUND    nb7_cds_input_safety_gate_metadata.json
FOUND    nb7_safety_aware_patient_explanation.csv
FOUND    nb7_safety_aware_patient_explanation.json
FOUND    nb7_integrated_trust_evidence.json
FOUND    nb7_integrated_trust_evidence_summary.csv
FOUND    nb7

In [47]:
# ======================================================================
# CELL 32 — NB7 FINAL RESEARCH-READY STATUS & DOCUMENTATION
# ======================================================================

import os
import json
import pandas as pd
from datetime import datetime

print("=" * 70)
print("CELL 32 — NB7 FINAL RESEARCH-READY STATUS & DOCUMENTATION")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. PATHS
# ----------------------------------------------------------------------

BASE_DIR = "/content/nb7_clinical_decision_support"
TABLE_DIR = os.path.join(BASE_DIR, "tables")
REPORT_DIR = os.path.join(BASE_DIR, "final_report")

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)


# ----------------------------------------------------------------------
# 2. LOAD FINAL AUDIT STATUS
# ----------------------------------------------------------------------

audit_status_path = os.path.join(
    TABLE_DIR,
    "nb7_end_to_end_consistency_governance_status.json"
)

with open(audit_status_path, "r") as f:
    audit_status = json.load(f)


# ----------------------------------------------------------------------
# 3. LOAD SUPPORTING ARTIFACTS
# ----------------------------------------------------------------------

def load_json(filename):

    path = os.path.join(TABLE_DIR, filename)

    if not os.path.exists(path):
        return None

    with open(path, "r") as f:
        return json.load(f)


def load_csv(filename):

    path = os.path.join(TABLE_DIR, filename)

    if not os.path.exists(path):
        return None

    return pd.read_csv(path)


trust = load_json(
    "nb7_integrated_trust_evidence.json"
)

safety_gate = load_json(
    "nb7_cds_input_safety_gate_metadata.json"
)

explanation = load_json(
    "nb7_safety_aware_patient_explanation.json"
)

cds_summary = load_json(
    "nb7_structured_cds_summary.json"
)

remediation = load_json(
    "nb7_patient_data_remediation.json"
)

cohort_status = load_json(
    "nb7_cohort_reconciliation_status.json"
)

prediction_df = load_csv(
    "nb7_prototype_patient_prediction.csv"
)

input_quality_df = load_csv(
    "nb7_prototype_input_quality_audit.csv"
)


# ----------------------------------------------------------------------
# 4. CORE FINAL STATUS
# ----------------------------------------------------------------------

overall_pass = bool(
    audit_status.get("overall_pass", False)
)

overall_status = audit_status.get(
    "overall_status",
    "UNKNOWN"
)

critical_failure_count = int(
    audit_status.get(
        "critical_failure_count",
        0
    )
)


# ----------------------------------------------------------------------
# 5. PROTOTYPE DETAILS
# ----------------------------------------------------------------------

prototype_seqn = None
prediction_probability = None
prediction_class = None

if prediction_df is not None and len(prediction_df) > 0:

    seqn_cols = [
        c for c in prediction_df.columns
        if c.upper() == "SEQN"
    ]

    if seqn_cols:
        prototype_seqn = prediction_df[
            seqn_cols[0]
        ].iloc[0]

    probability_cols = [
        c for c in prediction_df.columns
        if "prob" in c.lower()
    ]

    if probability_cols:
        prediction_probability = float(
            prediction_df[
                probability_cols[0]
            ].iloc[0]
        )

    class_cols = [
        c for c in prediction_df.columns
        if (
            "predicted_class" in c.lower()
            or "class" in c.lower()
        )
    ]

    if class_cols:
        threshold_cols = [
            c for c in class_cols
            if "threshold" in c.lower()
        ]

        selected_class_col = (
            threshold_cols[0]
            if threshold_cols
            else class_cols[-1]
        )

        prediction_class = int(
            prediction_df[
                selected_class_col
            ].iloc[0]
        )


LOCKED_THRESHOLD = 0.35


# ----------------------------------------------------------------------
# 6. INPUT QUALITY
# ----------------------------------------------------------------------

raw_completeness = None
clinical_completeness = None

if input_quality_df is not None:

    lower_map = {
        c.lower(): c
        for c in input_quality_df.columns
    }

    for key in [
        "raw_completeness_pct",
        "completeness_pct"
    ]:

        if key in lower_map:

            raw_completeness = float(
                input_quality_df[
                    lower_map[key]
                ].iloc[0]
            )

            break


    for key in [
        "clinical_measurement_completeness_pct",
        "clinical_completeness_pct"
    ]:

        if key in lower_map:

            clinical_completeness = float(
                input_quality_df[
                    lower_map[key]
                ].iloc[0]
            )

            break


# ----------------------------------------------------------------------
# 7. SAFETY STATUS
# ----------------------------------------------------------------------

gate_status = (
    safety_gate.get("safety_gate")
    or safety_gate.get("gate_status")
    or safety_gate.get("status")
    if isinstance(safety_gate, dict)
    else None
)

explanation_status = (
    explanation.get("explanation_status")
    or explanation.get("status")
    if isinstance(explanation, dict)
    else None
)

remediation_status = (
    remediation.get("status")
    or remediation.get("reassessment_status")
    if isinstance(remediation, dict)
    else None
)


# ----------------------------------------------------------------------
# 8. AUTHORITATIVE BOUNDARY
# ----------------------------------------------------------------------

authoritative_mapping = (
    "NOT_ESTABLISHED"
)

if isinstance(cohort_status, dict):

    cohort_text = json.dumps(
        cohort_status
    ).lower()

    if (
        "not_established" in cohort_text
        or "not established" in cohort_text
    ):
        authoritative_mapping = (
            "NOT_ESTABLISHED"
        )


# ----------------------------------------------------------------------
# 9. DOCUMENTATION GAP INVENTORY
# ----------------------------------------------------------------------

artifact_inventory_path = os.path.join(
    TABLE_DIR,
    "nb7_end_to_end_artifact_inventory.csv"
)

artifact_inventory = pd.read_csv(
    artifact_inventory_path
)

missing_artifacts = artifact_inventory.loc[
    artifact_inventory["status"] == "MISSING",
    "artifact"
].tolist()


# ----------------------------------------------------------------------
# 10. FINAL RESEARCH STATUS
# ----------------------------------------------------------------------

final_research_status = {

    "notebook": "NB7",

    "title": (
        "Clinical Decision Support Prototype"
    ),

    "status": overall_status,

    "audit_passed": overall_pass,

    "critical_failures": (
        critical_failure_count
    ),

    "prototype": {

        "prototype_seqn": prototype_seqn,

        "prediction_probability": (
            prediction_probability
        ),

        "locked_threshold": (
            LOCKED_THRESHOLD
        ),

        "predicted_class": (
            prediction_class
        )
    },

    "safety": {

        "safety_gate": gate_status,

        "explanation_status": (
            explanation_status
        ),

        "remediation_status": (
            remediation_status
        ),

        "clinical_interpretation": (
            "BLOCKED"
        ),

        "deployment_ready": False
    },

    "input_quality": {

        "raw_completeness_pct": (
            raw_completeness
        ),

        "clinical_measurement_completeness_pct": (
            clinical_completeness
        )
    },

    "governance": {

        "authoritative_nb3_mapping": (
            authoritative_mapping
        ),

        "prototype_evaluation_separation": True,

        "individualized_clinical_risk_assessment": (
            False
        ),

        "clinical_diagnosis_or_treatment": (
            False
        ),

        "deployment_claim": False
    },

    "documentation": {

        "expected_artifacts": int(
            len(artifact_inventory)
        ),

        "found_artifacts": int(
            (
                artifact_inventory["status"]
                == "FOUND"
            ).sum()
        ),

        "missing_artifacts": int(
            len(missing_artifacts)
        ),

        "missing_artifact_names": (
            missing_artifacts
        )
    },

    "research_position": (
        "NB7 demonstrates a safety-aware clinical "
        "decision-support prototype using the locked "
        "NB3 predictive model and previously established "
        "NB5 trust/safety evidence and NB6 explanation "
        "consistency evidence. The prototype is not a "
        "clinical validation study, individualized clinical "
        "risk assessment, diagnostic system, treatment "
        "recommendation system, or deployment-ready CDS tool."
    ),

    "key_methodological_boundary": (
        "The prototype patient's exact membership in the "
        "authoritative NB3 held-out cohort was not established. "
        "No SEQN-to-NB3 prediction mapping was inferred."
    ),

    "key_safety_boundary": (
        "The demonstrated patient had insufficient observed "
        "input information. Clinical interpretation was "
        "blocked, missing/imputed values were not treated "
        "as observed clinical evidence, and reassessment "
        "was required."
    )
}


# ----------------------------------------------------------------------
# 11. SAVE FINAL STATUS JSON
# ----------------------------------------------------------------------

final_status_path = os.path.join(
    TABLE_DIR,
    "nb7_final_research_ready_status.json"
)

with open(final_status_path, "w") as f:

    json.dump(
        final_research_status,
        f,
        indent=2,
        default=str
    )


# ----------------------------------------------------------------------
# 12. SAVE FINAL SUMMARY CSV
# ----------------------------------------------------------------------

summary_rows = [

    [
        "Notebook",
        "NB7"
    ],

    [
        "Overall audit status",
        overall_status
    ],

    [
        "Audit passed",
        overall_pass
    ],

    [
        "Critical failures",
        critical_failure_count
    ],

    [
        "Prototype SEQN",
        prototype_seqn
    ],

    [
        "Prototype probability",
        prediction_probability
    ],

    [
        "Locked threshold",
        LOCKED_THRESHOLD
    ],

    [
        "Predicted class",
        prediction_class
    ],

    [
        "Safety gate",
        gate_status
    ],

    [
        "Explanation status",
        explanation_status
    ],

    [
        "Remediation status",
        remediation_status
    ],

    [
        "Clinical interpretation",
        "BLOCKED"
    ],

    [
        "Deployment ready",
        False
    ],

    [
        "Authoritative NB3 mapping",
        authoritative_mapping
    ],

    [
        "Prototype/evaluation separation",
        True
    ],

    [
        "Missing documentation artifacts",
        len(missing_artifacts)
    ]
]

summary_df = pd.DataFrame(
    summary_rows,
    columns=[
        "indicator",
        "value"
    ]
)

summary_csv_path = os.path.join(
    TABLE_DIR,
    "nb7_final_research_ready_summary.csv"
)

summary_df.to_csv(
    summary_csv_path,
    index=False
)


# ----------------------------------------------------------------------
# 13. FINAL RESEARCH-READY REPORT
# ----------------------------------------------------------------------

report_path = os.path.join(
    REPORT_DIR,
    "NB7_FINAL_RESEARCH_READY_REPORT.txt"
)

timestamp = datetime.now().isoformat()

with open(report_path, "w") as f:

    f.write(
        "NB7 — FINAL RESEARCH-READY REPORT\n"
    )

    f.write("=" * 70 + "\n\n")

    f.write(
        f"Generated: {timestamp}\n\n"
    )

    f.write(
        "1. PURPOSE\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        "NB7 implements a clinical decision-support "
        "prototype on top of the locked NB3 early-detection "
        "model. It integrates patient-level prediction, "
        "input-quality validation, safety gating, "
        "model-derived explanation, trust evidence, "
        "and reassessment logic.\n\n"
    )

    f.write(
        "2. MODEL GOVERNANCE\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        "The predictive model and preprocessing pipeline "
        "remain locked. The NB7 prototype does not retrain "
        "or modify the NB3 model.\n"
    )

    f.write(
        f"Locked decision threshold: {LOCKED_THRESHOLD}\n\n"
    )

    f.write(
        "3. PROTOTYPE DEMONSTRATION\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        f"Prototype SEQN: {prototype_seqn}\n"
    )

    f.write(
        f"Model probability: {prediction_probability}\n"
    )

    f.write(
        f"Predicted class: {prediction_class}\n\n"
    )

    f.write(
        "The demonstration patient is a prototype "
        "example and is not claimed to be part of the "
        "authoritative NB3 held-out evaluation cohort.\n\n"
    )

    f.write(
        "4. SAFETY GATE\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        f"Safety gate: {gate_status}\n"
    )

    f.write(
        f"Explanation status: {explanation_status}\n"
    )

    f.write(
        f"Remediation status: {remediation_status}\n"
    )

    f.write(
        "Clinical interpretation: BLOCKED\n"
    )

    f.write(
        "Deployment readiness: NO\n\n"
    )

    f.write(
        "5. INPUT-QUALITY BOUNDARY\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        f"Raw input completeness: {raw_completeness}%\n"
    )

    f.write(
        f"Clinical measurement completeness: "
        f"{clinical_completeness}%\n\n"
    )

    f.write(
        "The model can technically produce a prediction "
        "because its preprocessing pipeline handles missing "
        "values. However, model output under substantial "
        "missingness is not treated as sufficient evidence "
        "for individualized clinical interpretation.\n\n"
    )

    f.write(
        "6. TRUSTWORTHINESS EVIDENCE\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        "NB3 provides predictive performance evidence.\n"
    )

    f.write(
        "NB5 provides calibration, uncertainty, fairness, "
        "and safety evidence.\n"
    )

    f.write(
        "NB6 provides model-level explanation consistency "
        "evidence.\n"
    )

    f.write(
        "These evidence streams are retained as separate "
        "layers and are not conflated with clinical validity.\n\n"
    )

    f.write(
        "7. GOVERNANCE BOUNDARIES\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        "• Prototype/evaluation separation: MAINTAINED\n"
    )

    f.write(
        "• Exact NB3 patient mapping: NOT ESTABLISHED\n"
    )

    f.write(
        "• Clinical interpretation: BLOCKED\n"
    )

    f.write(
        "• Diagnosis/treatment recommendation: NOT PROVIDED\n"
    )

    f.write(
        "• Deployment claim: NOT MADE\n"
    )

    f.write(
        "• Missing/imputed values: NOT treated as observed\n"
    )

    f.write(
        "• Explanation consistency: model-level evidence only\n\n"
    )

    f.write(
        "8. DOCUMENTATION GAPS\n"
    )

    f.write("-" * 70 + "\n")

    if missing_artifacts:

        for filename in missing_artifacts:

            f.write(
                f"• {filename}\n"
            )

        f.write(
            "\nThese are documentation artifacts that were "
            "not available in the final tables directory. "
            "They were not fabricated or reconstructed.\n\n"
        )

    else:

        f.write(
            "No documentation gaps identified.\n\n"
        )

    f.write(
        "9. FINAL STATUS\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        f"Overall status: {overall_status}\n"
    )

    f.write(
        f"Critical failures: {critical_failure_count}\n"
    )

    f.write(
        f"Audit passed: {overall_pass}\n\n"
    )

    f.write(
        "CONCLUSION\n"
    )

    f.write("-" * 70 + "\n")

    f.write(
        "NB7 successfully establishes a governed clinical "
        "decision-support prototype around the locked NB3 "
        "model. The end-to-end governance audit passed "
        "with zero critical failures. The demonstrated "
        "patient remains intentionally blocked from "
        "individualized clinical interpretation because "
        "of insufficient observed input information. "
        "Therefore, the prototype demonstrates the "
        "architecture and safety logic of trustworthy "
        "clinical decision support rather than clinical "
        "deployment or clinical efficacy.\n"
    )


# ----------------------------------------------------------------------
# 14. FINAL DISPLAY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("NB7 FINAL RESEARCH STATUS")
print("=" * 70)

print(
    f"Overall status:        {overall_status}"
)

print(
    f"Audit passed:          {overall_pass}"
)

print(
    f"Critical failures:     {critical_failure_count}"
)

print(
    f"Prototype SEQN:        {prototype_seqn}"
)

print(
    f"Probability:           {prediction_probability}"
)

print(
    f"Locked threshold:      {LOCKED_THRESHOLD}"
)

print(
    f"Predicted class:       {prediction_class}"
)

print("\nSafety:")
print(
    f"  Gate:                {gate_status}"
)

print(
    f"  Explanation:         {explanation_status}"
)

print(
    f"  Remediation:         {remediation_status}"
)

print(
    "  Clinical use:        BLOCKED"
)

print(
    "  Deployment:          NO"
)

print("\nGovernance:")
print(
    "  NB3 mapping:         NOT_ESTABLISHED"
)

print(
    "  Prototype separated: TRUE"
)

print(
    f"\nDocumentation gaps:    {len(missing_artifacts)}"
)

print("\nSaved:")
print(
    f"  {final_status_path}"
)

print(
    f"  {summary_csv_path}"
)

print(
    f"  {report_path}"
)

print("=" * 70)
print("NB7 FINAL DOCUMENTATION COMPLETED")
print("=" * 70)

CELL 32 — NB7 FINAL RESEARCH-READY STATUS & DOCUMENTATION

NB7 FINAL RESEARCH STATUS
Overall status:        PASSED_WITH_DOCUMENTATION_GAPS
Audit passed:          True
Critical failures:     0
Prototype SEQN:        109263.0
Probability:           0.0171951694927695
Locked threshold:      0.35
Predicted class:       0

Safety:
  Gate:                INSUFFICIENT_FOR_CLINICAL_INTERPRETATION
  Explanation:         CLINICAL_EXPLANATION_BLOCKED
  Remediation:         REASSESSMENT_REQUIRED
  Clinical use:        BLOCKED
  Deployment:          NO

Governance:
  NB3 mapping:         NOT_ESTABLISHED
  Prototype separated: TRUE

Documentation gaps:    5

Saved:
  /content/nb7_clinical_decision_support/tables/nb7_final_research_ready_status.json
  /content/nb7_clinical_decision_support/tables/nb7_final_research_ready_summary.csv
  /content/nb7_clinical_decision_support/final_report/NB7_FINAL_RESEARCH_READY_REPORT.txt
NB7 FINAL DOCUMENTATION COMPLETED
